# AdaGRPO V3: early exploration under a fixed budget

Five initial K=5 groups, followed by frozen V2 adaptive allocation. The live screen is explicitly authorized, bounded to two hours, and cannot demonstrate formal research success. Definition cells launch no model work.


In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dspy==3.3.1", "transformers==5.17.0",
                           "datasets==5.0.1", "numpy>=2.0,<3", "matplotlib>=3.10,<4", "nbformat>=5.10", "ipykernel", "nbclient", "psutil"])

import copy, gc, hashlib, importlib.metadata, json, math, os, random, re, sys, tempfile, time, uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import torch
import dspy
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import GPT2LMHeadModel, GPT2Tokenizer

VERSIONS = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "dspy", "datasets", "numpy", "matplotlib"]}
print("Python:", sys.version, "\nPackages:", VERSIONS, "\nCUDA:", torch.cuda.is_available())
import platform, shutil, psutil
from collections import Counter, defaultdict


## Configuration and protocol
Adaptive K is automatic. Bounds and the run budget are experiment safety controls. Membership uses a fixed data seed; each training seed has its own shared replacement schedule. Test data loading is deferred until explicit final testing.


In [ ]:
PROTOCOL = "adagrpo-gsm8k-numeric-figure2-v2-verified-v1"
PRESET = "smoke"
ALLOCATION_MODE = "adaptive"
STATIC_K = 5
RUN_OPERATION = False


def build_config(preset=None, **overrides):
    preset = preset or PRESET
    presets = dict(smoke=(2,10,2,2), screen=(12,50,20,12), pilot=(20,50,20,10), formal=(200,200,50,50))
    if preset not in presets:
        raise ValueError("preset must be smoke, screen, pilot or formal")
    episodes,pool,validation,interval=presets[preset]
    cfg=dict(protocol=PROTOCOL, dataset="gsm8k", dataset_name="openai/gsm8k",dataset_config="main",
        dataset_revision=None, data_seed=42, seed=42, preset=preset, num_episodes=episodes,
        train_pool_size=pool, validation_size=validation, validation_interval=interval, schedule_episodes=200,
        test_scope="full_official", allocation_mode=ALLOCATION_MODE, static_k=STATIC_K,
        k_min=2,k_max=6,probe_size=2,expansion_size=2,all_correct_min=2,
        concentration_min=4,concentration_neff=1.25,contrast_epsilon=1e-12,
        adaptive_budget_per_episode=4,training_budget=None,controller_version="credit-refine-v2",sampling_version="candidate-stream-v2",membership_manifest=None,suite_ledger=None,allocation_schedule=None,
        policy_model="gpt2",policy_revision="607a30d783dfa663caf39e06633721c8d4cfcd7e",
        policy_device="cuda",policy_dropout=False,max_policy_tokens=1024,max_pipeline_steps=3,
        policy_sampling="restricted-first-token-multinomial", learning_rate=1e-5,
        optimizer="Adam",loss="negative-sum-logprob-times-centered-reward",reward_mode="numeric_exact",
        semantic_fallback=False,cache=False,ollama_model="llama3.2:3b",ollama_host="http://localhost:11434",
        ollama_num_gpu=0,ollama_num_ctx=2048,execution_max_tokens=512,execution_temperature=0.0,
        request_timeout=180,results_root="results/adagrpo_verified",checkpoint_interval=4 if preset=='screen' else 5 if preset!='formal' else 25,
        evaluation_seed_offset=100000,wall_limit_seconds=3600 if preset in ('pilot','screen') else 900 if preset=='smoke' else None)
    cfg.update(overrides)
    if cfg['probe_size']!=2 or cfg['expansion_size']!=2 or cfg['k_max']!=6 or cfg['adaptive_budget_per_episode']!=4: raise ValueError('Frozen v2 controller requires probe2/expand2/max6/budget4')
    if cfg['all_correct_min']!=2 or cfg['contrast_epsilon']!=1e-12: raise ValueError('Frozen stopping thresholds differ')
    if cfg['allocation_mode'] not in ('adaptive','static','scheduled'): raise ValueError('Invalid allocation_mode')
    if cfg['static_k'] not in (2,4,5,8,10,15): raise ValueError('Unsupported static_k')
    for key in ('num_episodes','train_pool_size','validation_size','validation_interval','schedule_episodes',
                'checkpoint_interval','k_min','k_max','probe_size','expansion_size','all_correct_min',
                'concentration_min','ollama_num_ctx','execution_max_tokens','max_policy_tokens'):
        if type(cfg[key]) is not int or cfg[key]<=0: raise ValueError(key+' must be a positive integer')
    if not 2<=cfg['k_min']<=cfg['probe_size']<=cfg['k_max']: raise ValueError('Invalid K bounds')
    if cfg['policy_device'] not in ('cpu','cuda') or cfg['learning_rate']<=0: raise ValueError('Invalid device/rate')
    if cfg['reward_mode']!='numeric_exact' or cfg['semantic_fallback'] or cfg['cache']: raise ValueError('Protocol forbids judging/caching')
    cfg['schedule_episodes']=max(cfg['num_episodes'],cfg['schedule_episodes'])
    if cfg['allocation_mode']=='scheduled':
        schedule=cfg['allocation_schedule']
        if not isinstance(schedule,list) or len(schedule)!=cfg['num_episodes'] or any(type(k) is not int or not 2<=k<=6 for k in schedule): raise ValueError('Invalid shuffled K schedule')
        cfg['training_budget']=sum(schedule) if cfg['training_budget'] is None else cfg['training_budget']
    if cfg['training_budget'] is None:
        cfg['training_budget']=cfg['num_episodes']*(cfg['adaptive_budget_per_episode'] if cfg['allocation_mode']=='adaptive' else cfg['static_k'])
    minimum=cfg['k_min'] if cfg['allocation_mode'] in ('adaptive','scheduled') else cfg['static_k']
    if type(cfg['training_budget']) is not int or cfg['training_budget']<minimum*cfg['num_episodes']:
        raise ValueError('Budget cannot support minimum groups')
    if cfg['protocol']!=PROTOCOL or cfg['controller_version']!='credit-refine-v2' or cfg['sampling_version']!='candidate-stream-v2': raise ValueError('Unsupported protocol/controller')
    if cfg['loss']!='negative-sum-logprob-times-centered-reward' or cfg['optimizer']!='Adam' or cfg['policy_dropout']:
        raise ValueError('Figure 2/Adam/dropout protocol mismatch')
    for key in ('concentration_neff','contrast_epsilon','adaptive_budget_per_episode'):
        if not math.isfinite(cfg[key]) or cfg[key]<=0: raise ValueError('Invalid '+key)
    if cfg['wall_limit_seconds'] is not None and cfg['wall_limit_seconds']<=0: raise ValueError('Invalid wall limit')
    return cfg

CONFIG=build_config()
print('Definitions only. Configuration:',CONFIG)


## Hardware, inference, RNG, and immutable identities
Cache and provider retries are disabled. Calls are logical LM.forward attempts; physical HTTP counts are unavailable. Missing tokens remain unavailable. Evaluation temporarily isolates all sampling RNG.


In [ ]:
def json_request(host, route, payload=None, timeout=15):
    data = None if payload is None else json.dumps(payload).encode()
    request = Request(host.rstrip("/")+route, data=data, headers={"Content-Type":"application/json"})
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

def ollama_snapshot(cfg):
    try:
        return json_request(cfg["ollama_host"], "/api/ps", timeout=5)
    except Exception as exc:
        return {"unavailable": f"{type(exc).__name__}: {exc}"}

def judge_inference_parameters(lm, overrides=None):
    # Record provider request settings, never authentication keys or headers.
    configured = dict(getattr(lm, "kwargs", {}) or {})
    configured.update(overrides or {})
    allowed = ("api_base", "temperature", "max_tokens", "seed", "num_ctx", "num_gpu",
               "top_p", "top_k", "repeat_penalty", "stop", "timeout", "keep_alive", "rollout_id")
    return dict(model=getattr(lm,"model",None), model_type=getattr(lm,"model_type",None),
                cache=configured.get("cache",getattr(lm,"cache",None)),
                num_retries=getattr(lm,"num_retries",None),
                parameters={key:copy.deepcopy(configured[key]) for key in allowed if key in configured},
                scope="DSPy/LiteLLM request settings; physical HTTP/server settings not observed")

class MeasuredLM(dspy.LM):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        self.events=[]
    def forward(self,*args,**kwargs):
        event=dict(seconds=None,prompt_tokens=None,generated_tokens=None,error=None,completed=False)
        self.events.append(event)  # count logical attempts even when interrupted
        started=time.perf_counter()
        try:
            response=super().forward(*args,**kwargs)
            usage=dict(getattr(response,'usage',{}) or {})
            event.update(prompt_tokens=usage.get('prompt_tokens'),generated_tokens=usage.get('completion_tokens'))
            return response
        except BaseException as exc:
            event['error']=f'{type(exc).__name__}: {exc}'
            raise
        finally:
            event['seconds']=time.perf_counter()-started; event['completed']=True

def initialize_ollama(cfg):
    try:
        version = json_request(cfg["ollama_host"], "/api/version")
        tags = json_request(cfg["ollama_host"], "/api/tags")["models"]
        model = next((m for m in tags if m.get("name")==cfg["ollama_model"] or m.get("model")==cfg["ollama_model"]), None)
        if model is None:
            raise RuntimeError(f"Missing {cfg['ollama_model']}; run ollama pull {cfg['ollama_model']}")
        show = json_request(cfg["ollama_host"], "/api/show", {"model":cfg["ollama_model"]})
    except Exception as exc:
        raise RuntimeError(f"Ollama unavailable at {cfg['ollama_host']}: {exc}") from exc
    options = dict(api_base=cfg["ollama_host"], api_key="", cache=False, num_retries=0,
                   temperature=cfg["execution_temperature"], max_tokens=cfg["execution_max_tokens"],
                   seed=cfg["seed"], num_ctx=cfg["ollama_num_ctx"], timeout=cfg["request_timeout"], keep_alive="5m")
    if cfg["ollama_num_gpu"] is not None:
        options["num_gpu"] = cfg["ollama_num_gpu"]
    lm = MeasuredLM("ollama_chat/"+cfg["ollama_model"], **options)
    dspy.configure(lm=lm)
    metadata = {"version":version, "model":model, "details":show.get("details"),
                "model_info":show.get("model_info"), "options":options, "ps":ollama_snapshot(cfg)}
    return lm, metadata

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(states):
    random.setstate(states["python"]); np.random.set_state(states["numpy"]); torch.set_rng_state(states["torch"])
    if states["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("Checkpoint has CUDA RNG state; resume in its original GPU environment")
        torch.cuda.set_rng_state_all(states["cuda"])

@contextmanager
def isolated_rng(seed):
    saved = rng_state()
    seed_everything(seed)
    try:
        yield
    finally:
        restore_rng(saved)

def hardware_metadata():
    return dict(python=sys.version, packages=VERSIONS, cuda_available=torch.cuda.is_available(),
                cuda_version=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
                cpu=platform.processor(),system_ram_bytes=psutil.virtual_memory().total,process_rss_bytes=psutil.Process().memory_info().rss)


## Numeric extraction and dataset manifests
The inherited last-number fallback can misidentify a reasoning number as a final answer. Its use is logged; no semantic rescue changes the reward. Targets are never supplied to policy or execution prompts.


In [ ]:
def canonical_number(text):
    try:
        number = Decimal(str(text).strip().replace(",", ""))
        if not number.is_finite():
            return None
        return "0" if number == 0 else format(number.normalize(), "f")
    except (InvalidOperation, ValueError):
        return None

def canonical_target(answer):
    if "####" not in answer:
        raise ValueError("GSM8K answer lacks #### delimiter")
    target = canonical_number(answer.rsplit("####",1)[1])
    if target is None:
        raise ValueError(f"Invalid GSM8K numeric target: {answer!r}")
    return target

def load_gsm8k(cfg):
    cfg=copy.deepcopy(cfg)
    if cfg['dataset_revision'] is None:
        cfg['dataset_revision']=HfApi().dataset_info(cfg['dataset_name']).sha
    # Never load the official test split during smoke, pilot, or training.
    dataset={'train':load_dataset(cfg['dataset_name'],cfg['dataset_config'],revision=cfg['dataset_revision'],split='train')}
    return cfg,dataset

def dataset_example(dataset, split, index):
    item = dataset[split][int(index)]
    return dict(id=f"gsm8k:{split}:{index}", split=split, index=int(index), question=item["question"],
                answer=canonical_target(item["answer"]), rationale=item["answer"])

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path,value,overwrite=False):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and not overwrite: raise FileExistsError(path)
    tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with tmp.open('w',encoding='utf-8') as stream:
            json.dump(value,stream,ensure_ascii=False,indent=2,allow_nan=False); stream.flush(); os.fsync(stream.fileno())
        os.replace(tmp,path)
    finally:
        tmp.unlink(missing_ok=True)


In [ ]:
def selection_payload(cfg,datasets):
    train=datasets['train']; n=len(train)
    if cfg['train_pool_size']+cfg['validation_size']>n: raise ValueError('Selection exceeds GSM8K train split')
    order=np.random.default_rng(cfg['data_seed']).permutation(n)
    if cfg.get('membership_manifest'):
        frozen=json.loads(Path(cfg['membership_manifest']).read_text(encoding='utf-8'))
        if frozen['dataset_revision']!=cfg['dataset_revision']: raise ValueError('Frozen dataset revision mismatch')
        indices=copy.deepcopy(frozen['indices'])
        if len(indices['train'])!=cfg['train_pool_size'] or len(indices['validation'])!=cfg['validation_size']: raise ValueError('Frozen split sizes differ')
        if any(type(i) is not int or not 0<=i<n for ids in indices.values() for i in ids): raise ValueError('Frozen index out of range')
        if any(len(ids)!=len(set(ids)) for ids in indices.values()): raise ValueError('Frozen indices repeated')
    else:
        indices=dict(train=order[:cfg['train_pool_size']].tolist(),validation=order[cfg['train_pool_size']:cfg['train_pool_size']+cfg['validation_size']].tolist())
    subsets={k:[dataset_example(datasets,'train',i) for i in ids] for k,ids in indices.items()}
    if cfg.get('membership_manifest') and frozen['content_hashes']!={k:stable_hash(rows) for k,rows in subsets.items()}: raise ValueError('Frozen dataset content mismatch')
    schedule=np.random.default_rng(cfg['seed']+2).integers(0,cfg['train_pool_size'],size=cfg['schedule_episodes']).tolist()
    identity={k:cfg[k] for k in ('dataset_name','dataset_config','dataset_revision','data_seed','seed','train_pool_size','validation_size','schedule_episodes')}
    payload=dict(identity=identity,indices=indices,ids={k:[r['id'] for r in rows] for k,rows in subsets.items()},
        content_hashes={k:stable_hash(rows) for k,rows in subsets.items()},episode_pool_offsets=schedule,
        episode_question_ids=[subsets['train'][i]['id'] for i in schedule],test_scope='full_official_deferred')
    return payload,subsets

def prepare_selection(cfg,datasets,persist=True):
    payload,subsets=selection_payload(cfg,datasets); sid=stable_hash(payload)
    path=Path(cfg['results_root'])/'gsm8k'/'selections'/(sid+'.json')
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8'))!=payload: raise RuntimeError('Selection corruption')
    elif persist: write_json(path,payload)
    if set(payload['indices']['train']) & set(payload['indices']['validation']): raise RuntimeError('Split leakage')
    return dict(subsets=subsets,selection=payload,selection_id=sid,selection_path=str(path))


In [ ]:
MODULES = ["CoT", "Predict"]
SIGNATURES = ["question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
              "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
              "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
              "text -> response", "prompt -> generated_text", "query -> generated_text"]
ACTIONS = MODULES + SIGNATURES + ["stop"]

def valid_actions(partial):
    return MODULES if not partial else SIGNATURES if len(partial)==1 else ["stop"]


In [ ]:
def parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        digest.update(name.encode()); digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def initialize_policy(cfg):
    if cfg["policy_device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA policy requested but unavailable. Select the GPU kernel or explicitly use CPU")
    seed_everything(cfg["seed"])
    try:
        tokenizer = GPT2Tokenizer.from_pretrained(cfg["policy_model"],revision=cfg["policy_revision"])
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.truncation_side = "left"  # preserves Pipeline suffix if a custom long prompt exceeds 1024
        model = GPT2LMHeadModel.from_pretrained(cfg["policy_model"],revision=cfg["policy_revision"])
    except Exception as exc:
        raise RuntimeError("GPT-2 files unavailable: "+str(exc)) from exc
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.to(cfg["policy_device"]).eval()
    optimizer = torch.optim.Adam(model.parameters(),lr=cfg["learning_rate"],foreach=False)
    first_tokens = {action:tokenizer.encode(action,add_special_tokens=False)[0] for action in ACTIONS}
    initial_digest = parameter_digest(model)
    return model, tokenizer, optimizer, first_tokens, initial_digest

def release_experiment(session):
    if session is not None:
        for key in ("policy","optimizer","tokenizer"):
            session.pop(key,None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
def action_log_probs(model, tokenizer, first_tokens, prompt, partial, cfg):
    choices = valid_actions(partial)
    text = f"Prompt: {prompt} Pipeline: {' '.join(partial)}"
    encoded = tokenizer(text,return_tensors="pt",padding=True,truncation=True,
                        max_length=min(cfg["max_policy_tokens"],model.config.n_positions))
    device = next(model.parameters()).device
    encoded = {key:value.to(device) for key,value in encoded.items()}
    logits = model(**encoded,use_cache=False).logits[0,-1]
    ids = torch.tensor([first_tokens[action] for action in choices],device=device)
    return choices, torch.log_softmax(logits[ids],dim=0)

def action_statistics(choices, scores):
    probabilities = scores.detach().double().exp()
    return dict(probabilities=dict(zip(choices,probabilities.cpu().tolist())),
                entropy_nats=float(-(probabilities*scores.detach().double()).sum()))


def probe_policy(session, prompt):
    # Enumerate both signature conditionals; no draws, graphs, or RNG changes.
    model, cfg = session["policy"], session["config"]
    previous_mode = model.training
    model.eval()
    try:
        with torch.no_grad():
            choices, scores = action_log_probs(model,session["tokenizer"],session["first_tokens"],prompt,[],cfg)
            modules = action_statistics(choices,scores)
            signatures = {}
            for module in MODULES:
                choices, scores = action_log_probs(model,session["tokenizer"],session["first_tokens"],prompt,[module],cfg)
                signatures[module] = action_statistics(choices,scores)
        expected = sum(modules["probabilities"][m]*signatures[m]["entropy_nats"] for m in MODULES)
        return dict(module=modules,signature_given_module=signatures,
                    expected_signature_entropy_nats=expected,
                    joint_pipeline_entropy_nats=modules["entropy_nats"]+expected,
                    entropy_units="nats",scope="fixed current training question; deterministic stop excluded")
    finally:
        model.train(previous_mode)

def sample_actions(model, tokenizer, first_tokens, prompt, cfg, generator=None):
    partial, log_probs, diagnostics = [], [], []
    with torch.no_grad():
        for _ in range(cfg["max_pipeline_steps"]):
            choices, scores = action_log_probs(model,tokenizer,first_tokens,prompt,partial,cfg)
            index = int(torch.multinomial(scores.exp(),1,generator=generator).item())
            action = choices[index]
            diagnostics.append(dict(state_actions=list(partial),selected_action=action,
                                    selected_probability=float(scores[index].double().exp()),
                                    **action_statistics(choices,scores)))
            partial.append(action); log_probs.append(float(scores[index]))
            if action == "stop":
                break
    return dict(actions=partial,log_probs=log_probs,action_diagnostics=diagnostics)

def replay_log_probs(model, tokenizer, first_tokens, prompt, trajectory, cfg):
    partial, log_probs = [], []
    for action in trajectory["actions"]:
        choices, scores = action_log_probs(model,tokenizer,first_tokens,prompt,partial,cfg)
        if action not in choices:
            raise ValueError("Trajectory action violates released action grammar")
        log_probs.append(scores[choices.index(action)])
        partial.append(action)
    return log_probs

def action_collision_report(first_tokens):
    groups = {}
    for signature in SIGNATURES:
        groups.setdefault(first_tokens[signature],[]).append(signature)
    return {token:actions for token,actions in groups.items() if len(actions)>1}


In [ ]:
def generate_pipeline(model, tokenizer, first_tokens, prompt, cfg, generator=None):
    trajectory = sample_actions(model,tokenizer,first_tokens,prompt,cfg,generator=generator)
    actions = trajectory["actions"]
    trajectory["module"] = actions[0] if actions else None
    trajectory["signature"] = actions[1] if len(actions)>1 else None
    return trajectory


## Numeric reward and infrastructure failures
Malformed or incorrect outputs receive zero. Service failures abort the group; their attempts remain in the journal. Adapter retries, if any, are separately measured LM calls.


In [ ]:
NUMBER_PATTERN = re.compile(r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|[-+]?\.\d+")

def extract_answer(response):
    if response is None:
        return None
    text = str(response).strip()
    brackets = re.findall(r"\[([^\[\]]+)\]",text)
    if brackets:
        text = brackets[-1].strip()
        matches = list(NUMBER_PATTERN.finditer(text))
        if len(matches)!=1:
            return None
        return canonical_number(matches[0].group())
    if "####" in text:
        text = text.rsplit("####",1)[1].strip()
    matches = list(NUMBER_PATTERN.finditer(text))
    return canonical_number(matches[-1].group()) if matches else None

def compare_answers(response, target):
    prediction = extract_answer(response)
    truth = canonical_number(target)
    return prediction is not None and truth is not None and prediction == truth


def extraction_details(response):
    text='' if response is None else str(response)
    source='brackets' if re.findall(r"\[([^\[\]]+)\]",text) else 'hash_delimiter' if '####' in text else 'last_number' if NUMBER_PATTERN.search(text) else 'unparseable'
    value=extract_answer(response)
    return dict(prediction=value,extraction_source=source,parse_valid=value is not None)

def compute_reward(prompt,response,target,lm=None):
    details=extraction_details(response)
    exact=compare_answers(response,target)
    return dict(reward=float(exact),numeric_exact_match=exact,semantic_used=False,judge_score=None,judge_error=None,**details)

class InfrastructureFailure(RuntimeError): pass
class BudgetExhausted(RuntimeError): pass
class WallLimitReached(RuntimeError): pass

def infrastructure_error(exc):
    # Walk wrapped causes; parse/adapter errors remain candidate-level failures.
    chain=[]; seen=set()
    while exc is not None and id(exc) not in seen:
        seen.add(id(exc)); chain.append(exc); exc=exc.__cause__ or exc.__context__
    names=' '.join(type(e).__name__.lower() for e in chain)
    text=' '.join(str(e).lower() for e in chain)
    return any(isinstance(e,(TimeoutError,ConnectionError,torch.cuda.OutOfMemoryError)) for e in chain) or any(
        x in names+' '+text for x in ('timeout','timed out','connection refused','connectionerror','apiconnection','out of memory','server disconnected','503 service','502 bad gateway','failed to connect'))


In [ ]:
def execute_pipeline(trajectory, prompt, lm):
    started = time.perf_counter()
    try:
        module, signature = trajectory["module"],trajectory["signature"]
        if module not in MODULES or signature not in SIGNATURES:
            raise ValueError("Invalid pipeline module or signature")
        input_field, output_field = [part.strip() for part in signature.split("->")]
        program = dspy.ChainOfThought(signature) if module == "CoT" else dspy.Predict(signature)
        instruction = "system instruction: Must give your final answer in square brackets without fail! e.g., [final answer] like this: [5] \n prompt: "
        with dspy.context(lm=lm):
            response = program(**{input_field:instruction+prompt})
        output = response.get(output_field)
        if output is None:
            raise ValueError("DSPy returned no signature output")
        return dict(output=str(output),success=True,error=None,seconds=time.perf_counter()-started)
    except Exception as exc:
        if infrastructure_error(exc): raise InfrastructureFailure(str(exc)) from exc
        return dict(output=None,success=False,error=f"{type(exc).__name__}: {exc}",seconds=time.perf_counter()-started)


## Controller v2: bounded refinement and bank pacing
Two correct probes stop at two. Other groups reach four; all-failure and cancelled-credit groups stop. Useful contrast may reach six only with saved budget. All collected candidates are retained. Neff remains a diagnostic, not a concentration stop. Gross failures remain charged; two future attempts per episode remain reserved.


In [ ]:
def policy_class(trajectory,first_tokens):
    return (int(first_tokens[trajectory['module']]),int(first_tokens[trajectory['signature']]))

def effective_policy_distribution(probe,first_tokens):
    probabilities=defaultdict(float)
    for module,pm in probe['module']['probabilities'].items():
        for signature,ps in probe['signature_given_module'][module]['probabilities'].items():
            probabilities[(int(first_tokens[module]),int(first_tokens[signature]))]+=float(pm)*float(ps)
    total=sum(probabilities.values())
    if not probabilities or not math.isclose(total,1.0,rel_tol=1e-5,abs_tol=1e-5): raise ValueError('Invalid joint policy probabilities')
    values={c:p/total for c,p in probabilities.items()}
    return dict(neff=1.0/sum(p*p for p in values.values()),
                classes=[dict(key=list(c),probability=p) for c,p in sorted(values.items())])

def contrast_statistics(trajectories,rewards,first_tokens):
    if not rewards or len(trajectories)!=len(rewards): raise ValueError('Invalid contrast group')
    mean,advantages=group_advantages(rewards)
    totals=defaultdict(float)
    for t,a in zip(trajectories,advantages): totals[policy_class(t,first_tokens)]+=float(a)
    return dict(credit_contrast=sum(abs(x) for x in totals.values()),class_count=len(totals),
        mixed_rewards=min(rewards)!=max(rewards),class_credit=[dict(key=list(c),centered_total=a) for c,a in sorted(totals.items())])

def controller_decision(cfg,trajectories,rewards,first_tokens,neff,allowance):
    n=len(rewards)
    if any(r not in (0.,1.) for r in rewards): raise ValueError('Controller requires binary numeric rewards')
    if n<cfg['k_min']: return dict(stop=False,reason='probe',actual_k=n)
    stats=contrast_statistics(trajectories,rewards,first_tokens); reason=None
    if cfg['allocation_mode'] in ('static','scheduled'):
        reason='static_k' if n>=allowance else None
    elif n==2 and all(r==1. for r in rewards): reason='easy_probe'
    elif n>=4 and all(r==0. for r in rewards): reason='bounded_failure'
    elif n>=4 and stats['credit_contrast']<=cfg['contrast_epsilon']: reason='credit_cancelled'
    elif n>=6: reason='informative_refinement'
    elif n>=allowance: reason='allowance_reached'
    return dict(stop=reason is not None,reason=reason or 'expand',actual_k=n,neff=neff,allowance=allowance,**stats)

def read_jsonl(path):
    path=Path(path)
    return [json.loads(line) for line in path.read_text(encoding='utf-8').split('\n') if line.strip()] if path.exists() else []

def repair_journal_tail(root):
    """Repair only an unterminated final record; preserve its exact bytes first."""
    root=Path(root); path=root/'attempts.jsonl'
    if not path.exists(): return None
    raw=path.read_bytes(); lines=raw.splitlines(keepends=True); offset=0
    for index,line in enumerate(lines):
        try:
            if line.strip(): json.loads(line.decode('utf-8'))
        except (json.JSONDecodeError,UnicodeDecodeError) as exc:
            if index!=len(lines)-1 or line.endswith(b'\n'):
                raise ValueError('Corruption inside a committed journal record; refusing automatic repair') from exc
            backup=root/('journal_tail_'+uuid.uuid4().hex+'.bin')
            with backup.open('xb') as stream:
                stream.write(raw[offset:]); stream.flush(); os.fsync(stream.fileno())
            with path.open('r+b') as stream:
                stream.truncate(offset); stream.flush(); os.fsync(stream.fileno())
            event=dict(kind='journal_tail_repaired',backup=backup.name,removed_bytes=len(raw)-offset,
                scope='Valid started attempts remain charged; incomplete finish usage remains unavailable')
            append_journal(dict(run_dir=root),event)
            return event
        offset+=len(line)
    if raw and not raw.endswith(b'\n'):
        with path.open('ab') as stream:
            stream.write(b'\n'); stream.flush(); os.fsync(stream.fileno())
        return dict(kind='journal_newline_restored',removed_bytes=0)
    return None

def append_journal(session,event):
    root=session.get('run_dir')
    if root is None: return
    event=dict(timestamp=datetime.now(timezone.utc).isoformat(),**event)
    with (Path(root)/'attempts.jsonl').open('a',encoding='utf-8') as stream:
        stream.write(json.dumps(event,ensure_ascii=False,allow_nan=False)+'\n'); stream.flush(); os.fsync(stream.fileno())

def journal_counts(session):
    events=read_jsonl(Path(session['run_dir'])/'attempts.jsonl')
    starts=[r for r in events if r['kind']=='training_attempt_started']
    ids=[r['attempt_id'] for r in starts]
    if len(ids)!=len(set(ids)): raise RuntimeError('Duplicate journal attempt ID')
    return len(starts)

def budget_snapshot(session):
    cfg,state=session['config'],session['state']; used=journal_counts(session)
    future=cfg['num_episodes']-state['episode']-1
    if cfg['allocation_mode']=='static': minimum=cfg['static_k']; bank=0; allowance=minimum
    elif cfg['allocation_mode']=='scheduled':
        minimum=cfg['allocation_schedule'][state['episode']]; bank=0; allowance=minimum
        future_cost=sum(cfg['allocation_schedule'][state['episode']+1:])
        if cfg['training_budget']-used-future_cost<minimum: raise BudgetExhausted('Shuffled schedule cannot fit gross budget')
        return dict(allowance=allowance,bank=0,gross_before=used,future_reservation=future_cost)
    else:
        minimum=cfg['k_min']; bank=max(0,cfg['adaptive_budget_per_episode']*state['episode']-used)
        allowance=min(cfg['k_max'],cfg['training_budget']-used-minimum*future,4+bank)
    available=cfg['training_budget']-used-minimum*future
    if available<minimum or allowance<minimum: raise BudgetExhausted('Gross remaining budget cannot reserve minimum groups')
    return dict(allowance=allowance,bank=bank,gross_before=used,future_reservation=minimum*future)


def episode_allowance(session): return budget_snapshot(session)['allowance']

def check_deadline(session):
    if session['config'].get('suite_ledger'):
        ledger=json.loads(Path(session['config']['suite_ledger']).read_text(encoding='utf-8'))
        if datetime.now(timezone.utc).timestamp()>=ledger['deadline_unix']: raise WallLimitReached('Persisted two-hour suite deadline reached')
    if session.get('deadline') is not None and time.perf_counter()>=session['deadline']:
        raise WallLimitReached('Configured wall-time limit reached; current partial group is not committed')

def call_statistics(events):
    def total(key):
        values=[e.get(key) for e in events]
        return sum(values) if all(v is not None for v in values) else None
    return dict(execution_llm_calls=len(events),physical_http_requests=None,prompt_tokens=total('prompt_tokens'),
        generated_tokens=total('generated_tokens'),llm_seconds=total('seconds'),
        llm_failed_calls=sum(e.get('error') is not None for e in events),call_count_definition='logical LM.forward attempts, including adapter retries; no judges')

def candidate_seed(cfg,kind,episode=None,candidate=None,question_id=None,split=None):
    key=[cfg['sampling_version'],cfg['seed'],kind]
    key += [episode,candidate] if kind=='training' else [split,question_id]
    return int.from_bytes(hashlib.sha256(json.dumps(key,separators=(',',':')).encode()).digest()[:8],'little') % (2**63-1)


def candidate_generator(session,kind,episode=None,candidate=None,question_id=None,split=None):
    seed=candidate_seed(session['config'],kind,episode,candidate,question_id,split)
    device=next(session['policy'].parameters()).device
    return torch.Generator(device=device).manual_seed(seed),seed


def recorded_attempt(session,example,kind,candidate=None,split=None):
    check_deadline(session)
    if session.get('progress'): print(f"{datetime.now().strftime('%H:%M:%S')} START {kind} ep={session['state']['episode']+(kind=='training')} candidate={candidate} q={example['id']}",flush=True)
    attempt_id=uuid.uuid4().hex; before=len(session['lm'].events)
    shared=dict(attempt_id=attempt_id,episode=session['state']['episode']+(1 if kind=='training' else 0),
                question_id=example['id'],candidate=candidate,split=split,evaluation_identity=session.get('evaluation_identity'))
    append_journal(session,dict(kind=kind+'_attempt_started',**shared))
    begin=time.perf_counter()
    try:
        generator,seed=candidate_generator(session,kind,shared['episode'],candidate,example['id'],split)
        trajectory=generate_pipeline(session['policy'],session['tokenizer'],session['first_tokens'],example['question'],session['config'],generator=generator)
        trajectory['sampling_seed']=seed
        trajectory['sampling_version']=session['config']['sampling_version']
        result=execute_pipeline(trajectory,example['question'],session['lm'])
        reward=compute_reward(example['question'],result['output'],example['answer'])
        item=dict(attempt_id=attempt_id,trajectory=trajectory,execution=result,reward=reward)
    except BaseException as exc:
        append_journal(session,dict(kind=kind+'_attempt_finished',**shared,error=f'{type(exc).__name__}: {exc}',
             resources=call_statistics(session['lm'].events[before:]),seconds=time.perf_counter()-begin))
        raise
    append_journal(session,dict(kind=kind+'_attempt_finished',**shared,**{k:v for k,v in item.items() if k!='attempt_id'},
             error=None,resources=call_statistics(session['lm'].events[before:]),seconds=time.perf_counter()-begin))
    return item

def collect_group(session,example,probe=None):
    budget=budget_snapshot(session); allowance=budget['allowance']; cfg=session['config']; before=time.perf_counter()
    probe=probe or probe_policy(session,example['question'])
    distribution=effective_policy_distribution(probe,session['first_tokens'])
    overhead=time.perf_counter()-before; items=[]; trace=[]
    target=min(cfg['probe_size'],allowance) if cfg['allocation_mode']=='adaptive' else allowance
    while True:
        while len(items)<target:
            items.append(recorded_attempt(session,example,'training',candidate=len(items)))
        begin=time.perf_counter()
        decision=controller_decision(cfg,[i['trajectory'] for i in items],[i['reward']['reward'] for i in items],
            session['first_tokens'],distribution['neff'],allowance)
        trace.append(decision); overhead+=time.perf_counter()-begin
        if decision['stop']: break
        target=min(allowance,len(items)+cfg['expansion_size'])
    return dict(trajectories=[i['trajectory'] for i in items],execution_results=[i['execution'] for i in items],
        reward_details=[i['reward'] for i in items],attempt_ids=[i['attempt_id'] for i in items],actual_k=len(items),
        stop_reason=trace[-1]['reason'],decision_trace=trace,effective_distribution=distribution,
        controller_seconds=overhead,allowance=allowance,bank=budget['bank'],gross_before=budget['gross_before'])


## Figure 2 update
No K scaling, reward standard-deviation normalization, clipping, or additional objectives. Sequential replay must match the no-gradient rollout probabilities. Finite equal-reward groups still perform Adam.step; momentum can move parameters.


In [ ]:
def group_advantages(rewards):
    values = np.asarray(rewards,dtype=np.float64)
    if len(values)==0 or not np.isfinite(values).all():
        raise ValueError("GRPO requires a nonempty finite group")
    return float(values.mean()), values-values.mean()

def figure2_loss(log_prob_trajectories, advantages):
    if len(log_prob_trajectories)!=len(advantages):
        raise ValueError("One advantage per candidate is required")
    return -sum(sum(log_probs)*float(advantage) for log_probs,advantage in zip(log_prob_trajectories,advantages))

def update_policy(session, prompt, trajectories, rewards):
    if session.get("evaluation_only",False):
        raise RuntimeError("Evaluation-only checkpoint: use explicit resume before updating")
    model, optimizer = session["policy"],session["optimizer"]
    cfg = session["config"]
    if len(trajectories)!=len(rewards) or len(rewards)<cfg["k_min"] or (cfg["allocation_mode"]=="static" and len(rewards)!=cfg["static_k"]) or (cfg["allocation_mode"]=="adaptive" and len(rewards)>cfg["k_max"]):
        raise ValueError("Invalid actual group size")
    mean, advantages = group_advantages(rewards)
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    model.eval()
    for trajectory, advantage in zip(trajectories,advantages):
        scores = replay_log_probs(model,session["tokenizer"],session["first_tokens"],prompt,trajectory,cfg)
        replay_values = [float(score.detach()) for score in scores]
        if not np.allclose(replay_values,trajectory["log_probs"],rtol=2e-5,atol=2e-5):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError("Replay differs from rollout; parameters/mode/state changed before update")
        loss = figure2_loss([scores],[advantage])
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite Figure 2 loss; no optimizer step permitted")
        total_loss += float(loss.detach())
        loss.backward()
        del loss, scores
    norm_sq = sum(float(parameter.grad.detach().double().square().sum())
                  for parameter in model.parameters() if parameter.grad is not None)
    norm = math.sqrt(norm_sq)
    if not math.isfinite(norm):
        optimizer.zero_grad(set_to_none=True)
        raise RuntimeError("Nonfinite gradient; no optimizer step permitted")
    optimizer.step()
    session.pop("loaded_checkpoint",None)
    optimizer.zero_grad(set_to_none=True)
    return dict(group_mean=mean,advantages=advantages.tolist(),loss=total_loss,gradient_norm=norm,
                zero_advantage_group=bool(np.all(advantages==0)),optimizer_step=True)


## Runs, resources, and durable checkpoints
Only the latest full checkpoint and validation-selected best policy are retained. Immutable generation files are published before pointer updates. Pruning is restricted to this newly created run. Checkpoint corruption is detected by SHA-256; local and cloud runs have separate runtime identities.


In [ ]:
def method_label(cfg):
    return 'adaptive' if cfg['allocation_mode']=='adaptive' else 'shuffled_k' if cfg['allocation_mode']=='scheduled' else f"static_k{cfg['static_k']}"

def initialize_experiment(cfg,create_run=True):
    cfg,datasets=load_gsm8k(cfg); data=prepare_selection(cfg,datasets,persist=create_run)
    lm,lm_metadata=initialize_ollama(cfg)
    model,tokenizer,optimizer,first_tokens,digest=initialize_policy(cfg)
    session=dict(config=cfg,data=data,lm=lm,llm_metadata=lm_metadata,policy=model,tokenizer=tokenizer,optimizer=optimizer,
        first_tokens=first_tokens,initial_policy_digest=digest,run_dir=None,in_episode=False,evaluation_only=False,
        state=dict(episode=0,updates=0,train=[],validation=[],seen=[],best=None,status='initialized',gross_training_attempts=0),
        deadline=time.perf_counter()+cfg['wall_limit_seconds'] if cfg['wall_limit_seconds'] else None)
    if create_run:
        root=Path(cfg['results_root'])/'gsm8k'/method_label(cfg)/f"seed{cfg['seed']}"/cfg['preset']/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+uuid.uuid4().hex[:8])
        root.mkdir(parents=True,exist_ok=False); (root/'checkpoints').mkdir(); session['run_dir']=root
        write_json(root/'config.json',dict(config=cfg,selection_id=data['selection_id'],selection=data['selection'],
            initial_policy_digest=digest,first_tokens=first_tokens,execution_model=lm_metadata,hardware=hardware_metadata()))
    return session

def gpu_statistics(model):
    cuda=next(model.parameters()).is_cuda
    return dict(peak_allocated_gpu_bytes=torch.cuda.max_memory_allocated() if cuda else None,
        peak_reserved_gpu_bytes=torch.cuda.max_memory_reserved() if cuda else None,
        process_rss_bytes=psutil.Process().memory_info().rss,system_available_ram_bytes=psutil.virtual_memory().available)

def file_digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def notebook_code_digest(path):
    notebook=json.loads(Path(path).read_text(encoding='utf-8'))
    return stable_hash([''.join(c['source']) if isinstance(c['source'],list) else c['source'] for c in notebook['cells'] if c['cell_type']=='code'])


def write_jsonl(path,rows):
    path=Path(path); tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with tmp.open('w',encoding='utf-8') as stream:
            for row in rows: stream.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n')
            stream.flush(); os.fsync(stream.fileno())
        os.replace(tmp,path)
    finally: tmp.unlink(missing_ok=True)

def synchronize_logs(session):
    write_jsonl(session['run_dir']/'training.jsonl',session['state']['train'])
    write_jsonl(session['run_dir']/'validation.jsonl',session['state']['validation'])

def runtime_identity(session):
    return dict(packages=VERSIONS,ollama_version=session['llm_metadata']['version'],
        execution_model_digest=session['llm_metadata']['model']['digest'],hardware=hardware_metadata()['gpu'],
        cuda=torch.version.cuda,python=platform.python_version())

def checkpoint_payload(session,policy_only=False):
    if session['in_episode']: raise RuntimeError('Cannot checkpoint an incomplete episode')
    if session.get('evaluation_only'): raise RuntimeError('Evaluation-only session cannot checkpoint')
    session['state']['gross_training_attempts']=journal_counts(session)
    result=dict(format_version=2,protocol=PROTOCOL,config=copy.deepcopy(session['config']),
        policy=session['policy'].state_dict(),state=copy.deepcopy(session['state']),rng=rng_state(),
        selection=copy.deepcopy(session['data']['selection']),selection_id=session['data']['selection_id'],
        initial_policy_digest=session['initial_policy_digest'],runtime=runtime_identity(session),policy_only=policy_only)
    if not policy_only: result['optimizer']=session['optimizer'].state_dict()
    return result

def publish_checkpoint(session,role,policy_only=False):
    root=Path(session['run_dir']); directory=root/'checkpoints'
    path=directory/(role+'_'+uuid.uuid4().hex+'.pt'); tmp=path.with_suffix('.tmp')
    try:
        with tmp.open('wb') as stream:
            torch.save(checkpoint_payload(session,policy_only),stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(tmp,path)
        pointer=dict(checkpoint=str(Path('checkpoints')/path.name),sha256=file_digest(path),episode=session['state']['episode'])
        write_json(root/(role+'.json'),pointer,overwrite=True)
        return pointer
    finally: tmp.unlink(missing_ok=True)

def save_checkpoint(session,validation=None):
    if validation is not None:
        old=session['state']['best']
        if old is None or validation['numeric_exact_match_accuracy']>old['score']:
            best=dict(episode=validation['episode'],score=validation['numeric_exact_match_accuracy'])
            session['state']['best']=best
            try: best['pointer']=publish_checkpoint(session,'best',policy_only=True)
            except BaseException:
                session['state']['best']=old; raise
    publish_checkpoint(session,'latest')
    # Both publications are durable. Preserve the references in latest state.
    keep={json.loads((session['run_dir']/'latest.json').read_text())['checkpoint']}
    if session['state']['best']: keep.add(session['state']['best']['pointer']['checkpoint'])
    for old in (session['run_dir']/'checkpoints').glob('*.pt'):
        if str(Path('checkpoints')/old.name) not in keep: old.unlink()
    synchronize_logs(session); write_summary(session)

def load_run(root,for_resume=True,role=None):
    root=Path(root); meta=json.loads((root/'config.json').read_text(encoding='utf-8')); cfg=meta['config']
    role=role or ('latest' if for_resume else 'best')
    pointer=json.loads((root/(role+'.json')).read_text(encoding='utf-8'))
    path=(root/pointer['checkpoint']).resolve()
    if path.parent!=(root/'checkpoints').resolve() or file_digest(path)!=pointer['sha256']: raise ValueError('Checkpoint path/checksum mismatch')
    payload=torch.load(path,map_location='cpu',weights_only=False) # own trusted checkpoints only
    if payload.get('format_version')!=2 or payload.get('protocol')!=PROTOCOL or payload['config']!=cfg: raise ValueError('Checkpoint protocol/configuration mismatch')
    if for_resume and (role!='latest' or payload['policy_only']): raise ValueError('Resume requires full latest checkpoint')
    if for_resume: repair_journal_tail(root)
    session=initialize_experiment(cfg,create_run=False); session['run_dir']=root
    try:
        if payload['runtime']!=runtime_identity(session): raise ValueError('Runtime/model changed; create a separate controlled run')
        if payload['selection']!=session['data']['selection'] or payload['selection_id']!=meta['selection_id']: raise ValueError('Selection/schedule mismatch')
        if payload['initial_policy_digest']!=session['initial_policy_digest']: raise ValueError('Pretrained policy differs')
        session['policy'].load_state_dict(payload['policy']); session['state']=payload['state']; session['evaluation_only']=not for_resume
        session['loaded_checkpoint']=dict(role=role,sha256=pointer['sha256'],episode=pointer['episode'])
        if not for_resume and session['state']['best'] is not None: session['state']['best']['pointer']=pointer
        if for_resume:
            if session['state']['episode']>=cfg['num_episodes'] and session['state']['status']=='completed': raise ValueError('Run already complete')
            session['optimizer'].load_state_dict(payload['optimizer']); restore_rng(payload['rng'])
            # Latest is authoritative after a crash between best/latest publications.
            durable_best=session['state']['best']
            if durable_best:
                bp=durable_best['pointer']; best_file=root/bp['checkpoint']
                if file_digest(best_file)!=bp['sha256']: raise ValueError('Durable best checkpoint corrupted')
                write_json(root/'best.json',bp,overwrite=True)
            else: (root/'best.json').unlink(missing_ok=True)
            session['state']['gross_training_attempts']=journal_counts(session)
            synchronize_logs(session)
            append_journal(session,dict(kind='resume',episode=session['state']['episode'],gross_training_attempts=journal_counts(session)))
        return session
    except BaseException:
        release_experiment(session); raise
    finally: del payload; gc.collect()


## Validation, training, and recovery
Every validation question uses a separate deterministic stream. A policy-weight digest, selection identity, execution digest and sampling protocol bind progress records. Finished journal outcomes survive a crash before progress publication. Different identities archive/reject reuse. Training candidates are never cached. Full official testing remains guarded and explicit.


In [ ]:
def evaluate_module(session,split='validation',episode=None,examples=None):
    if split not in ('validation','full_test'): raise ValueError('Unsupported evaluation split')
    if split=='full_test' and examples is None: raise ValueError('Explicit final test examples required')
    examples=examples if examples is not None else session['data']['subsets']['validation']
    train_ids=set(session['data']['selection']['ids']['train'])
    if not examples or len({e['id'] for e in examples})!=len(examples) or any(e['id'] in train_ids for e in examples): raise ValueError('Evaluation leakage/duplicates/empty')
    cfg=session['config']; ep=session['state']['episode'] if episode is None else episode
    identity=dict(protocol=PROTOCOL,policy_sha256=parameter_digest(session['policy']),episode=ep,
        selection_id=session['data']['selection_id'],sampling_version=cfg['sampling_version'],seed=cfg['seed'],
        execution_digest=session['llm_metadata']['model']['digest'],runtime=runtime_identity(session),
        execution_configuration={k:cfg[k] for k in ('reward_mode','ollama_model','ollama_num_gpu','ollama_num_ctx','execution_max_tokens','execution_temperature')},
        split=split,questions=stable_hash(examples))
    identity_id=stable_hash(identity); root=Path(session['run_dir']); pending=root/f'{split}_progress_ep{ep}.json'
    if pending.exists():
        old=json.loads(pending.read_text(encoding='utf-8'))
        if old['identity']!=identity:
            archive=root/(pending.stem+'_discarded_'+uuid.uuid4().hex[:8]+'.json')
            os.replace(pending,archive)
            append_journal(session,dict(kind='evaluation_reuse_rejected',episode=ep,old_identity=old['identity_id'],new_identity=identity_id,archive=archive.name))
    # The finished journal is authoritative even if a crash occurred just before progress publication.
    done={}; ids={e['id'] for e in examples}
    for e in read_jsonl(root/'attempts.jsonl'):
        if e['kind']=='evaluation_attempt_finished' and e.get('evaluation_identity')==identity_id and e.get('error') is None and e['question_id'] in ids:
            done.setdefault(e['question_id'],dict(question_id=e['question_id'],attempt_id=e['attempt_id'],trajectory=e['trajectory'],execution=e['execution'],reward=e['reward'],_resources=e['resources'],_seconds=e['seconds']))
    mode=session['policy'].training; session['policy'].eval(); session['evaluation_identity']=identity_id
    def publish():
        write_json(pending,dict(identity=identity,identity_id=identity_id,records=list(done.values()),
            evaluated_checkpoint=json.loads((root/'latest.json').read_text()) if (root/'latest.json').exists() else None),overwrite=True)
    publish()
    try:
        with torch.no_grad():
            for ex in examples:
                if ex['id'] in done: continue
                before=len(session['lm'].events); begin=time.perf_counter()
                item=recorded_attempt(session,ex,'evaluation',split=split)
                done[ex['id']]=dict(question_id=ex['id'],**item,_resources=call_statistics(session['lm'].events[before:]),_seconds=time.perf_counter()-begin)
                publish()
    finally:
        session.pop('evaluation_identity',None); session['policy'].train(mode)
    records=[dict(done[e['id']],target=e['answer']) for e in examples]
    resources=[r['_resources'] for r in records]
    def total(key):
        values=[r.get(key) for r in resources]; return sum(values) if all(v is not None for v in values) else None
    return dict(split=split,episode=ep,identity=identity,identity_id=identity_id,
        question_ids=[r['question_id'] for r in records],questions=len(records),records=records,
        numeric_exact_match_accuracy=sum(r['reward']['numeric_exact_match'] for r in records)/len(records),
        attempted_pipeline_evaluations=len(records),seconds=sum(r['_seconds'] for r in records),
        time_scope='committed candidate execution incl sampling; discarded work reported in gross ledger',
        cumulative_training_pipeline_evaluations=sum(r['actual_k'] for r in session['state']['train']),
        gross_training_attempts=journal_counts(session),selected_checkpoint=session.get('loaded_checkpoint'),
        execution_llm_calls=total('execution_llm_calls'),physical_http_requests=None,prompt_tokens=total('prompt_tokens'),
        generated_tokens=total('generated_tokens'),llm_seconds=total('llm_seconds'),llm_failed_calls=total('llm_failed_calls'),
        call_count_definition='logical LM.forward attempts for committed evaluation records')

def train_grpo(session):
    if session.get('evaluation_only'): raise RuntimeError('Cannot train evaluation-only checkpoint')
    if session['in_episode']: raise RuntimeError('Restore latest checkpoint before continuing interrupted episode')
    cfg,state=session['config'],session['state']
    if state['status']=='completed': raise RuntimeError('Run already completed')
    state['status']='running'
    try:
        while True:
            ep=state['episode']
            should_validate=ep==0 or ep%cfg['validation_interval']==0 or ep==cfg['num_episodes']
            if should_validate and ep not in {v['episode'] for v in state['validation']}:
                if not (session['run_dir']/'latest.json').exists() or json.loads((session['run_dir']/'latest.json').read_text())['episode']!=ep: save_checkpoint(session)
                validation=evaluate_module(session,episode=ep)
                state['validation'].append(validation)
                save_checkpoint(session,validation)
            elif not (session['run_dir']/'latest.json').exists(): save_checkpoint(session)
            if ep==cfg['num_episodes']:
                state['status']='completed'; save_checkpoint(session); break
            check_deadline(session); episode_allowance(session)
            ex=session['data']['subsets']['train'][session['data']['selection']['episode_pool_offsets'][ep]]
            session['in_episode']=True
            if next(session['policy'].parameters()).is_cuda: torch.cuda.reset_peak_memory_stats()
            begin=time.perf_counter(); start=len(session['lm'].events)
            before=probe_policy(session,ex['question'])
            group=collect_group(session,ex,before)
            rewards=[r['reward'] for r in group['reward_details']]
            update=update_policy(session,ex['question'],group['trajectories'],rewards)
            after=probe_policy(session,ex['question'])
            seen=set(state['seen']); seen.add(ex['id']); actual=group['actual_k']
            row=dict(episode=ep+1,question_id=ex['id'],target=ex['answer'],
                question_pool_offset=session['data']['selection']['episode_pool_offsets'][ep],
                actual_k=actual,attempted_pipeline_evaluations=actual,generated_candidates=actual,replayed_candidates=actual,
                actions=[t['actions'] for t in group['trajectories']],rewards=rewards,
                unique_pipeline_count=len({tuple(t['actions']) for t in group['trajectories']}),
                attempt_ids=group['attempt_ids'],candidate_details=[dict(trajectory=t,execution=e,reward=r) for t,e,r in
                    zip(group['trajectories'],group['execution_results'],group['reward_details'])],
                controller_decisions=group['decision_trace'],stop_reason=group['stop_reason'],allowance=group['allowance'],
                effective_policy_classes=group['effective_distribution'],controller_seconds=group['controller_seconds'],bank=group['bank'],gross_before=group['gross_before'],
                policy_diagnostics=dict(before_update=before,after_update=after),policy_change=policy_change(before,after,session['first_tokens']),
                successful_pipeline_evaluations=sum(e['success'] for e in group['execution_results']),
                failed_pipeline_evaluations=sum(not e['success'] for e in group['execution_results']),
                unique_training_questions=len(seen),optimizer_update_count=state['updates']+1,
                gross_training_attempts=journal_counts(session),episode_seconds=time.perf_counter()-begin,
                pipeline_execution_seconds=sum(e['seconds'] for e in group['execution_results']),
                ollama_memory=ollama_snapshot(cfg),
                **update,**call_statistics(session['lm'].events[start:]),**gpu_statistics(session['policy']))
            state['train'].append(row); state['episode']=ep+1; state['updates']+=1; state['seen']=sorted(seen)
            state['gross_training_attempts']=journal_counts(session); session['in_episode']=False
            synchronize_logs(session)
            if state['episode']%cfg['checkpoint_interval']==0: save_checkpoint(session)
            write_summary(session)
            print(f"{method_label(cfg)} episode={state['episode']}/{cfg['num_episodes']} K={actual} stop={row['stop_reason']} reward={update['group_mean']:.3f} grad={update['gradient_norm']:.3e}",flush=True)
    except BaseException as exc:
        status='budget_exhausted' if isinstance(exc,BudgetExhausted) else 'wall_limit' if isinstance(exc,WallLimitReached) else 'infrastructure_failure' if isinstance(exc,InfrastructureFailure) or infrastructure_error(exc) else 'interrupted'
        state['status']=status
        append_journal(session,dict(kind='interruption',episode=state['episode'],in_episode=session['in_episode'],
            error=f'{type(exc).__name__}: {exc}',status=status,gross_training_attempts=journal_counts(session),
            failure_memory=gpu_statistics(session['policy'])))
        if not session['in_episode']: save_checkpoint(session)
        else: write_summary(session)
        if status not in ('budget_exhausted','wall_limit','infrastructure_failure'): raise
        print(status, str(exc), 'Resume latest durable checkpoint; partial group is discarded.',flush=True)
    return session

_train_grpo_implementation=train_grpo

def train_grpo(session):
    segment=uuid.uuid4().hex; started=time.perf_counter()
    append_journal(session,dict(kind='training_segment_started',segment_id=segment))
    try: return _train_grpo_implementation(session)
    finally:
        append_journal(session,dict(kind='training_segment_finished',segment_id=segment,seconds=time.perf_counter()-started))
        write_summary(session)

def journal_resource_totals(root):
    events=read_jsonl(Path(root)/'attempts.jsonl')
    starts={e['attempt_id']:e for e in events if e['kind'].endswith('_attempt_started')}
    finishes={e['attempt_id']:e for e in events if e['kind'].endswith('_attempt_finished')}
    complete=set(starts)==set(finishes)
    values=[f['resources'] for f in finishes.values()]
    segments={e['segment_id'] for e in events if e['kind']=='training_segment_started'}
    ended={e['segment_id']:e['seconds'] for e in events if e['kind']=='training_segment_finished'}
    segment_seconds=sum(ended.values()) if segments==set(ended) else None
    training_finishes=[e for e in finishes.values() if e['kind']=='training_attempt_finished']
    evaluation_finishes=[e for e in finishes.values() if e['kind']=='evaluation_attempt_finished']
    def total(key):
        a=[r.get(key) for r in values]
        return sum(a) if complete and all(v is not None for v in a) else None
    return dict(gross_training_attempts=sum(e['kind']=='training_attempt_started' for e in starts.values()),
        gross_evaluation_attempts=sum(e['kind']=='evaluation_attempt_started' for e in starts.values()),
        training_session_wall_seconds=segment_seconds,
        wall_scope='training/resume segments include validation, checkpoint I/O and discarded work; excludes model setup and standalone final testing',
        gross_training_logical_calls=sum(e['resources']['execution_llm_calls'] for e in training_finishes) if complete else None,
        gross_evaluation_logical_calls=sum(e['resources']['execution_llm_calls'] for e in evaluation_finishes) if complete else None,
        observed_training_attempt_seconds=sum(e['seconds'] for e in training_finishes),
        journal_resources_complete=complete,observed_llm_calls=sum(r['execution_llm_calls'] for r in values),
        total_llm_calls=total('execution_llm_calls'),total_prompt_tokens=total('prompt_tokens'),
        total_generated_tokens=total('generated_tokens'),total_llm_seconds=total('llm_seconds'),physical_http_requests=None)

def experiment_summary(session):
    cfg,state=session['config'],session['state']; rows=state['train']; root=session['run_dir']
    gross=journal_resource_totals(root); committed=sum(r['actual_k'] for r in rows)
    pointer=json.loads((root/'latest.json').read_text(encoding='utf-8')) if (root/'latest.json').exists() else {}
    tests=[json.loads(p.read_text(encoding='utf-8')) for p in root.glob('full_test_*.json')]
    def total(key):
        values=[r.get(key) for r in rows]
        return sum(values) if all(v is not None for v in values) else None
    completed=state['status']=='completed' and state['episode']==cfg['num_episodes'] and pointer.get('episode')==state['episode'] and any(v['episode']==state['episode'] for v in state['validation'])
    evaluation_ids={r['attempt_id'] for v in state['validation'] for r in v['records']}
    evaluation_ids.update(r['attempt_id'] for v in tests for r in v['records'])
    events=read_jsonl(root/'attempts.jsonl'); estarts=[e for e in events if e['kind']=='evaluation_attempt_started']; eends=[e for e in events if e['kind']=='evaluation_attempt_finished']
    counts=Counter((e.get('evaluation_identity'),e['question_id']) for e in estarts)
    return dict(committed_evaluation_attempts=len(evaluation_ids),uncommitted_evaluation_attempts=len(estarts)-len(evaluation_ids),
        interrupted_evaluation_attempts=len(estarts)-len(eends),repeated_evaluation_attempts=sum(max(0,v-1) for v in counts.values()),
        protocol=PROTOCOL,config=cfg,method=method_label(cfg),selection_id=session['data']['selection_id'],
        initial_policy_digest=session['initial_policy_digest'],runtime_identity=runtime_identity(session),
        status=state['status'],completed=completed,episode=state['episode'],durable_episode=pointer.get('episode',0),
        optimizer_updates=state['updates'],best=state['best'],committed_training_attempts=committed,
        discarded_training_attempts=gross['gross_training_attempts']-committed,actual_k=[r['actual_k'] for r in rows],
        stop_reasons=dict(Counter(r['stop_reason'] for r in rows)),training_seconds=total('episode_seconds'),
        controller_seconds=total('controller_seconds'),training_llm_calls=total('execution_llm_calls'),
        training_prompt_tokens=total('prompt_tokens'),training_generated_tokens=total('generated_tokens'),
        validation_seconds=sum(v['seconds'] for v in state['validation']),
        evaluation_seconds=sum(v['seconds'] for v in state['validation'])+sum(v['seconds'] for v in tests),
        zero_advantage_groups=sum(r['zero_advantage_group'] for r in rows),
        informative_gradient_episodes=sum(r['gradient_norm']>1e-8 for r in rows),
        peak_allocated_gpu_bytes=max((r['peak_allocated_gpu_bytes'] or 0 for r in rows),default=0) if torch.cuda.is_available() else None,
        peak_process_rss_snapshot_bytes=max((r['process_rss_bytes'] for r in rows),default=None),
        final_held_out_accuracy=tests[0]['numeric_exact_match_accuracy'] if len(tests)==1 else None,
        test_count=len(tests),**gross)

def write_summary(session):
    result=experiment_summary(session); write_json(session['run_dir']/'summary.json',result,overwrite=True); return result

def test_model(session):
    if not session.get('evaluation_only') or session.get('loaded_checkpoint',{}).get('role')!='best':
        raise RuntimeError('Explicitly load validation-selected best policy before testing')
    summary=json.loads((session['run_dir']/'summary.json').read_text(encoding='utf-8'))
    if not summary['completed'] or session['config']['preset']!='formal': raise RuntimeError('Final testing requires completed formal run')
    if list(session['run_dir'].glob('full_test_*.json')): raise RuntimeError('Final test already recorded; no best-score selection')
    cfg=session['config']
    test=load_dataset(cfg['dataset_name'],cfg['dataset_config'],revision=cfg['dataset_revision'],split='test')
    if len(test)!=1319: raise ValueError('Unexpected official GSM8K test size; inspect protocol')
    examples=[dataset_example({'test':test},'test',i) for i in range(len(test))]
    result=evaluate_module(session,'full_test',examples=examples)
    result.update(selection_rule='maximum validation numeric EM, earliest tie',protocol=PROTOCOL,
        test_content_hash=stable_hash(examples))
    write_json(session['run_dir']/('full_test_'+uuid.uuid4().hex[:8]+'.json'),result)
    # Evaluation session contains best-epoch history; preserve final training summary.
    summary.update(final_held_out_accuracy=result['numeric_exact_match_accuracy'],test_count=1,
        evaluation_seconds=summary['validation_seconds']+result['seconds'],**journal_resource_totals(session['run_dir']))
    write_json(session['run_dir']/'summary.json',summary,overwrite=True)
    return result


def policy_change(before,after,first_tokens):
    p={tuple(c['key']):c['probability'] for c in effective_policy_distribution(before,first_tokens)['classes']}
    q={tuple(c['key']):c['probability'] for c in effective_policy_distribution(after,first_tokens)['classes']}
    return dict(total_variation=.5*sum(abs(p[k]-q[k]) for k in p),
        forward_kl=sum(v*math.log(max(v,1e-300)/max(q[k],1e-300)) for k,v in p.items() if v>0),
        scope='current question effective action probabilities; not full parameter norm')


## Artifact integrity, matched comparisons, and formal acceptance
Compare new numeric-reward runs only. Every completed episode must be backed by started/finished journal IDs and centered advantages. Formal acceptance requires all three seeds and all four methods; final tests are paired by question IDs. Bootstrap resamples both seeds and questions with the same draws across arms.


In [ ]:
def verify_run_artifacts(root,summary=None):
    root=Path(root); summary=summary or json.loads((root/'summary.json').read_text(encoding='utf-8'))
    meta=json.loads((root/'config.json').read_text(encoding='utf-8')); cfg=summary['config']
    def require(ok,message):
        if not ok: raise ValueError(f'Invalid artifacts in {root}: {message}')
    require(cfg==meta['config'] and cfg['protocol']==PROTOCOL,'protocol/config snapshot')
    require(summary['completed'] and summary['episode']==summary['durable_episode']==cfg['num_episodes'],'incomplete run')
    require(stable_hash(meta['selection'])==summary['selection_id']==meta['selection_id'],'selection digest')
    require(summary['initial_policy_digest']==meta['initial_policy_digest'],'initial policy digest')
    require(summary['runtime_identity']['packages']==meta['hardware']['packages'] and summary['runtime_identity']['execution_model_digest']==meta['execution_model']['model']['digest'],'runtime snapshot identity')
    rows=read_jsonl(root/'training.jsonl'); vals=read_jsonl(root/'validation.jsonl'); journal=read_jsonl(root/'attempts.jsonl')
    if cfg['allocation_mode']=='scheduled' and cfg['preset']=='formal':
        source=json.loads((root/'schedule_source.json').read_text())
        source_summary=json.loads((Path(source['adaptive_run'])/'summary.json').read_text())
        require(source_summary['completed'] and source_summary['method']=='adaptive' and source_summary['config']['seed']==cfg['seed'],'shuffled source completed/seed')
        require(source_summary['actual_k']==source['source_k'] and sorted(source['source_k'])==sorted(cfg['allocation_schedule']),'shuffled K multiset')
        require(np.random.default_rng(source['permutation_seed']).permutation(source['source_k']).tolist()==cfg['allocation_schedule']==source['shuffled_k'],'shuffled permutation')
    starts={e['attempt_id']:e for e in journal if e['kind']=='training_attempt_started'}
    finishes={e['attempt_id']:e for e in journal if e['kind']=='training_attempt_finished'}
    require(len(starts)==sum(e['kind']=='training_attempt_started' for e in journal),'duplicate attempt IDs')
    require([r['episode'] for r in rows]==list(range(1,cfg['num_episodes']+1)),'episode sequence')
    committed=set(); seen=set(); cumulative=0
    for r in rows:
        k=r['actual_k']; require(cfg['k_min']<=k<=cfg['k_max'] if cfg['allocation_mode']=='adaptive' else k==cfg['allocation_schedule'][r['episode']-1] if cfg['allocation_mode']=='scheduled' else k==cfg['static_k'],'group bound')
        require(k==r['attempted_pipeline_evaluations']==r['generated_candidates']==r['replayed_candidates'],'candidate totals')
        require(len(r['rewards'])==len(r['advantages'])==len(r['candidate_details'])==len(r['attempt_ids'])==k,'candidate arrays')
        require(r['successful_pipeline_evaluations']+r['failed_pipeline_evaluations']==k,'candidate outcome totals')
        mean,adv=group_advantages(r['rewards']); require(math.isclose(mean,r['group_mean']) and np.allclose(adv,r['advantages']),'centered advantages')
        require(all(x in (0.,1.) for x in r['rewards']),'binary reward')
        require(r['zero_advantage_group']==bool(np.all(adv==0)),'zero-advantage flag')
        require(math.isfinite(r['loss']) and math.isfinite(r['gradient_norm']),'finite update')
        for aid,detail,reward in zip(r['attempt_ids'],r['candidate_details'],r['rewards']):
            require(aid in starts and aid in finishes and finishes[aid].get('error') is None and aid not in committed,'journal provenance')
            require(starts[aid]['episode']==r['episode'] and starts[aid]['question_id']==r['question_id'],'journal question')
            require(detail['reward']==finishes[aid]['reward'] and detail['trajectory']==finishes[aid]['trajectory'],'journal candidate')
            require(detail['trajectory']['sampling_seed']==candidate_seed(cfg,'training',r['episode'],r['attempt_ids'].index(aid)),'training sampling seed')
            require(compute_reward('',detail['execution']['output'],r['target'])['reward']==reward,'numeric reward recomputation')
            committed.add(aid)
        offset=meta['selection']['episode_pool_offsets'][r['episode']-1]
        require(offset==r['question_pool_offset'] and meta['selection']['ids']['train'][offset]==r['question_id'],'episode schedule')
        seen.add(r['question_id']); require(r['unique_training_questions']==len(seen),'unique questions')
        require(r['optimizer_update_count']==r['episode'] and r['optimizer_step'],'one optimizer step per episode')
        require(r['controller_decisions'][-1]['stop'] and r['controller_decisions'][-1]['reason']==r['stop_reason'],'stop trace')
        if 'first_tokens' in meta:
            distribution=effective_policy_distribution(r['policy_diagnostics']['before_update'],meta['first_tokens'])
            require(math.isclose(distribution['neff'],r['effective_policy_classes']['neff']),'effective policy diversity')
            before_count=list(starts).index(r['attempt_ids'][0])
            reserve=(cfg['num_episodes']-r['episode'])*(cfg['k_min'] if cfg['allocation_mode']=='adaptive' else cfg['static_k'])
            bank=max(0,cfg['adaptive_budget_per_episode']*(r['episode']-1)-before_count) if cfg['allocation_mode']=='adaptive' else 0
            expected_allowance=min(cfg['k_max'],cfg['training_budget']-before_count-reserve,4+bank) if cfg['allocation_mode']=='adaptive' else cfg['allocation_schedule'][r['episode']-1] if cfg['allocation_mode']=='scheduled' else cfg['static_k']
            require(r['bank']==bank and r['gross_before']==before_count,'bank/gross provenance')
            require(r['allowance']==expected_allowance,'reserved episode allowance')
            for index,decision in enumerate(r['controller_decisions']):
                prefix=decision['actual_k']
                expected_decision=controller_decision(cfg,[d['trajectory'] for d in r['candidate_details'][:prefix]],r['rewards'][:prefix],meta['first_tokens'],distribution['neff'],r['allowance'])
                require(decision==expected_decision,'controller decision recomputation')
                require(decision['stop']==(index==len(r['controller_decisions'])-1),'stopping at first decision')
        elif cfg['preset']=='formal': require(False,'formal controller provenance missing')
        cumulative+=k
    require(cumulative==summary['committed_training_attempts'] and len(starts)==summary['gross_training_attempts'],'gross/committed totals')
    require(len(starts)<=cfg['training_budget'] and summary['discarded_training_attempts']==len(starts)-cumulative,'gross budget')
    expected=[0]+[e for e in range(1,cfg['num_episodes']+1) if e%cfg['validation_interval']==0 or e==cfg['num_episodes']]
    require([v['episode'] for v in vals]==expected,'validation schedule')
    for v in vals:
        require(v['question_ids']==meta['selection']['ids']['validation'],'validation identity')
        require(not set(v['question_ids']) & set(meta['selection']['ids']['train']),'validation leakage')
        require(v['questions']==v['attempted_pipeline_evaluations']==cfg['validation_size'],'validation size')
        require(v['cumulative_training_pipeline_evaluations']==sum(r['actual_k'] for r in rows if r['episode']<=v['episode']),'variable-K validation budget')
        require(all(compare_answers(r['execution']['output'],r['target'])==r['reward']['numeric_exact_match'] for r in v['records']),'validation reward recomputation')
        require(v['numeric_exact_match_accuracy']==sum(r['reward']['numeric_exact_match'] for r in v['records'])/v['questions'],'validation score')
        all_finished={e['attempt_id']:e for e in journal if e['kind']=='evaluation_attempt_finished'}
        require(v['identity_id']==stable_hash(v['identity']),'evaluation identity hash')
        require(v['identity']['selection_id']==summary['selection_id'] and v['identity']['sampling_version']==cfg['sampling_version'],'evaluation identity configuration')
        for record in v['records']:
            event=all_finished.get(record['attempt_id'],{})
            require(event.get('evaluation_identity')==v['identity_id'] and event.get('trajectory')==record['trajectory'] and event.get('reward')==record['reward'],'evaluation journal provenance')
            require(record['trajectory']['sampling_seed']==candidate_seed(cfg,'evaluation',question_id=record['question_id'],split='validation'),'evaluation sampling seed')
    best=max(vals,key=lambda v:(v['numeric_exact_match_accuracy'],-v['episode']))
    require(summary['best']['episode']==best['episode'] and summary['best']['score']==best['numeric_exact_match_accuracy'],'best checkpoint selection')
    for role in ('best','latest'):
        pointer=json.loads((root/(role+'.json')).read_text(encoding='utf-8')); path=(root/pointer['checkpoint']).resolve()
        require(path.parent==(root/'checkpoints').resolve() and file_digest(path)==pointer['sha256'],role+' checkpoint checksum')
        require(pointer['episode']==(best['episode'] if role=='best' else cfg['num_episodes']),role+' checkpoint episode')
    tests=[json.loads(p.read_text(encoding='utf-8')) for p in root.glob('full_test_*.json')]
    require(len(tests)<=1,'repeated final tests')
    for t in tests:
        bp=json.loads((root/'best.json').read_text(encoding='utf-8'))
        require(t['selected_checkpoint']==dict(role='best',sha256=bp['sha256'],episode=bp['episode']),'test checkpoint')
        require(t['questions']==len(set(t['question_ids']))==1319 and t['split']=='full_test','full official test identity')
        require(t['question_ids']==[f'gsm8k:test:{i}' for i in range(1319)],'official test question IDs')
        require(all(compare_answers(r['execution']['output'],r['target'])==r['reward']['numeric_exact_match'] for r in t['records']),'test reward recomputation')
        require(t['numeric_exact_match_accuracy']==sum(r['reward']['numeric_exact_match'] for r in t['records'])/1319,'test accuracy')
        require(summary['final_held_out_accuracy']==t['numeric_exact_match_accuracy'],'test summary')
    return dict(root=str(root),summary=summary,train=rows,validation=vals,tests=tests,metadata=meta)

def compare_runs(run_dirs):
    roots=[Path(p).resolve() for p in run_dirs]
    if not roots or len(roots)!=len(set(roots)): raise ValueError('Provide unique explicit run directories')
    runs=[verify_run_artifacts(root) for root in roots]; reference=None; selections={}; membership=None
    ignored={'allocation_mode','static_k','training_budget','results_root','seed','allocation_schedule'}
    for run in runs:
        s=run['summary']; cfg=s['config']; meta=run['metadata']; seed=cfg['seed']
        identity=dict(config={k:v for k,v in cfg.items() if k not in ignored},runtime=s['runtime_identity'],initial=s['initial_policy_digest'])
        if reference is not None and identity!=reference: raise ValueError('Incompatible non-allocation protocol/runtime')
        reference=identity
        if seed in selections and selections[seed]!=s['selection_id']: raise ValueError('Per-seed schedule differs')
        selections[seed]=s['selection_id']
        members=(meta['selection']['ids'],meta['selection']['content_hashes'])
        if membership is not None and members!=membership: raise ValueError('Dataset membership differs across seeds')
        membership=members
    return sorted(runs,key=lambda r:(r['summary']['config']['seed'],r['summary']['method']))

def paired_hierarchical_bootstrap(delta,resamples=10000,seed=2026):
    delta=np.asarray(delta,dtype=np.float64)
    if delta.ndim!=2 or delta.shape[0]<2 or not delta.shape[1] or not np.isfinite(delta).all():
        raise ValueError('Need finite paired seed-by-question differences')
    rng=np.random.default_rng(seed); boot=np.empty(resamples)
    for i in range(resamples):
        si=rng.integers(0,delta.shape[0],size=delta.shape[0]); qi=rng.integers(0,delta.shape[1],size=delta.shape[1])
        boot[i]=delta[np.ix_(si,qi)].mean()
    return np.quantile(boot,[.025,.975]).tolist()

def acceptance_report(runs,resamples=10000,bootstrap_seed=2026):
    runs=compare_runs([r['root'] for r in runs])  # acceptance always revalidates source artifacts
    if resamples!=10000: raise ValueError('Formal protocol requires 10,000 bootstrap resamples')
    lookup={}
    for run in runs:
        s=run['summary']; key=(s['config']['seed'],s['method'])
        if key in lookup: raise ValueError('Repeated run for seed/method; report separately')
        if s['config']['preset']!='formal' or s['config']['num_episodes']!=200 or s['config']['train_pool_size']!=200 or s['config']['validation_size']!=50 or not s['completed'] or len(run['tests'])!=1: raise ValueError('Completed formal runs and explicit full tests required')
        lookup[key]=run
    seeds=[42,43,44]; methods=['adaptive','static_k2','static_k4','static_k5','shuffled_k']
    if set(lookup)!={(s,m) for s in seeds for m in methods}: raise ValueError('Require all five methods at seeds 42,43,44')
    arrays={}
    test_hash=None
    for method in methods:
        values=[]
        for seed in seeds:
            t=lookup[(seed,method)]['tests'][0]
            if test_hash is not None and test_hash!=t['test_content_hash']: raise ValueError('Test content changed')
            test_hash=t['test_content_hash']
            values.append([r['reward']['numeric_exact_match'] for r in t['records']])
        arrays[method]=np.asarray(values,dtype=np.float64)
    delta=arrays['adaptive']-arrays['static_k5']
    low,high=paired_hierarchical_bootstrap(delta,resamples,bootstrap_seed); costs={m:sum(lookup[(s,m)]['summary']['gross_training_attempts'] for s in seeds) for m in methods}
    saving=1-costs['adaptive']/costs['static_k5']; accuracy={m:float(a.mean()) for m,a in arrays.items()}
    limitations=[f'{m} achieves equal/higher observed accuracy for equal/lower cost; adaptive allocation benefit not established' for m in ('static_k2','static_k4','shuffled_k') if costs[m]<=costs['adaptive'] and accuracy[m]>=accuracy['adaptive']]
    return dict(target_demonstrated=bool(saving>=.20-1e-12 and low>=-.02),
        interpretation='target demonstrated under prespecified bootstrap' if saving>=.20-1e-12 and low>=-.02 else 'target not demonstrated; may be inconclusive or inferior',
        accuracy=accuracy,gross_training_attempts=costs,pipeline_saving=saving,accuracy_difference=float(delta.mean()),
        paired_95_interval=[float(low),float(high)],bootstrap_resamples=resamples,seeds=seeds,limitations=limitations,
        caveat='Three training seeds give limited uncertainty resolution; no guaranteed future performance.')

def plot_comparison(runs,save_dir=None):
    fig,axes=plt.subplots(2,3,figsize=(16,8))
    for run in runs:
        s=run['summary']; label=f"{s['method']} seed {s['config']['seed']}"; rows=run['train']; vals=run['validation']
        axes[0,0].plot([r['episode'] for r in rows],[r['actual_k'] for r in rows],label=label)
        axes[0,1].plot([v['cumulative_training_pipeline_evaluations'] for v in vals],[v['numeric_exact_match_accuracy'] for v in vals],marker='o',label=label)
        axes[0,2].plot([r['episode'] for r in rows],[r['gradient_norm'] for r in rows],label=label)
        axes[1,0].bar(label,s['gross_training_attempts']); axes[1,1].bar(label,s['training_seconds'])
        for reason,count in s['stop_reasons'].items(): axes[1,2].scatter(label+'/'+reason,count)
    for ax,title in zip(axes.flat,('Actual K by episode','Validation EM by committed training budget','Gradient norm (sum loss)','Gross training attempts','Training seconds','Controller stop counts')):
        ax.set_title(title)
    axes[0,0].legend(fontsize=7); axes[0,1].set_xlabel('Actual committed pipeline evaluations')
    for ax in axes[1]: ax.tick_params(axis='x',rotation=30)
    fig.tight_layout()
    if save_dir is not None:
        root=Path(save_dir); root.mkdir(parents=True,exist_ok=True); fig.savefig(root/'adagrpo_comparison.png',dpi=160)
    return fig


## Explicit local study and screening gate
`run_improvement_suite()` runs development smoke followed by fresh seed-43 static5/adaptive/static4 arms. A single immutable 120-minute deadline covers all new model work. Explicit resume never resets it. A partial study is incomplete. One frozen candidate only; no automatic tuning or cloud launch. Use `resume=True` only for the existing ledger.


In [ ]:
DEVELOPMENT_POOL = [3082, 2184, 5897, 2437, 6330, 819, 3812, 5878, 5302, 5876, 5308, 1534, 2663, 4247, 4632, 1727, 1601, 3848, 6009, 1948, 56, 5079, 3115, 289, 5710, 3693, 2271, 5468, 1789, 3703, 4306, 6042, 2502, 4423, 6851, 6195, 4389, 1483, 4079, 2727, 7079, 963, 3418, 6491, 4977, 1837, 2068, 7114, 2477, 2428]
DEVELOPMENT_REVISION = '740312add88f781978c0658806c59bc2815b9866'
DEVELOPMENT_SOURCE_SHA256 = '19cd17290166275ae79d6e4467e4c9af19589b131d2c71bc76492dd577efe2c7'
HISTORICAL_EXCLUDED_IDS = ['gsm8k:train:130', 'gsm8k:train:1332', 'gsm8k:train:1483', 'gsm8k:train:1534', 'gsm8k:train:1601', 'gsm8k:train:1727', 'gsm8k:train:1789', 'gsm8k:train:1837', 'gsm8k:train:1890', 'gsm8k:train:1938', 'gsm8k:train:1948', 'gsm8k:train:2068', 'gsm8k:train:2126', 'gsm8k:train:2184', 'gsm8k:train:2271', 'gsm8k:train:2428', 'gsm8k:train:2437', 'gsm8k:train:2477', 'gsm8k:train:2502', 'gsm8k:train:2663', 'gsm8k:train:2691', 'gsm8k:train:2727', 'gsm8k:train:2857', 'gsm8k:train:289', 'gsm8k:train:3082', 'gsm8k:train:3115', 'gsm8k:train:3418', 'gsm8k:train:3693', 'gsm8k:train:3703', 'gsm8k:train:3812', 'gsm8k:train:3848', 'gsm8k:train:4079', 'gsm8k:train:4247', 'gsm8k:train:4306', 'gsm8k:train:4389', 'gsm8k:train:4423', 'gsm8k:train:4632', 'gsm8k:train:4943', 'gsm8k:train:4977', 'gsm8k:train:5079', 'gsm8k:train:5196', 'gsm8k:train:5302', 'gsm8k:train:5308', 'gsm8k:train:5429', 'gsm8k:train:5468', 'gsm8k:train:5514', 'gsm8k:train:5517', 'gsm8k:train:5560', 'gsm8k:train:56', 'gsm8k:train:5710', 'gsm8k:train:5809', 'gsm8k:train:5876', 'gsm8k:train:5878', 'gsm8k:train:5897', 'gsm8k:train:5941', 'gsm8k:train:6009', 'gsm8k:train:6042', 'gsm8k:train:6118', 'gsm8k:train:6195', 'gsm8k:train:623', 'gsm8k:train:6330', 'gsm8k:train:6491', 'gsm8k:train:6588', 'gsm8k:train:6675', 'gsm8k:train:6851', 'gsm8k:train:7079', 'gsm8k:train:7114', 'gsm8k:train:819', 'gsm8k:train:907', 'gsm8k:train:963']

def preflight(cfg=None):
    cfg=cfg or build_config(); version=json_request(cfg['ollama_host'],'/api/version')
    tags=json_request(cfg['ollama_host'],'/api/tags')['models']
    model=next((m for m in tags if m.get('name')==cfg['ollama_model']),None)
    if model is None: raise RuntimeError('Install the selected model explicitly: ollama pull '+cfg['ollama_model'])
    if cfg['policy_device']=='cuda' and not torch.cuda.is_available(): raise RuntimeError('CUDA PyTorch unavailable')
    free=shutil.disk_usage(Path.cwd()).free
    if free<6*1024**3: raise RuntimeError('Need at least 6 GB free for local checkpoints')
    return dict(hardware=hardware_metadata(),ollama=version,execution_model=model,free_disk_bytes=free)

def historical_membership(results_root='results'):
    used=set(HISTORICAL_EXCLUDED_IDS)
    def visit(value):
        if isinstance(value,str) and re.fullmatch(r'gsm8k:train:\d+',value): used.add(value)
        elif isinstance(value,dict):
            for v in value.values(): visit(v)
        elif isinstance(value,list):
            for v in value: visit(v)
    root=Path(results_root)
    sources=sorted(set(root.rglob('config.json'))|set(root.glob('**/selections/*.json'))|set(root.glob('**/membership/*.json')))
    for path in sources: visit(json.loads(path.read_text(encoding='utf-8')))
    return used


def freeze_membership(cfg,purpose='screen'):
    if purpose not in ('screen','formal'): raise ValueError('Invalid membership purpose')
    path=Path(cfg['results_root'])/'membership'/(purpose+'_v2.json')
    if path.exists(): return str(path)
    cfg,datasets=load_gsm8k(cfg); train=datasets['train']
    if cfg['dataset_revision']!=DEVELOPMENT_REVISION: raise ValueError('Pinned development dataset revision differs')
    pool=list(DEVELOPMENT_POOL)
    excluded=historical_membership(); blocked={int(q.rsplit(':',1)[1]) for q in excluded}
    order=np.random.default_rng(20261020).permutation(len(train)).tolist()
    if purpose=='formal': pool += [i for i in order if i not in blocked and i not in pool][:150]
    validation=[i for i in order if i not in blocked and i not in pool][:20 if purpose=='screen' else 50]
    indices=dict(train=pool,validation=validation)
    subsets={k:[dataset_example(datasets,'train',i) for i in ids] for k,ids in indices.items()}
    if len(validation)!=(20 if purpose=='screen' else 50): raise ValueError('Insufficient fresh validation questions')
    value=dict(protocol=PROTOCOL,purpose=purpose,dataset_revision=cfg['dataset_revision'],indices=indices,
        ids={k:[e['id'] for e in rows] for k,rows in subsets.items()},content_hashes={k:stable_hash(rows) for k,rows in subsets.items()},
        excluded_historical_ids=sorted(excluded),selection_seed=20261020,source_training_manifest_sha256=DEVELOPMENT_SOURCE_SHA256)
    write_json(path,value); return str(path)


def local_pilot_gate(run_dirs):
    if len(run_dirs)!=3: return dict(promising=False,status='incomplete',reason='Need all three matched screening arms')
    summaries=[json.loads((Path(p)/'summary.json').read_text()) for p in run_dirs]
    if any(not s['completed'] for s in summaries): return dict(promising=False,status='incomplete',reason='Partial comparisons cannot pass')
    runs=compare_runs(run_dirs); by={r['summary']['method']:r for r in runs}
    if set(by)!={'adaptive','static_k4','static_k5'}: raise ValueError('Need adaptive/static4/static5')
    a,b,c=[by[m]['summary'] for m in ('adaptive','static_k4','static_k5')]
    initialization=by['adaptive']['validation'][0]['numeric_exact_match_accuracy']
    saving=1-a['gross_training_attempts']/c['gross_training_attempts']
    conditions=dict(compute=saving>=.20-1e-12,accuracy=a['best']['score']>=c['best']['score']-.05-1e-12,
        improves_initial=a['best']['score']>initialization+1e-12,beats_static4=a['best']['score']>b['best']['score']+1e-12,
        finite_updates=all(math.isfinite(r['gradient_norm']) for run in runs for r in run['train']),
        nonzero_gradient=a['informative_gradient_episodes']>0,varied_k=len(set(a['actual_k']))>1)
    return dict(promising=all(conditions.values()),status='screening_only',conditions=conditions,pipeline_saving=saving,
        accuracy={m:r['summary']['best']['score'] for m,r in by.items()},initial_accuracy=initialization,
        no_final_test_access=True,caveat='One fresh seed and twenty validation questions; not formal acceptance')


def suite_ledger_path(results_root='results/adagrpo_v2'): return Path(results_root)/'local_suite.json'


def run_improvement_suite(results_root='results/adagrpo_v2',resume=False):
    root=Path(results_root); ledger_path=suite_ledger_path(root)
    notebook=Path('AdaGRPO_GSM8K_v2.ipynb'); source_hash=notebook_code_digest(notebook)
    revision=DEVELOPMENT_REVISION
    import io
    from contextlib import redirect_stdout
    fixture_output=io.StringIO()
    try:
        with redirect_stdout(fixture_output): checks=adagrpo_self_checks()
    finally:
        output=Path('docs/audit_artifacts/adagrpo_v2_fixture_output.log');output.parent.mkdir(parents=True,exist_ok=True);output.write_text(fixture_output.getvalue(),encoding='utf-8')
    print(f"ENGINEERING CHECKS PASSED: {checks['passed']} (synthetic; no external model calls)",flush=True)
    checks.update(notebook_code_sha256=source_hash,notebook_sha256=file_digest(notebook))
    write_json(Path('docs/audit_artifacts/adagrpo_v2_self_checks.json'),checks,overwrite=True)
    if ledger_path.exists():
        if not resume: raise RuntimeError('A suite already exists; explicit resume keeps the original deadline. No reruns/tuning.')
        ledger=json.loads(ledger_path.read_text())
        if ledger['notebook_code_sha256']!=source_hash: raise ValueError('Suite code changed; do not silently change a frozen candidate')
    else:
        if resume: raise ValueError('No suite to resume')
        # Dataset bookkeeping precedes all model work and is frozen once.
        manifest=freeze_membership(build_config('screen',results_root=str(root),dataset_revision=revision))
        now=datetime.now(timezone.utc).timestamp()
        ledger=dict(protocol=PROTOCOL,notebook_sha256=file_digest(notebook),notebook_code_sha256=source_hash,created_unix=now,deadline_unix=now+7200,
            membership_manifest=manifest,membership_sha256=file_digest(manifest),dataset_revision=revision,steps=[],status='running',
            budget_scope='All new local model work: setup, smoke, training, validation, failures and resume; pauses do not reset deadline')
        write_json(ledger_path,ledger)
    if ledger['status']=='completed': raise RuntimeError('Frozen suite already completed; inspect its report')
    if file_digest(ledger['membership_manifest'])!=ledger['membership_sha256']: raise ValueError('Frozen membership changed')
    stages=[('smoke','adaptive',5,42),('screen','static',5,43),('screen','adaptive',5,43),('screen','static',4,43)]
    try:
        for number,(preset,mode,k,seed) in enumerate(stages):
            if datetime.now(timezone.utc).timestamp()>=ledger['deadline_unix']: raise WallLimitReached('Suite deadline reached')
            cfg=build_config(preset,allocation_mode=mode,static_k=k,seed=seed,dataset_revision=revision,
                results_root=str(root),membership_manifest=ledger['membership_manifest'] if preset=='screen' else None,
                suite_ledger=str(ledger_path),wall_limit_seconds=None,checkpoint_interval=4 if preset=='screen' else 1)
            preflight(cfg)
            entry=next((e for e in ledger['steps'] if e['stage']==number),None)
            if entry and entry.get('completed'): continue
            if entry and entry.get('run_dir'):
                session=load_run(entry['run_dir'])
            else:
                session=initialize_experiment(cfg)
                entry=dict(stage=number,preset=preset,method=method_label(cfg),run_dir=str(session['run_dir']),completed=False)
                ledger['steps'].append(entry); write_json(ledger_path,ledger,overwrite=True)
            session['progress']=True
            print('RUN',number+1,'of',len(stages),entry['method'],entry['run_dir'],flush=True)
            try:
                if cfg['preset']=='screen':
                    previous=[e for e in ledger['steps'] if e['preset']=='screen' and e.get('completed')]
                    if previous:
                        ref=json.loads((Path(previous[0]['run_dir'])/'config.json').read_text())
                        if (session['data']['selection_id'],session['initial_policy_digest'],session['llm_metadata']['model']['digest'])!=(ref['selection_id'],ref['initial_policy_digest'],ref['execution_model']['model']['digest']): raise ValueError('Matched study identities differ')
                if number==0 and not entry.get('recovery_pause_done'):
                    original_attempt=globals()['recorded_attempt']; paused=[False]
                    def smoke_pause(*args,**kwargs):
                        item=original_attempt(*args,**kwargs)
                        if args[2]=='evaluation' and not paused[0]:
                            paused[0]=True
                            raise WallLimitReached('Intentional smoke interruption after durable evaluation finish')
                        return item
                    try:
                        globals()['recorded_attempt']=smoke_pause
                        train_grpo(session)
                    finally: globals()['recorded_attempt']=original_attempt
                    if not paused[0] or session['state']['status']!='wall_limit' or session['state']['updates']!=0:
                        raise RuntimeError('Smoke failed before its intended recovery pause')
                    entry['recovery_pause_done']=True;write_json(ledger_path,ledger,overwrite=True)
                    root_before=session['run_dir']; release_experiment(session)
                    session=load_run(root_before);session['progress']=True
                    print('SMOKE restored durable policy/optimizer; continue unfinished validation only',flush=True)
                train_grpo(session); summary=write_summary(session)
                if number==0:
                    if summary['gross_evaluation_attempts']!=4 or summary['repeated_evaluation_attempts']!=0 or summary['optimizer_updates']!=2:
                        raise RuntimeError('Live validation recovery repeated work or lost updates')
                    entry['live_validation_recovery']=dict(passed=True,gross_evaluation_attempts=4,repeated_evaluation_attempts=0,optimizer_updates=2)

                entry.update(completed=summary['completed'],status=summary['status'],gross_training_attempts=summary['gross_training_attempts'])
                if summary['completed']: verify_run_artifacts(session['run_dir'],summary)
                write_json(ledger_path,ledger,overwrite=True)
                if not summary['completed']:
                    ledger['status']='incomplete'; write_json(ledger_path,ledger,overwrite=True); return ledger
            finally: release_experiment(session)
        paths=[e['run_dir'] for e in ledger['steps'] if e['preset']=='screen']
        runs=compare_runs(paths); gate=local_pilot_gate(paths)
        fig=plot_comparison(runs,root/'comparison'); plt.close(fig)
        ledger.update(status='completed',gate=gate,finished_unix=datetime.now(timezone.utc).timestamp(),
            comparison=[dict(method=r['summary']['method'],best=r['summary']['best'],gross_training_attempts=r['summary']['gross_training_attempts'],actual_k=r['summary']['actual_k']) for r in runs])
        write_json(ledger_path,ledger,overwrite=True)
        write_json(Path('docs/audit_artifacts/adagrpo_v2_local_study.json'),ledger,overwrite=True)
        print('SCREENING RESULT',json.dumps(gate),flush=True); return ledger
    except BaseException as exc:
        ledger.update(status='incomplete',last_error=type(exc).__name__+': '+str(exc));write_json(ledger_path,ledger,overwrite=True)
        if isinstance(exc,WallLimitReached): print(str(exc),flush=True); return ledger
        raise


def run_local_suite(preset='smoke',results_root='results/adagrpo_v2',ollama_num_gpu=0):
    raise RuntimeError('Use run_improvement_suite for the approved shared deadline; individual work must be explicitly configured')


def formal_workload_estimate(seconds_per_pipeline=None):
    training=3*200*(2+4+5+4+4); validation=3*5*5*50; testing=3*5*1319; total=training+validation+testing
    return dict(training_max=training,validation=validation,full_test=testing,total_pipeline_executions_max=total,
        estimated_seconds=total*seconds_per_pipeline if seconds_per_pipeline is not None else None,
        note='Five arms incl shuffled K; excludes failures/setup. Not the old 27,828 estimate.')


def run_formal_study(results_root,allow_formal_training=False):
    if not allow_formal_training: raise RuntimeError('Formal cloud work requires explicit opt-in and a passing local gate')
    ledger=json.loads(suite_ledger_path().read_text())
    if ledger['notebook_code_sha256']!=notebook_code_digest('AdaGRPO_GSM8K_v2.ipynb'): raise RuntimeError('Frozen controller code changed since screening')
    if ledger.get('status')!='completed' or not ledger.get('gate',{}).get('promising'): raise RuntimeError('Local screening has not passed; no cloud spend')
    cfg,_=load_gsm8k(build_config('formal',results_root=results_root,ollama_num_gpu=None))
    manifest=freeze_membership(cfg,'formal'); paths=[]; identities=None
    for seed in (42,43,44):
        adaptive_schedule=None
        for mode,k in [('adaptive',5),('static',2),('static',4),('static',5),('scheduled',5)]:
            schedule=np.random.default_rng(seed+90000).permutation(adaptive_schedule).tolist() if mode=='scheduled' else None
            arm=build_config('formal',seed=seed,allocation_mode=mode,static_k=k,results_root=results_root,
                dataset_revision=cfg['dataset_revision'],ollama_num_gpu=None,membership_manifest=manifest,allocation_schedule=schedule)
            session=initialize_experiment(arm)
            try:
                ids=(session['data']['selection']['ids'],session['llm_metadata']['model']['digest'],session['initial_policy_digest'])
                if identities is not None and ids!=identities: raise ValueError('Formal membership/model changed')
                identities=ids; train_grpo(session); paths.append(str(session['run_dir']))
                if session['state']['status']!='completed': return dict(completed=False,run_dirs=paths)
                if mode=='adaptive': adaptive_schedule=list(session['state']['train'][i]['actual_k'] for i in range(200))
                if mode=='scheduled': write_json(session['run_dir']/'schedule_source.json',dict(adaptive_run=paths[-5],permutation_seed=seed+90000,source_k=adaptive_schedule,shuffled_k=schedule))
            finally: release_experiment(session)
    return dict(completed=True,run_dirs=paths,final_testing='Explicitly load best policies and call test_model; not automatic')


def run_cloud_smoke(results_root,allow_cloud_training=False):
    raise RuntimeError('No additional cloud smoke authorized; inspect local gate and plan cloud work explicitly')


## Formal/cloud study: opt in only after local screening passes
Mount durable storage and use a fresh cloud namespace. Preserve Llama 3.2 3B identity; capture dependency versions/hardware, and explicitly opt into any GPU offload. Do not resume a laptop checkpoint under a changed runtime. `formal_workload_estimate(measured_seconds_per_pipeline)` includes five arms: adaptive/static2/static4/static5/shuffled-K. Clean maximum: 34,935 executions. Full test loading remains an explicit operation after completed formal training. Freeze controller and membership first. A shuffled schedule preserves the adaptive K multiset but permutes its episode placement in a fresh independently initialized arm; it does not prove optional-stopping unbiasedness. No cloud runs are launched by definition cells.


## Notebook-local engineering checks
Tiny policies and deterministic mocks verify mechanics; their artifacts are temporary fixtures and never benchmark evidence. No dataset downloads or Llama calls occur. Run this before the bounded live smoke suite.


In [ ]:
def adagrpo_self_checks():
    from unittest.mock import patch
    from types import SimpleNamespace
    checks=[]
    def check(name,condition):
        if not condition: raise AssertionError(name)
        checks.append(name)
    def raises(name,fn,error=ValueError):
        try: fn()
        except error: checks.append(name)
        else: raise AssertionError(name+' did not raise')
    tokens={a:(0 if a=='CoT' else 1 if a=='Predict' else 5 if a=='stop' else 2+SIGNATURES.index(a)%3) for a in ACTIONS}
    def trajectory(module='CoT',signature=None):
        signature=signature or SIGNATURES[0]
        return dict(module=module,signature=signature,actions=[module,signature,'stop'])
    a,b=trajectory(),trajectory('Predict'); collision=trajectory(signature=SIGNATURES[3]); other=trajectory(signature=SIGNATURES[1])
    cfg=build_config('smoke',policy_device='cpu')
    for response,target,expected in [('[5]','5',True),('[50]','5',False),('[-1,250.50]','-1250.5',True),
        ('work 99 #### 4','4',True),('the result is 8','8',True),('[1 2]','2',False),('[nan]','0',False),('', '0',False),(None,'0',False),('[0.50]','.5',True)]:
        check('numeric '+str(response),compare_answers(response,target)==expected)
    judge=SimpleNamespace(__call__=lambda *a: (_ for _ in ()).throw(AssertionError('judge called')))
    check('numeric reward no judge',compute_reward('q','[0]','1',judge)['reward']==0.)
    check('bracket provenance',extraction_details('[5]')['extraction_source']=='brackets')
    check('hash provenance',extraction_details('#### 5')['extraction_source']=='hash_delimiter')
    check('fallback provenance',extraction_details('answer 5')['extraction_source']=='last_number')
    raises('nonfinite reward',lambda:group_advantages([0.,float('nan')]))
    raises('nonbinary controller',lambda:controller_decision(cfg,[a,b],[0.,.5],tokens,2.,8))
    d=lambda ts,rs,neff=8.,limit=15:controller_decision(cfg,ts,rs,tokens,neff,limit)
    check('mixed distinguishable refines',not d([a,b],[1.,0.])['stop'] and d([a,b],[1.,0.])['credit_contrast']>0)
    check('two successes stop',d([a,b],[1.,1.])['reason']=='easy_probe')
    check('four cancelled credits stop',d([a,b,a,b],[1.]*4)['reason']=='credit_cancelled')
    check('all failures expand',not d([a,b],[0.,0.])['stop'])
    check('bounded failure regardless concentration',d([a]*4,[0.]*4,1.1)['reason']=='bounded_failure')
    check('bounded failure with diverse policy',d([a]*4,[0.]*4,8.)['reason']=='bounded_failure')
    check('budget stop',d([a,b],[0.,0.],8.,2)['reason']=='allowance_reached')
    check('odd final allowance',d([a,b,a],[0.,0.,0.],8.,3)['reason']=='allowance_reached')
    check('collision contrast cancelled',not d([a,collision],[1.,0.])['stop'])
    check('class credit cancellation stops',d([a,a,other,other],[1.,0.,1.,0.])['reason']=='credit_cancelled')
    check('noncancelled contrast refines',not d([a,other],[1.,0.])['stop'])
    check('infrastructure classification',infrastructure_error(TimeoutError('timeout')) and not infrastructure_error(ValueError('parse')))
    for k in (2,4,5,8,10,15): check('static config '+str(k),build_config(allocation_mode='static',static_k=k)['training_budget']==2*k)
    raises('invalid bounds',lambda:build_config(k_min=1))
    raises('invalid budget',lambda:build_config(training_budget=1))
    raises('semantic protocol rejected',lambda:build_config(semantic_fallback=True))
    fake_data={'train':[dict(question=f'q{i}',answer='solution #### 1') for i in range(20)]}
    c1=build_config('smoke',train_pool_size=10,validation_size=2,dataset_revision='fixture')
    selection,subsets=selection_payload(c1,fake_data)
    s2,_=selection_payload(dict(c1,allocation_mode='static',static_k=5),fake_data)
    s3,_=selection_payload(dict(c1,seed=43),fake_data)
    check('same cross-arm selection/schedule',selection==s2)
    check('same cross-seed membership',selection['ids']==s3['ids'])
    check('independent training schedules',selection['episode_pool_offsets']!=s3['episode_pool_offsets'])
    check('no train validation leakage',not set(selection['ids']['train'])&set(selection['ids']['validation']))
    class TinyPolicy(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.logits=torch.nn.Parameter(torch.tensor([.1,-.1,.2,-.2,.05,0.])); self.config=SimpleNamespace(n_positions=1024)
        def forward(self,**kwargs): return SimpleNamespace(logits=self.logits.reshape(1,1,-1).expand(1,2,-1))
    class TinyTokenizer:
        def __call__(self,*args,**kwargs): return {'input_ids':torch.tensor([[0,1]])}
    def fake_execute(t,prompt,lm):
        lm.events.append(dict(seconds=.001,prompt_tokens=3,generated_tokens=2,error=None,completed=True))
        return dict(output='[1]' if t['module']=='CoT' else '[0]',success=True,error=None,seconds=.001)
    with tempfile.TemporaryDirectory(prefix='adagrpo-synthetic-') as tmp:
        def fixture(config=None,create_run=True):
            config=config or build_config('smoke',policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=2,checkpoint_interval=1,results_root=tmp)
            seed_everything(config['seed']); model=TinyPolicy().eval()
            data=prepare_selection(config,fake_data)
            root=Path(tmp)/uuid.uuid4().hex if create_run else None
            session=dict(config=config,policy=model,optimizer=torch.optim.Adam(model.parameters(),lr=config['learning_rate'],foreach=False),
                tokenizer=TinyTokenizer(),first_tokens=tokens,lm=SimpleNamespace(events=[]),
                llm_metadata=dict(version={'version':'fixture'},model={'digest':'fixture'}),initial_policy_digest=parameter_digest(model),
                data=data,run_dir=root,in_episode=False,evaluation_only=False,deadline=None,
                state=dict(episode=0,updates=0,train=[],validation=[],seen=[],best=None,status='initialized',gross_training_attempts=0))
            if create_run:
                root.mkdir(); (root/'checkpoints').mkdir()
                write_json(root/'config.json',dict(config=config,selection_id=data['selection_id'],selection=data['selection'],initial_policy_digest=session['initial_policy_digest'],first_tokens=tokens,execution_model=session['llm_metadata'],hardware=hardware_metadata()))
            return session
        session=fixture(); probe=probe_policy(session,'q'); distribution=effective_policy_distribution(probe,tokens)
        check('effective class probabilities normalized',math.isclose(sum(x['probability'] for x in distribution['classes']),1.))
        check('collision classes aggregated',len(distribution['classes'])==6)
        check('neff bounds',1<=distribution['neff']<=6)
        saved=rng_state(); probe_policy(session,'q'); check('probe consumes no torch RNG',torch.equal(saved['torch'],rng_state()['torch']))
        with isolated_rng(999): torch.rand(5); random.random(); np.random.random()
        check('evaluation RNG isolation',torch.equal(saved['torch'],rng_state()['torch']))
        # Exact sum-gradient and Adam parameter equivalence against a reference update.
        ts=[generate_pipeline(session['policy'],session['tokenizer'],tokens,'q',session['config']) for _ in range(4)]
        rewards=[1.,0.,1.,0.]; ref=TinyPolicy(); opt=torch.optim.Adam(ref.parameters(),lr=session['config']['learning_rate'],foreach=False)
        _,advantages=group_advantages(rewards)
        ref_loss=figure2_loss([replay_log_probs(ref,session['tokenizer'],tokens,'q',t,session['config']) for t in ts],advantages)
        ref_loss.backward(); ref_norm=math.sqrt(sum(float(x.grad.double().square().sum()) for x in ref.parameters())); opt.step()
        result=update_policy(session,'q',ts,rewards)
        check('sum loss equivalence',math.isclose(result['loss'],float(ref_loss.detach()),abs_tol=1e-7))
        check('gradient norm equivalence',math.isclose(result['gradient_norm'],ref_norm,rel_tol=1e-6,abs_tol=1e-8))
        check('Adam parameter equivalence',all(torch.allclose(x,y,atol=1e-8) for x,y in zip(session['policy'].parameters(),ref.parameters())))
        # Ensure nonzero momentum, independently of random sample collisions.
        ts=[]
        for t in [a,b]:
            t=copy.deepcopy(t); t['log_probs']=[float(x.detach()) for x in replay_log_probs(session['policy'],session['tokenizer'],tokens,'q',t,session['config'])]; ts.append(t)
        update_policy(session,'q',ts,[1.,0.]); previous=session['policy'].logits.detach().clone()
        for t in ts: t['log_probs']=[float(x.detach()) for x in replay_log_probs(session['policy'],session['tokenizer'],tokens,'q',t,session['config'])]
        equal=update_policy(session,'q',ts,[1.,1.])
        check('equal rewards zero current gradient',equal['gradient_norm']==0. and equal['optimizer_step'])
        check('Adam momentum preserved',not torch.equal(previous,session['policy'].logits.detach()))
        broken=copy.deepcopy(ts); broken[0]['log_probs'][0]+=1
        raises('rollout replay mismatch',lambda:update_policy(session,'q',broken,[1.,0.]),RuntimeError)
        # Gross budget reserves future minimum groups and charges unfinished work.
        reserve=fixture(); reserve['config']['training_budget']=4
        check('future minimum reservation',episode_allowance(reserve)==2)
        append_journal(reserve,dict(kind='training_attempt_started',attempt_id='lost',episode=1,question_id='q'))
        raises('lost attempts exhaust reserved budget',lambda:episode_allowance(reserve),BudgetExhausted)
        check('unfinished resource totals unavailable',journal_resource_totals(reserve['run_dir'])['total_llm_calls'] is None)
        journal=reserve['run_dir']/'attempts.jsonl'
        with journal.open('ab') as stream: stream.write(b'{"kind":"training_attempt_fini')
        repair=repair_journal_tail(reserve['run_dir'])
        check('truncated journal tail preserved', (reserve['run_dir']/repair['backup']).read_bytes()==b'{"kind":"training_attempt_fini')
        check('tail repair retains gross cost',journal_counts(reserve)==1 and journal_resource_totals(reserve['run_dir'])['total_llm_calls'] is None)
        append_journal(reserve,dict(kind='unicode_fixture',message='hello'+chr(0x2028)+'world'))
        check('JSONL Unicode separator preserved',read_jsonl(journal)[-1]['message']=='hello'+chr(0x2028)+'world')
        complete=journal.read_bytes().rstrip(b'\n'); journal.write_bytes(complete)
        repair_journal_tail(reserve['run_dir']); check('valid last record newline restored',journal.read_bytes().endswith(b'\n'))
        damaged=fixture(); (damaged['run_dir']/'attempts.jsonl').write_bytes(b'bad committed record\n')
        raises('committed journal corruption rejected',lambda:repair_journal_tail(damaged['run_dir']))

        # End-to-end mocked runs exercise the real collector, trainer, checkpoints, and audits.
        paths=[]
        with patch.dict(globals(),execute_pipeline=fake_execute,initialize_experiment=fixture,ollama_snapshot=lambda cfg:dict(scope='synthetic fixture')):
            for mode,k in [('static',5),('adaptive',5)]:
                ss=fixture(build_config('smoke',allocation_mode=mode,static_k=k,policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=2,checkpoint_interval=1,results_root=tmp))
                train_grpo(ss); paths.append(str(ss['run_dir']))
                audit=verify_run_artifacts(ss['run_dir']); check('completed audit '+mode,audit['summary']['completed'])
                check('gross bound '+mode,journal_counts(ss)<=ss['config']['training_budget'])
            check('cross-method comparison',len(compare_runs(paths))==2)
            fig=plot_comparison(compare_runs(paths)); plt.close(fig); checks.append('variable-K diagnostic plots')
            root=Path(paths[-1]); original=(root/'training.jsonl').read_bytes(); rows=read_jsonl(root/'training.jsonl'); rows[0]['actual_k']+=1; write_jsonl(root/'training.jsonl',rows)
            raises('tampered group rejected',lambda:verify_run_artifacts(root))
            (root/'training.jsonl').write_bytes(original)
            # Pause mid-group, restore RNG/optimizer, and charge the discarded attempt.
            config=build_config('smoke',allocation_mode='static',static_k=2,num_episodes=3,training_budget=7,validation_interval=1,checkpoint_interval=1,policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=2,results_root=tmp)
            interrupted=fixture(config); original_attempt=recorded_attempt
            def interrupt_attempt(session,example,kind,candidate=None,split=None):
                item=original_attempt(session,example,kind,candidate,split)
                if kind=='training' and session['state']['episode']==1 and candidate==0: raise WallLimitReached('synthetic pause')
                return item
            with patch.dict(globals(),recorded_attempt=interrupt_attempt): train_grpo(interrupted)
            check('partial group not committed',interrupted['state']['episode']==1 and interrupted['in_episode'])
            durable_pointer=json.loads((interrupted['run_dir']/'best.json').read_text())
            write_json(interrupted['run_dir']/'best.json',dict(durable_pointer,episode=999),overwrite=True)
            resumed=load_run(interrupted['run_dir'])
            check('recover staged best/latest transaction',json.loads((resumed['run_dir']/'best.json').read_text())==durable_pointer)
            train_grpo(resumed)
            check('resume completed',resumed['state']['status']=='completed')
            check('discarded attempt charged',write_summary(resumed)['discarded_training_attempts']==1)
            verify_run_artifacts(resumed['run_dir']); checks.append('resumed artifacts valid')
            uninterrupted=fixture(config); train_grpo(uninterrupted)
            check('resume policy matches uninterrupted',torch.equal(resumed['policy'].logits,uninterrupted['policy'].logits))
            check('only latest and best retained',len(list((resumed['run_dir']/'checkpoints').glob('*.pt')))==2)
            cp=json.loads((resumed['run_dir']/'latest.json').read_text()); checkpoint=resumed['run_dir']/cp['checkpoint']; raw=checkpoint.read_bytes(); checkpoint.write_bytes(raw+b'corruption')
            raises('checkpoint checksum rejection',lambda:load_run(resumed['run_dir']))
            checkpoint.write_bytes(raw)
        with patch.dict(globals(),execute_pipeline=fake_execute,initialize_experiment=fixture,ollama_snapshot=lambda cfg:dict(scope='synthetic fixture')):
            failed=fixture()
            original_attempt=recorded_attempt
            def unavailable(t,p,lm): raise InfrastructureFailure('synthetic offline service')
            def fail_training(session,example,kind,candidate=None,split=None):
                if kind=='training':
                    with patch.dict(globals(),execute_pipeline=unavailable): return original_attempt(session,example,kind,candidate,split)
                return original_attempt(session,example,kind,candidate,split)
            with patch.dict(globals(),recorded_attempt=fail_training): train_grpo(failed)
            check('infrastructure failure no policy update',failed['state']['updates']==0 and not failed['state']['train'])
            check('infrastructure failure still charged',journal_counts(failed)==1 and write_summary(failed)['discarded_training_attempts']==1)
            check('failure memory recorded',all('failure_memory' in e for e in read_jsonl(failed['run_dir']/'attempts.jsonl') if e['kind']=='interruption'))
        check('bootstrap identical predictions',paired_hierarchical_bootstrap(np.zeros((3,12)),100)==[0.,0.])
        check('bootstrap paired constant difference',np.allclose(paired_hierarchical_bootstrap(np.full((3,12),-.1),100),[-.1,-.1]))
        check('bootstrap repeatability',paired_hierarchical_bootstrap(np.arange(30).reshape(3,10),100)==paired_hierarchical_bootstrap(np.arange(30).reshape(3,10),100))
        # Exercise the collector itself through repeated expansion and an odd last round.
        def zero_execute(t,p,lm): return dict(output='[0]',success=True,error=None,seconds=0.)
        def one_execute(t,p,lm): return dict(output='[1]',success=True,error=None,seconds=0.)
        cap_cfg=build_config('smoke',num_episodes=1,training_budget=15,policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=2,results_root=tmp)
        capped=fixture(cap_cfg)
        with patch.dict(globals(),execute_pipeline=zero_execute):
            group=collect_group(capped,capped['data']['subsets']['train'][0])
        check('collector no K15 drain',group['actual_k']==4 and [x['actual_k'] for x in group['decision_trace']]==[2,4])
        check('collector cap charges every attempt',journal_counts(capped)==4)
        successful=fixture(cap_cfg)
        with patch.dict(globals(),execute_pipeline=one_execute):
            group=collect_group(successful,successful['data']['subsets']['train'][0])
        check('collector all-correct exit',group['actual_k']==2 and group['stop_reason']=='easy_probe')
        concentrated=fixture(cap_cfg)
        with torch.no_grad(): concentrated['policy'].logits.copy_(torch.tensor([100.,-100.,100.,-100.,-100.,0.]))
        with patch.dict(globals(),execute_pipeline=zero_execute):
            group=collect_group(concentrated,concentrated['data']['subsets']['train'][0])
        check('collector bounded-failure exit',group['actual_k']==4 and group['stop_reason']=='bounded_failure')
        # Formal inference cannot be launched from a local pilot or without opt-in.
        raises('formal training explicit opt-in',lambda:run_formal_study(tmp),RuntimeError)
        raises('cloud smoke explicit opt-in',lambda:run_cloud_smoke(tmp),RuntimeError)
        raises('operation guards full training',lambda:run_operation(build_config('formal'),operation='train'),RuntimeError)
        check('formal workload exact',formal_workload_estimate()['total_pipeline_executions_max']==34935)
        raises('formal report requires all seeds',lambda:acceptance_report([]))
        # Synthetic paired formal arrays exercise reporting, never research artifacts.
        formal=[]
        for seed in (42,43,44):
            for method,cost in [('adaptive',800),('static_k2',400),('static_k4',800),('static_k5',1000),('shuffled_k',800)]:
                predictions=[dict(reward=dict(numeric_exact_match=(i%5!=0))) for i in range(1319)]
                formal.append(dict(root='synthetic-'+str(seed)+'-'+method,
                    summary=dict(config=dict(preset='formal',seed=seed,num_episodes=200,train_pool_size=200,validation_size=50),
                        completed=True,method=method,gross_training_attempts=cost),
                    tests=[dict(test_content_hash='synthetic-only',records=predictions)]))
        with patch.dict(globals(),compare_runs=lambda roots:formal):
            report=acceptance_report(formal)
            check('paired formal acceptance numerical target',report['target_demonstrated'])
            check('cheaper fixed control limitation reported',len(report['limitations'])==3)
            for run in formal:
                if run['summary']['method']=='adaptive':
                    for record in run['tests'][0]['records']: record['reward']['numeric_exact_match']=False
            check('budget cap alone insufficient',not acceptance_report(formal)['target_demonstrated'])

    return dict(passed=len(checks),checks=checks,scope='synthetic engineering only; no Llama, downloads, or benchmark evidence')


def exact_allocation_diagnostic(kind,allowance,probabilities=(.5,.3,.2),reward_probabilities=(.8,.5,.2)):
    """Exact categorical toy expectation. Fixtures only, never empirical benchmark evidence."""
    p=np.asarray(probabilities,dtype=float); mu=np.asarray(reward_probabilities,dtype=float)
    if kind not in ('fixed','v1','v2') or not np.isclose(p.sum(),1) or np.any(p<=0): raise ValueError('Invalid toy')
    outcome=[float(p[c]*(mu[c] if r else 1-mu[c])) for c in range(len(p)) for r in (0,1)]
    active={(0,)*(2*len(p)):1.}; mean=np.zeros(len(p)); second=0.; cost=0.; mass=0.
    for n in range(1,allowance+1):
        nxt=defaultdict(float)
        for state,weight in active.items():
            for j,pr in enumerate(outcome):
                if not pr: continue
                counts=list(state); counts[j]+=1; nxt[tuple(counts)]+=weight*pr
        active={}
        for counts,weight in nxt.items():
            successes=sum(counts[1::2]); credit=np.array([counts[2*c+1]-(counts[2*c]+counts[2*c+1])*successes/n for c in range(len(p))])
            contrast=np.abs(credit).sum()>1e-12; at_round=n%2==0 or n==allowance; stop=n==allowance
            if kind=='v1' and at_round and n>=2: stop=stop or (0<successes<n and contrast) or (n>=4 and successes==n)
            if kind=='v2' and at_round and n>=2: stop=stop or (n==2 and successes==2) or (n>=4 and not contrast)
            if stop:
                mass+=weight; cost+=weight*n; mean+=weight*credit; second+=weight*float(credit@credit)
            else: active[counts]=weight
    true=p*(mu-float(p@mu)); denominator=np.linalg.norm(mean)*np.linalg.norm(true)
    return dict(kind=kind,allowance=allowance,probabilities=p.tolist(),reward_probabilities=mu.tolist(),probability_mass=mass,
        expected_attempts=cost,expected_gradient=mean.tolist(),true_reward_gradient=true.tolist(),gradient_variance=max(0.,second-float(mean@mean)),
        cosine=float(mean@true/denominator) if denominator>1e-15 else None,scope='exact fixed-policy categorical fixture, not GPT-2 accuracy')


_base_v2_checks=adagrpo_self_checks

def adagrpo_self_checks():
    from unittest.mock import patch
    from types import SimpleNamespace
    base=_base_v2_checks(); checks=list(base['checks']); theory=[]
    def check(name,value):
        if not value: raise AssertionError(name)
        checks.append(name)
    def raises(name,fn,error=ValueError):
        try: fn()
        except error: checks.append(name)
        else: raise AssertionError(name+' did not raise')
    class TinyPolicy(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.logits=torch.nn.Parameter(torch.tensor([.1,-.1,.2,-.2,.05,0.])); self.config=SimpleNamespace(n_positions=1024)
        def forward(self,**kw): return SimpleNamespace(logits=self.logits.reshape(1,1,-1).expand(1,2,-1))
    class TinyTokenizer:
        def __call__(self,*args,**kw): return {'input_ids':torch.tensor([[0,1]])}
    tokens={a:(0 if a=='CoT' else 1 if a=='Predict' else 5 if a=='stop' else 2+SIGNATURES.index(a)%3) for a in ACTIONS}
    fake_data={'train':[dict(question='fixture q'+str(i),answer='reason #### 1') for i in range(25)]}
    with tempfile.TemporaryDirectory(prefix='adagrpo-v2-checks-') as tmp:
        def fixture(**overrides):
            cfg=build_config('smoke',policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=3,results_root=tmp,**overrides)
            root=Path(tmp)/uuid.uuid4().hex;root.mkdir();(root/'checkpoints').mkdir()
            model=TinyPolicy().eval()
            return dict(config=cfg,policy=model,optimizer=torch.optim.Adam(model.parameters(),lr=1e-5,foreach=False),tokenizer=TinyTokenizer(),
                first_tokens=tokens,lm=SimpleNamespace(events=[]),llm_metadata=dict(model=dict(digest='fixture'),version=dict(version='fixture')),
                run_dir=root,data=prepare_selection(cfg,fake_data),initial_policy_digest=parameter_digest(model),deadline=None,in_episode=False,evaluation_only=False,
                state=dict(episode=0,updates=0,train=[],validation=[],seen=[],best=None,status='initialized',gross_training_attempts=0))
        def execute(t,p,lm):
            lm.events.append(dict(seconds=.001,prompt_tokens=3,generated_tokens=2,error=None,completed=True))
            return dict(output='[1]' if t['module']=='Predict' else '[0]',success=True,error=None,seconds=.001)
        ss=fixture(num_episodes=3,training_budget=12)
        check('v2 initial no borrowed six',budget_snapshot(ss)['allowance']==4 and budget_snapshot(ss)['bank']==0)
        for i in range(2):append_journal(ss,dict(kind='training_attempt_started',attempt_id=str(i),episode=1,question_id='fixture'))
        ss['state']['episode']=1;check('v2 savings permit six',budget_snapshot(ss)['allowance']==6 and budget_snapshot(ss)['bank']==2)
        append_journal(ss,dict(kind='training_attempt_started',attempt_id='lost',episode=2,question_id='fixture'))
        check('v2 lost attempt reduces bank',budget_snapshot(ss)['allowance']==5 and budget_snapshot(ss)['bank']==1)
        for i in range(3,8):append_journal(ss,dict(kind='training_attempt_started',attempt_id=str(i),episode=2,question_id='fixture'))
        ss['state']['episode']=2;check('v2 spending savings restores four',budget_snapshot(ss)['allowance']==4)
        cfg=build_config(policy_device='cpu')
        def trajectory(module):return dict(module=module,signature=SIGNATURES[0],actions=[module,SIGNATURES[0],'stop'])
        a,b=trajectory('CoT'),trajectory('Predict')
        decide=lambda ts,rs,allow=6:controller_decision(cfg,ts,rs,tokens,2.,allow)
        check('v2 contrast at four expands with bank',not decide([a,b,a,b],[0.,1.,0.,1.])['stop'])
        check('v2 six informative terminal',decide([a,b]*3,[0.,1.]*3)['reason']=='informative_refinement')
        check('v2 four failures terminal',decide([a,b]*2,[0.]*4)['reason']=='bounded_failure')
        check('v2 cancellation at four terminal',decide([a,a,b,b],[0.,1.,0.,1.])['reason']=='credit_cancelled')
        check('v2 odd allowance terminal',decide([a,b,a,b,a],[0.,1.,0.,1.,0.],5)['reason']=='allowance_reached')
        check('v2 allowance two terminal',decide([a,b],[0.,1.],2)['reason']=='allowance_reached')
        # Collector retains every refinement candidate under real journal accounting.
        refined=fixture(num_episodes=3,training_budget=12)
        refined['state']['episode']=1
        for i in range(2):append_journal(refined,dict(kind='training_attempt_started',attempt_id='prior'+str(i),episode=1,question_id='fixture'))
        stream=iter([copy.deepcopy(x) for x in [a,b]*3])
        with patch.dict(globals(),execute_pipeline=execute,generate_pipeline=lambda *args,**kw:next(stream)):
            group=collect_group(refined,refined['data']['subsets']['train'][0])
        check('v2 collector refinements 2 4 6',[d['actual_k'] for d in group['decision_trace']]==[2,4,6])
        check('v2 collector retains all six',len(group['trajectories'])==group['actual_k']==6 and journal_counts(refined)==8)
        # Per-candidate streams are independent of mode, skipped previous candidates and global RNG.
        first=fixture(); second=fixture(allocation_mode='static',static_k=4)
        ex=first['data']['subsets']['train'][0];saved=torch.get_rng_state().clone()
        with patch.dict(globals(),execute_pipeline=execute):
            t1=recorded_attempt(first,ex,'training',candidate=0)['trajectory']
            for i in range(7):recorded_attempt(first,ex,'training',candidate=i+1)
            t2=recorded_attempt(second,ex,'training',candidate=0)['trajectory']
        check('v2 matched candidate streams',t1==t2)
        check('v2 streams leave global RNG untouched',torch.equal(saved,torch.get_rng_state()))
        check('v2 next episode not shifted',candidate_seed(first['config'],'training',2,0)==candidate_seed(second['config'],'training',2,0))
        check('v2 candidates distinct stream IDs',candidate_seed(cfg,'training',1,0)!=candidate_seed(cfg,'training',1,1))
        check('v2 evaluation does not use training count',candidate_seed(cfg,'evaluation',1,3,'q','validation')==candidate_seed(cfg,'evaluation',9,99,'q','validation'))
        check('v2 training/evaluation separated',candidate_seed(cfg,'training',1,0)!=candidate_seed(cfg,'evaluation',question_id='q',split='validation'))
        # Simulate a hard interruption after a finished journal row but before progress publication.
        val=fixture(); original=recorded_attempt; visits=[0]
        def interrupted(*args,**kw):
            item=original(*args,**kw);visits[0]+=1
            if visits[0]==1: raise WallLimitReached('fixture interrupted before progress write')
            return item
        with patch.dict(globals(),execute_pipeline=execute,recorded_attempt=interrupted):
            raises('v2 partial validation interruption',lambda:evaluate_module(val),WallLimitReached)
        with patch.dict(globals(),execute_pipeline=execute): result=evaluate_module(val)
        starts=[e for e in read_jsonl(val['run_dir']/'attempts.jsonl') if e['kind']=='evaluation_attempt_started']
        check('v2 validation resumes only unfinished',len(starts)==3 and len(result['records'])==3)
        uninterrupted=fixture()
        with patch.dict(globals(),execute_pipeline=execute): normal=evaluate_module(uninterrupted)
        check('v2 validation recovery exactly matches',[(r['trajectory'],r['reward']) for r in result['records']]==[(r['trajectory'],r['reward']) for r in normal['records']])
        with patch.dict(globals(),execute_pipeline=execute): again=evaluate_module(val)
        check('v2 complete validation no repeated calls',len(val['lm'].events)==3 and again['identity_id']==result['identity_id'])
        with patch.dict(globals(),execute_pipeline=execute): reverse=evaluate_module(val,examples=list(reversed(val['data']['subsets']['validation'])))
        check('v2 evaluation order independent', {r['question_id']:r['trajectory'] for r in reverse['records']}=={r['question_id']:r['trajectory'] for r in result['records']})
        check('v2 different evaluation identity archived',bool(list(val['run_dir'].glob('*_discarded_*.json'))))
        with torch.no_grad():val['policy'].logits[0]+=.1
        with patch.dict(globals(),execute_pipeline=execute): changed=evaluate_module(val)
        check('v2 changed policy rejects reuse',changed['identity_id']!=result['identity_id'] and len(val['lm'].events)==9)
        check('v2 rejected work stays journaled',len([e for e in read_jsonl(val['run_dir']/'attempts.jsonl') if e['kind']=='evaluation_attempt_started'])==9)
        check('v2 model identity in validation',changed['identity']['execution_digest']=='fixture')
        expired=fixture(); path=Path(tmp)/'deadline.json';write_json(path,dict(deadline_unix=0))
        expired['config']['suite_ledger']=str(path)
        raises('v2 persistent deadline enforced',lambda:check_deadline(expired),WallLimitReached)
        check('v2 deadline unchanged',json.loads(path.read_text())['deadline_unix']==0)
        # Frozen membership must be disjoint, versioned, and content-verified.
        manifest=Path(tmp)/'members.json'; indices=dict(train=list(range(10)),validation=[20,21,22])
        subsets={k:[dataset_example(fake_data,'train',i) for i in ids] for k,ids in indices.items()}
        frozen=dict(dataset_revision='fixture',indices=indices,content_hashes={k:stable_hash(rows) for k,rows in subsets.items()})
        write_json(manifest,frozen);fc=build_config('smoke',dataset_revision='fixture',validation_size=3,membership_manifest=str(manifest))
        payload,_=selection_payload(fc,fake_data)
        check('v2 frozen split membership',payload['indices']==indices and not set(payload['ids']['train'])&set(payload['ids']['validation']))
        frozen['content_hashes']['validation']='corrupt';write_json(manifest,frozen,overwrite=True)
        raises('v2 frozen content tamper rejected',lambda:selection_payload(fc,fake_data))
        # Reject a v1 checkpoint before any model initialization.
        old=fixture();write_json(old['run_dir']/'config.json',dict(config=old['config']))
        cp=old['run_dir']/'checkpoints/old.pt';torch.save(dict(format_version=2,protocol='adagrpo-gsm8k-numeric-figure2-v1',config=old['config']),cp)
        write_json(old['run_dir']/'latest.json',dict(checkpoint='checkpoints/old.pt',sha256=file_digest(cp),episode=0))
        raises('v2 v1 checkpoint forbidden',lambda:load_run(old['run_dir']))
        before=probe_policy(first,'q');after=probe_policy(first,'q')
        check('v2 unchanged policy diagnostic',policy_change(before,after,tokens)['total_variation']==0.)
        with torch.no_grad():first['policy'].logits[0]+=.3
        check('v2 changed policy diagnostic',policy_change(before,probe_policy(first,'q'),tokens)['total_variation']>0.)
        schedule=[2,6,4]; sc=build_config('smoke',allocation_mode='scheduled',num_episodes=3,allocation_schedule=schedule)
        check('v2 scheduled arm budget',sc['training_budget']==12)
        raises('v2 invalid shuffled schedule',lambda:build_config(allocation_mode='scheduled',allocation_schedule=[1,7]))
        raises('v2 frozen controller thresholds',lambda:build_config(k_max=15))
        # Exhaustive budget arithmetic over completed episodes and gross costs.
        paced=fixture(num_episodes=12,training_budget=48)
        valid=0; rejected=0
        for ep in range(12):
            paced['state']['episode']=ep
            for gross in range(49):
                with patch.dict(globals(),journal_counts=lambda session,g=gross:g):
                    try: snapshot=budget_snapshot(paced)
                    except BudgetExhausted:
                        if 48-gross-2*(11-ep)>=2: raise AssertionError('Unexpected budget rejection')
                        rejected+=1;continue
                if not (2<=snapshot['allowance']<=6 and gross+snapshot['allowance']+snapshot['future_reservation']<=48): raise AssertionError('Budget arithmetic')
                if gross<=4*ep and gross+snapshot['allowance']>4*(ep+1): raise AssertionError('Clean run borrowed future refinement')
                valid+=1
        check('v2 exhaustive budget reservation and pacing',valid>0 and rejected>0)
        digest_file=Path(tmp)/'notebook.ipynb'
        minimal=dict(cells=[dict(cell_type='code',source='x=1',outputs=[]),dict(cell_type='markdown',source='text')])
        digest_file.write_text(json.dumps(minimal));code=notebook_code_digest(digest_file)
        minimal['cells'][0]['outputs']=[dict(text='new execution output')];digest_file.write_text(json.dumps(minimal))
        check('v2 code freeze ignores outputs',notebook_code_digest(digest_file)==code)
        minimal['cells'][0]['source']='x=2';digest_file.write_text(json.dumps(minimal))
        check('v2 code freeze detects source changes',notebook_code_digest(digest_file)!=code)
        same=fixture()
        with patch.dict(globals(),execute_pipeline=execute): start=evaluate_module(same)
        with torch.no_grad():same['policy'].logits[1]-=.4
        with patch.dict(globals(),execute_pipeline=execute): finish=evaluate_module(same)
        check('v2 policy-only change invalidates evaluation',start['identity']['questions']==finish['identity']['questions'] and start['identity']['policy_sha256']!=finish['identity']['policy_sha256'] and len(same['lm'].events)==6)
        same['config']['execution_max_tokens']=511
        with patch.dict(globals(),execute_pipeline=execute): changed_runtime=evaluate_module(same)
        check('v2 execution options invalidate evaluation',changed_runtime['identity_id']!=finish['identity_id'] and len(same['lm'].events)==9)
        for mu in [(1.,0.,1.),(.8,.5,.2),(.2,.5,.8)]:
            for kind,limit in [('fixed',2),('fixed',4),('fixed',5),('v1',15),('v2',4),('v2',6)]:
                d=exact_allocation_diagnostic(kind,limit,reward_probabilities=mu);theory.append(d)
                check('toy finite probability '+str(mu)+kind+str(limit),abs(d['probability_mass']-1)<1e-10 and np.isfinite(d['gradient_variance']) and 2<=d['expected_attempts']<=limit+1e-9)
                if kind=='fixed': check('toy original sum K-minus-one '+str(mu)+str(limit),np.allclose(d['expected_gradient'],np.array(d['true_reward_gradient'])*(limit-1),atol=1e-10))
        zero=exact_allocation_diagnostic('v2',6,reward_probabilities=(0.,0.,0.))
        check('v2 all-fail toy bounded at four',zero['expected_attempts']==4 and zero['expected_gradient']==[0.,0.,0.])
    return dict(passed=len(checks),base_checks=base['passed'],new_checks=len(checks)-base['passed'],checks=checks,theory=theory,
        scope='synthetic engineering only; no external model work, downloads, or accuracy claims')


## Operation switch
Definition cells are inert. The authorized local entry point is `run_improvement_suite()`, which creates one persisted two-hour ledger. `resume=True` retains its deadline and source hash. Do not launch separate ad hoc model runs outside that budget. Cloud work and official final tests remain explicit and gated.


In [ ]:
def run_operation(cfg=None,operation='initialize',selected_run=None,run_dirs=None):
    cfg=cfg or CONFIG
    if operation in ('initialize','train'):
        if operation=='train' and cfg['preset']=='formal':
            raise RuntimeError('Use the formal-study opt-in for full training')
        preflight(cfg); session=initialize_experiment(cfg)
        if operation=='train': train_grpo(session)
        return session
    if operation=='study': return run_improvement_suite(results_root=cfg['results_root'])
    if operation=='resume_study': return run_improvement_suite(results_root=cfg['results_root'],resume=True)
    if operation in ('smoke','pilot'): raise RuntimeError('Use the bounded study operation')
    if operation in ('resume','test'):
        if selected_run is None: raise ValueError('An explicit run directory is required')
        session=load_run(selected_run,for_resume=operation=='resume')
        if operation=='resume': train_grpo(session); return session
        try: return test_model(session)
        finally: release_experiment(session)
    if operation=='compare':
        runs=compare_runs(run_dirs or []); plot_comparison(runs); return runs
    raise ValueError('Unknown operation')

OPERATION='initialize'  # initialize / train / smoke / pilot / resume / test / compare
SELECTED_RUN=None
RUN_DIRS=[]
if RUN_OPERATION:
    OPERATION_RESULT=run_operation(CONFIG,OPERATION,SELECTED_RUN,RUN_DIRS)
else:
    print('AdaGRPO definitions ready. No model initialization, training, or testing launched.')


In [ ]:
# Study orchestration changes artifact ownership, never the V2 estimator.
VERIFIED_NOTEBOOK='AdaGRPO_GSM8K_v2_verified.ipynb'
FROZEN_V2='AdaGRPO_GSM8K_v2.ipynb'
FROZEN_V2_SHA256='5fd062295bdb8007f24681088311b47670a707910aabc9fa889b16d304fad10a'
EXPERIMENTAL_FUNCTIONS=('acceptance_report','paired_hierarchical_bootstrap','test_model','load_gsm8k','dataset_example','initialize_policy','initialize_ollama','canonical_number','canonical_target','extraction_details','compare_answers','execute_pipeline','recorded_attempt','evaluate_module','train_grpo','action_log_probs','sample_actions','replay_log_probs','generate_pipeline','probe_policy',
    'compute_reward','controller_decision','budget_snapshot','candidate_seed','candidate_generator','collect_group',
    'group_advantages','figure2_loss','update_policy','policy_class','effective_policy_distribution','contrast_statistics')
STUDY_METHODS=('adaptive','static_k2','static_k4','static_k5','shuffled_k')


def experimental_fingerprint(path):
    import ast
    notebook=json.loads(Path(path).read_text(encoding='utf-8'))
    text='\n'.join(''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code')
    tree=ast.parse(text);result={}
    for name in EXPERIMENTAL_FUNCTIONS:
        nodes=[node for node in tree.body if isinstance(node,ast.FunctionDef) and node.name==name]
        if len(nodes)!=(2 if name=='train_grpo' else 1):raise ValueError('Ambiguous/missing frozen definition '+name)
        result[name]=stable_hash([ast.dump(node,include_attributes=False) for node in nodes])
    return result


def expected_reference():
    root=Path('results/adagrpo_v2/gsm8k/adaptive/seed43/screen/20261020T154041-268c0802')
    if file_digest(FROZEN_V2)!=FROZEN_V2_SHA256:raise ValueError('Frozen V2 notebook changed')
    meta=json.loads((root/'config.json').read_text(encoding='utf-8'))
    summary=json.loads((root/'summary.json').read_text(encoding='utf-8'))
    return dict(source_run=str(root),config_sha256=file_digest(root/'config.json'),
        policy_initial_sha256=meta['initial_policy_digest'],model_digest=meta['execution_model']['model']['digest'],
        first_tokens=meta['first_tokens'],runtime=summary['runtime_identity'])


def historical_evaluation_ids(root='results'):
    # Read actual/declared evaluation membership, never a manifest's exclusion lists.
    used=set();base=Path(root)
    def collect(value):
        if isinstance(value,str) and re.fullmatch(r'gsm8k:train:\d+',value):used.add(value)
        elif isinstance(value,dict):
            for item in value.values():collect(item)
        elif isinstance(value,list):
            for item in value:collect(item)
    paths=set(base.rglob('config.json'))|set(base.glob('**/selections/*.json'))|set(base.glob('**/membership/*.json'))
    for p in sorted(paths):
        value=json.loads(p.read_text(encoding='utf-8'));selection=value.get('selection',value)
        collect(selection.get('ids',{}).get('validation',[]))
        if p.name=='confirmation.json':collect(value.get('examples',[]))
    for p in base.rglob('validation.jsonl'):
        for row in read_jsonl(p):
            collect(row.get('question_ids',[]));collect(row.get('records',[]))
    for p in base.rglob('result.json'):
        value=json.loads(p.read_text(encoding='utf-8'))
        if value.get('split') in ('validation','confirmation'):
            collect(value.get('question_ids',[]));collect(value.get('records',[]))
    return used


def latency_estimate(reference_roots):
    seconds=[];setup=None
    for root in sorted({str(Path(p).resolve()) for p in reference_roots}):
        for event in read_jsonl(Path(root)/'attempts.jsonl'):
            if event['kind'].endswith('_attempt_finished') and event.get('seconds') is not None:seconds.append(event['seconds'])
    workload=formal_workload_estimate()
    return dict(**workload,observations=len(seconds),median_seconds=float(np.median(seconds)) if seconds else None,
        p90_seconds=float(np.quantile(seconds,.9)) if seconds else None,
        estimated_pipeline_hours_at_mean=workload['total_pipeline_executions_max']*float(np.mean(seconds))/3600 if seconds else None,
        timing_scope='Recorded local candidate execution only; not a prediction of cloud speed',
        setup_seconds=setup,failure_overhead_seconds=None,setup_and_failures_included=False)


def _ledger(path):
    path=Path(path);value=json.loads(path.read_text(encoding='utf-8'))
    if value.get('protocol')!=PROTOCOL or value.get('format_version')!=1:raise ValueError('Study artifact protocol mismatch')
    if value['source_code_sha256']!=notebook_code_digest(VERIFIED_NOTEBOOK):raise ValueError('Frozen verified notebook code changed')
    if value['experimental_fingerprint']!=experimental_fingerprint(VERIFIED_NOTEBOOK):raise ValueError('Frozen experimental functions changed')
    if value['experimental_fingerprint']!=experimental_fingerprint(FROZEN_V2):raise ValueError('V2 estimator differs')
    if value['expected_reference']!=expected_reference():raise ValueError('Historical reference identity changed')
    if file_digest(value['membership_manifest'])!=value['membership_sha256']:raise ValueError('Frozen membership changed')
    if [(a['method'],a['seed']) for a in value['arms']]!=[(m,s) for s in (42,43,44) for m in STUDY_METHODS]:raise ValueError('Frozen arm order/membership changed')
    if value['plan_sha256']!=stable_hash([(a['key'],a['run_dir'],a['prepared_config_sha256'],a['dependency']) for a in value['arms']]):raise ValueError('Prepared arm plan changed')
    for entry in value['arms']:
        if entry['method']!='shuffled_k' and entry['config_sha256']!=entry['prepared_config_sha256']:raise ValueError('Frozen arm configuration changed')
        run=Path(entry['run_dir']).resolve();study=path.parent.resolve()
        if not run.is_relative_to(study/'runs') or run==study:raise ValueError('Reserved run escapes study runs directory')
        if entry.get('config') is not None and stable_hash(entry['config'])!=entry['config_sha256']:raise ValueError('Arm configuration digest mismatch')
    return value


def _save_ledger(path,value):
    write_json(path,value,overwrite=True)


@contextmanager
def study_owner(ledger_path):
    # OS advisory locks release automatically on process death; lock files remain harmless.
    path=Path(ledger_path).with_suffix('.lock');stream=path.open('a+b');locked=False
    try:
        if os.name=='nt':
            import msvcrt
            if path.stat().st_size==0:stream.write(b'0');stream.flush()
            stream.seek(0)
            try:msvcrt.locking(stream.fileno(),msvcrt.LK_NBLCK,1)
            except OSError as exc:raise RuntimeError('Another runner owns this study') from exc
        else:
            import fcntl
            try:fcntl.flock(stream,fcntl.LOCK_EX|fcntl.LOCK_NB)
            except OSError as exc:raise RuntimeError('Another runner owns this study') from exc
        locked=True;yield
    finally:
        if locked:
            if os.name=='nt':stream.seek(0);msvcrt.locking(stream.fileno(),msvcrt.LK_UNLCK,1)
            else:fcntl.flock(stream,fcntl.LOCK_UN)
        stream.close()


def _study_cfg(method,seed,study_dir,manifest,ledger_path,schedule=None):
    mode='adaptive' if method=='adaptive' else 'scheduled' if method=='shuffled_k' else 'static'
    k=int(method.rsplit('k',1)[1]) if method.startswith('static_k') else 5
    return build_config('formal',allocation_mode=mode,static_k=k,seed=seed,dataset_revision=DEVELOPMENT_REVISION,
        results_root=str(study_dir),membership_manifest=str(manifest),suite_ledger=str(ledger_path),
        wall_limit_seconds=None,ollama_num_gpu=0,allocation_schedule=schedule)


def prepare_study(study_dir='results/adagrpo_verified/studies/formal-v1',wall_limit_seconds=None,
                  _dataset=None,_reference=None):
    """No models or official tests. Private fixture arguments are used only by synthetic checks."""
    root=Path(study_dir);path=root/'study.json'
    if path.exists():
        ledger=_ledger(path)
        if ledger['wall_limit_seconds']!=wall_limit_seconds:raise ValueError('Prepared deadline policy differs')
        return dict(ledger_path=str(path),status=ledger['status'],workload=ledger['workload'])
    if root.exists() and any(root.iterdir()):raise ValueError('Partial/foreign preparation directory; preserve it and select a new destination')
    if wall_limit_seconds is not None and (not math.isfinite(wall_limit_seconds) or wall_limit_seconds<=0):raise ValueError('Invalid study duration')
    reference=_reference or expected_reference()
    if experimental_fingerprint(VERIFIED_NOTEBOOK)!=experimental_fingerprint(FROZEN_V2):raise ValueError('V2 experimental functions must match exactly')
    cfg=build_config('formal',dataset_revision=DEVELOPMENT_REVISION)
    if _dataset is None:cfg,datasets=load_gsm8k(cfg)
    else:datasets=_dataset
    train=datasets['train'];used=historical_membership();evaluated=historical_evaluation_ids()
    excluded_development=[i for i in DEVELOPMENT_POOL if f'gsm8k:train:{i}' in evaluated]
    pool=[i for i in DEVELOPMENT_POOL if f'gsm8k:train:{i}' not in evaluated]
    order=[int(i) for i in np.random.default_rng(20261021).permutation(len(train))]
    fresh=[i for i in order if f'gsm8k:train:{i}' not in used and i not in pool]
    needed=200-len(pool)
    if len(fresh)<needed+50:raise ValueError('Insufficient unused train-split questions')
    pool+=fresh[:needed];validation=fresh[needed:needed+50]
    if {f'gsm8k:train:{i}' for i in pool+validation}&evaluated:raise ValueError('Prior evaluation questions leaked into new study')
    indices=dict(train=pool,validation=validation)
    subsets={k:[dataset_example(datasets,'train',i) for i in ids] for k,ids in indices.items()}
    if len(set(pool))!=200 or len(set(validation))!=50 or set(pool)&set(validation):raise ValueError('Formal membership sizes/leakage')
    root.mkdir(parents=True,exist_ok=True);manifest=root/'membership.json'
    value=dict(protocol=PROTOCOL,dataset_revision=DEVELOPMENT_REVISION,indices=indices,
        ids={k:[e['id'] for e in rows] for k,rows in subsets.items()},content_hashes={k:stable_hash(rows) for k,rows in subsets.items()},
        prior_evaluation_ids=sorted(evaluated),excluded_development_indices=excluded_development,all_previous_membership_ids=sorted(used),selection_seed=20261021,
        official_test_loaded=False)
    write_json(manifest,value)
    arms=[]
    for seed in (42,43,44):
        for method in STUDY_METHODS:
            key=f'{method}:seed{seed}';config=None;selection_id=None;selection_hash=None
            if method!='shuffled_k':
                config=_study_cfg(method,seed,root,manifest,path)
                data=prepare_selection(config,datasets,persist=True);selection_id=data['selection_id'];selection_hash=stable_hash(data['selection'])
            arms.append(dict(key=key,method=method,seed=seed,status='pending',run_dir=str(root/'runs'/method/f'seed{seed}'),
                config=config,config_sha256=stable_hash(config) if config is not None else None,prepared_config_sha256=stable_hash(config) if config is not None else None,
                selection_id=selection_id,selection_hash=selection_hash,
                dependency=f'adaptive:seed{seed}' if method=='shuffled_k' else None,schedule_source=None))
    refs=[reference['source_run']]
    comparison=Path('results/adagrpo_v1_v2/comparison_ledger.json')
    if comparison.exists():refs += [e['run_dir'] for e in json.loads(comparison.read_text(encoding='utf-8'))['steps']]
    ledger=dict(format_version=1,protocol=PROTOCOL,controller_version='credit-refine-v2',sampling_version='candidate-stream-v2',
        source_code_sha256=notebook_code_digest(VERIFIED_NOTEBOOK),experimental_fingerprint=experimental_fingerprint(VERIFIED_NOTEBOOK),
        expected_reference=reference,membership_manifest=str(manifest),membership_sha256=file_digest(manifest),
        arms=arms,status='prepared',created_unix=datetime.now(timezone.utc).timestamp(),started_unix=None,deadline_unix=None,
        wall_limit_seconds=wall_limit_seconds,workload=latency_estimate(refs),official_test_access=False)
    ledger['plan_sha256']=stable_hash([(a['key'],a['run_dir'],a['prepared_config_sha256'],a['dependency']) for a in arms])
    write_json(path,ledger)
    return dict(ledger_path=str(path),status='prepared',workload=ledger['workload'])


_original_check_deadline=check_deadline

def check_deadline(session):
    cfg=session['config']
    if cfg.get('suite_ledger'):
        ledger=json.loads(Path(cfg['suite_ledger']).read_text(encoding='utf-8'))
        deadline=ledger.get('deadline_unix')
        if deadline is not None and datetime.now(timezone.utc).timestamp()>=deadline:raise WallLimitReached('Persisted study deadline reached')
    if session.get('deadline') is not None and time.perf_counter()>=session['deadline']:raise WallLimitReached('Configured run deadline reached')


def _bind_shuffled(ledger,entry):
    source=next(a for a in ledger['arms'] if a['key']==entry['dependency'])
    if source['status']!='completed':raise ValueError('Adaptive source must complete before shuffled K')
    run=verify_run_artifacts(source['run_dir']);schedule=[r['actual_k'] for r in run['train']]
    permutation_seed=entry['seed']+90000
    shuffled=np.random.default_rng(permutation_seed).permutation(schedule).tolist()
    provenance=dict(adaptive_run=source['run_dir'],source_arm_key=source['key'],source_summary_sha256=file_digest(Path(source['run_dir'])/'summary.json'),
        source_latest=json.loads((Path(source['run_dir'])/'latest.json').read_text(encoding='utf-8')),permutation_seed=permutation_seed,source_k=schedule,shuffled_k=shuffled)
    config=_study_cfg('shuffled_k',entry['seed'],Path(entry['run_dir']).parents[2],ledger['membership_manifest'],
        Path(entry['run_dir']).parents[2]/'study.json',shuffled)
    # Same data and seed imply the same schedule/selection as the adaptive source.
    entry.update(config=config,config_sha256=stable_hash(config),selection_id=source['selection_id'],selection_hash=source['selection_hash'],schedule_source=provenance)


def _validate_arm_files(ledger,entry,require_checkpoint=False):
    root=Path(entry['run_dir']);config=entry['config']
    if config is None:raise ValueError('Unbound arm configuration')
    meta_path=root/'config.json'
    if meta_path.exists():
        meta=json.loads(meta_path.read_text(encoding='utf-8'))
        if meta['config']!=config or meta['selection_id']!=entry['selection_id'] or stable_hash(meta['selection'])!=entry['selection_hash']:raise ValueError('Arm metadata/configuration mismatch')
        ref=ledger['expected_reference']
        if meta['initial_policy_digest']!=ref['policy_initial_sha256'] or meta['execution_model']['model']['digest']!=ref['model_digest'] or meta['first_tokens']!=ref['first_tokens']:raise ValueError('Policy/model/action identity mismatch')
    elif require_checkpoint:raise ValueError('Recovery metadata missing')
    checkpoint=root/'latest.json'
    if require_checkpoint or checkpoint.exists():
        pointer=json.loads(checkpoint.read_text(encoding='utf-8'));file=(root/pointer['checkpoint']).resolve()
        if file.parent!=(root/'checkpoints').resolve() or file_digest(file)!=pointer['sha256']:raise ValueError('Recovery checkpoint path/checksum mismatch')
        payload=torch.load(file,map_location='cpu',weights_only=False)
        if payload['protocol']!=PROTOCOL or payload['config']!=config or payload['policy_only']:raise ValueError('Incompatible resumable checkpoint')
        if payload['initial_policy_digest']!=ledger['expected_reference']['policy_initial_sha256'] or payload['runtime']!=ledger['expected_reference']['runtime']:raise ValueError('Checkpoint runtime/initialization changed')
        del payload
    if entry['schedule_source'] is not None:
        source_path=root/'schedule_source.json'
        if source_path.exists():source=json.loads(source_path.read_text(encoding='utf-8'))
        elif meta_path.exists() or checkpoint.exists() or read_jsonl(root/'attempts.jsonl'):raise ValueError('Recorded shuffled work lacks its provenance')
        else:source=entry['schedule_source']
        if source!=entry['schedule_source']:raise ValueError('Shuffled provenance mismatch')
        parent=next(a for a in ledger['arms'] if a['key']==entry['dependency'])
        if file_digest(Path(parent['run_dir'])/'summary.json')!=source['source_summary_sha256']:raise ValueError('Adaptive source changed after binding')
        if json.loads((Path(parent['run_dir'])/'latest.json').read_text(encoding='utf-8'))!=source['source_latest']:raise ValueError('Adaptive source checkpoint changed')
        expected=np.random.default_rng(entry['seed']+90000).permutation(source['source_k']).tolist()
        if source['source_arm_key']!=entry['dependency'] or source['permutation_seed']!=entry['seed']+90000 or expected!=source['shuffled_k'] or config['allocation_schedule']!=expected:raise ValueError('Shuffled schedule or source binding changed')


def _initialize_reserved(ledger,entry):
    cfg=entry['config'];root=Path(entry['run_dir']);session=initialize_experiment(cfg,create_run=False)
    try:
        ref=ledger['expected_reference']
        if session['initial_policy_digest']!=ref['policy_initial_sha256'] or session['llm_metadata']['model']['digest']!=ref['model_digest'] or session['first_tokens']!=ref['first_tokens'] or runtime_identity(session)!=ref['runtime']:raise ValueError('Runtime/model/initial policy changed; prepare a separate reviewed protocol')
        if session['data']['selection_id']!=entry['selection_id']:raise ValueError('Frozen selection changed')
        root.mkdir(parents=True,exist_ok=True);(root/'checkpoints').mkdir(exist_ok=True);session['run_dir']=root
        metadata=dict(config=cfg,selection_id=session['data']['selection_id'],selection=session['data']['selection'],
            initial_policy_digest=session['initial_policy_digest'],first_tokens=session['first_tokens'],execution_model=session['llm_metadata'],hardware=hardware_metadata())
        if entry['schedule_source'] is not None and not (root/'schedule_source.json').exists():write_json(root/'schedule_source.json',entry['schedule_source'])
        if (root/'config.json').exists():_validate_arm_files(ledger,entry)
        else:write_json(root/'config.json',metadata)
        save_checkpoint(session)
        return session
    except BaseException:release_experiment(session);raise


def run_study(ledger_path,resume=False,allow_training=False):
    if not allow_training:raise RuntimeError('Model work disabled: explicit allow_training=True is required')
    path=Path(ledger_path)
    with study_owner(path):
        ledger=_ledger(path)
        if ledger['started_unix'] is not None and not resume:raise RuntimeError('Started study requires explicit resume=True; deadlines never reset')
        if ledger['status']=='completed':return audit_study(path)
        now=datetime.now(timezone.utc).timestamp()
        if ledger['started_unix'] is None:
            ledger['started_unix']=now
            if ledger['wall_limit_seconds'] is not None:ledger['deadline_unix']=now+ledger['wall_limit_seconds']
        ledger['status']='running';_save_ledger(path,ledger)
        try:
            for entry in ledger['arms']:
                if ledger['deadline_unix'] is not None and datetime.now(timezone.utc).timestamp()>=ledger['deadline_unix']:raise WallLimitReached('Original study deadline reached')
                if entry['status']=='completed':
                    _validate_arm_files(ledger,entry,True);verify_run_artifacts(entry['run_dir']);continue
                if entry['method']=='shuffled_k' and entry['config'] is None:_bind_shuffled(ledger,entry);_save_ledger(path,ledger)
                root=Path(entry['run_dir']);_validate_arm_files(ledger,entry)
                summary=root/'summary.json'
                if summary.exists() and json.loads(summary.read_text(encoding='utf-8')).get('completed'):
                    _validate_arm_files(ledger,entry,True);verify_run_artifacts(root);entry['status']='completed';_save_ledger(path,ledger);continue
                entry['status']='initializing';_save_ledger(path,ledger)
                if not (root/'latest.json').exists() and read_jsonl(root/'attempts.jsonl'):raise ValueError('Recorded work has no valid durable checkpoint; preserve artifacts for explicit recovery')
                preflight(entry['config'])
                if (root/'latest.json').exists():
                    _validate_arm_files(ledger,entry,True);session=load_run(root)
                else:session=_initialize_reserved(ledger,entry)
                try:
                    entry['status']='running';_save_ledger(path,ledger);session['progress']=True
                    print('VERIFIED STUDY',entry['key'],str(root),flush=True)
                    train_grpo(session);result=write_summary(session)
                    if not result['completed']:
                        entry['status']='incomplete';ledger['status']='incomplete';_save_ledger(path,ledger);return audit_study(path)
                    verify_run_artifacts(root);entry['status']='completed';_save_ledger(path,ledger)
                finally:release_experiment(session)
            ledger['status']='completed';ledger['finished_unix']=datetime.now(timezone.utc).timestamp();_save_ledger(path,ledger)
            return audit_study(path)
        except BaseException as exc:
            if 'entry' in locals() and entry['status']!='completed':entry['status']='incomplete'
            ledger.update(status='incomplete',last_error=type(exc).__name__+': '+str(exc));_save_ledger(path,ledger);raise


def run_formal_study(*args,**kwargs):
    raise RuntimeError('Use prepare_study and run_study; verified orchestration requires its own frozen ledger')


def run_improvement_suite(*args,**kwargs):
    raise RuntimeError('Historical local suite is frozen; no new local model work is authorized')


def run_operation(cfg=None,operation='initialize',**kwargs):
    raise RuntimeError('Use explicit prepare_study/audit_study/run_study; training and final tests are separate opt-ins')


In [ ]:
def reconcile_run(root):
    """Read-only accounting. Unknown external work is never replaced by zero."""
    root=Path(root);events=read_jsonl(root/'attempts.jsonl')
    if not (root/'attempts.jsonl').exists():
        summary=json.loads((root/'summary.json').read_text(encoding='utf-8')) if (root/'summary.json').exists() else {}
        if summary.get('episode',0)>0 or (root/'result.json').exists() or any(read_jsonl(root/name) for name in ('training.jsonl','validation.jsonl')):raise ValueError('Existing work lacks a supported attempt journal; cost is unknown, not zero')
    starts={};finishes={}
    for event in events:
        if event['kind'].endswith('_attempt_started'):
            if event['attempt_id'] in starts:raise ValueError('Duplicate attempt start')
            starts[event['attempt_id']]=event
        if event['kind'].endswith('_attempt_finished'):
            if event['attempt_id'] in finishes:raise ValueError('Duplicate attempt finish')
            finishes[event['attempt_id']]=event
    if set(finishes)-set(starts):raise ValueError('Finished attempt lacks its journal start')
    training=read_jsonl(root/'training.jsonl');validations=read_jsonl(root/'validation.jsonl')
    if any('attempt_ids' not in row for row in training) or any('records' not in row for row in validations):raise ValueError('Unsupported historical journal schema; cost cannot be reconciled by this protocol')
    committed_train={aid for row in training for aid in row['attempt_ids']}
    committed_eval=set()
    for result in validations:
        committed_eval.update(r['attempt_id'] for r in result['records'])
    for path in sorted(set(root.glob('full_test_*.json'))|set(root.glob('result.json'))):
        result=json.loads(path.read_text(encoding='utf-8'));committed_eval.update(r['attempt_id'] for r in result['records'])
    if sum(len(row['attempt_ids']) for row in training)!=len(committed_train):raise ValueError('Training attempt committed more than once')
    if committed_train&committed_eval or (committed_train|committed_eval)-set(finishes):raise ValueError('Committed work lacks unique finished provenance')
    split_categories=('training','validation','confirmation','full_test','other_evaluation')
    def category(event):
        if event['kind'].startswith('training_'):return 'training'
        split=event.get('split');return split if split in split_categories else 'other_evaluation'
    categories={}
    for name in split_categories:
        ids={aid for aid,e in starts.items() if category(e)==name};done=ids&set(finishes)
        committed=ids&(committed_train if name=='training' else committed_eval)
        counts=Counter((starts[aid].get('evaluation_identity') or ('legacy_episode',starts[aid].get('episode'),starts[aid].get('split')),starts[aid].get('question_id')) for aid in ids) if name!='training' else Counter()
        exact_repeat_identity=all(starts[aid].get('evaluation_identity') is not None for aid in ids) if name!='training' else True
        resources=[finishes[aid].get('resources',{}) for aid in done]
        def metric(key):
            values=[r.get(key) for r in resources];known=[v for v in values if v is not None]
            observed=sum(known)
            return dict(observed=observed,total=observed if done==ids and len(known)==len(ids) else None,
                        unknown_attempts=len(ids)-len(known))
        categories[name]=dict(gross_attempts=len(ids),committed_attempts=len(committed),discarded_or_uncommitted_attempts=len(ids-committed),
            interrupted_attempts=len(ids-done),repeated_attempts=sum(max(0,n-1) for n in counts.values()),repeat_identity_complete=exact_repeat_identity,
            repeat_scope='Exact policy/evaluation identity' if exact_repeat_identity else 'Legacy episode/split proxy; policy weight identity unavailable',
            logical_model_calls=metric('execution_llm_calls'),prompt_tokens=metric('prompt_tokens'),generated_tokens=metric('generated_tokens'),
            observed_execution_seconds=sum(finishes[aid]['seconds'] for aid in done if finishes[aid].get('seconds') is not None),
            total_execution_seconds=sum(finishes[aid]['seconds'] for aid in done) if ids==done and all(finishes[aid].get('seconds') is not None for aid in done) else None,
            physical_http_requests=None,physical_requests_scope='Not measured; logical calls are not physical HTTP requests')
    totals={key:sum(c[key] for c in categories.values()) for key in ('gross_attempts','committed_attempts','discarded_or_uncommitted_attempts','interrupted_attempts','repeated_attempts')}
    diagnostic=[dict(episode=r['episode'],total_variation=r.get('policy_change',{}).get('total_variation'),
        note='Observed equal-reward Adam step; not classified as a defect') for r in training if r.get('zero_advantage_group')]
    mem={k:max((r[k] for r in training if r.get(k) is not None),default=None) for k in ('peak_allocated_gpu_bytes','peak_reserved_gpu_bytes','process_rss_bytes')}
    return dict(root=str(root),categories=categories,totals=totals,controller_seconds=sum(r.get('controller_seconds',0.) for r in training),
        equal_reward_policy_changes=diagnostic,memory_snapshots=mem,
        wall_segments=[dict(kind=e['kind'],seconds=e.get('seconds')) for e in events if e['kind'].endswith('_segment_finished')],
        authoritative_cost='All started journal attempts; observations are lower bounds when work is incomplete')


def unified_accounting(new_roots=(),historical_roots=()):
    historical={str(Path(p).resolve()) for p in historical_roots};new={str(Path(p).resolve()) for p in new_roots}
    if new&historical:raise ValueError('The same run cannot count as both historical and new work')
    result={}
    for label,paths in [('new',new),('historical_reused',historical)]:
        runs=[reconcile_run(p) for p in sorted(paths)]
        metrics={key:sum(r['totals'][key] for r in runs) for key in ('gross_attempts','committed_attempts','discarded_or_uncommitted_attempts','interrupted_attempts','repeated_attempts')}
        for key in ('logical_model_calls','prompt_tokens','generated_tokens'):
            values=[c[key] for r in runs for c in r['categories'].values()]
            metrics[key]=dict(observed=sum(v['observed'] for v in values),total=sum(v['total'] for v in values) if all(v['total'] is not None for v in values) else None,unknown_attempts=sum(v['unknown_attempts'] for v in values))
        metrics['physical_http_requests']=None
        result[label]=dict(unique_runs=len(runs),runs=runs,totals=metrics)
    return result


def audit_study(ledger_path):
    """Does not repair journals, load models, create checkpoints, or publish files."""
    ledger=_ledger(ledger_path);arms=[]
    for entry in ledger['arms']:
        root=Path(entry['run_dir']);errors=[];valid=False
        try:
            if entry['config'] is not None:_validate_arm_files(ledger,entry,entry['status'] in ('running','completed'))
            if entry['status']=='completed':verify_run_artifacts(root);valid=True
        except (ValueError,KeyError,OSError) as exc:errors.append(type(exc).__name__+': '+str(exc))
        arms.append(dict(key=entry['key'],status=entry['status'],completed_and_audited=valid,errors=errors,accounting=reconcile_run(root)))
    completed=all(a['completed_and_audited'] for a in arms)
    return dict(protocol=PROTOCOL,status=ledger['status'],completed=completed,arms=arms,workload=ledger['workload'],
        acceptance_ready=completed and all(len(list(Path(e['run_dir']).glob('full_test_*.json')))==1 for e in ledger['arms']),
        official_test_access_recorded=any(list(Path(e['run_dir']).glob('full_test_*.json')) for e in ledger['arms']),
        deadline_unix=ledger['deadline_unix'],source_code_sha256=ledger['source_code_sha256'])


def formal_acceptance(ledger_path):
    audit=audit_study(ledger_path)
    if not audit['acceptance_ready']:raise ValueError('Acceptance requires all fifteen completed audited arms and explicit official testing')
    ledger=_ledger(ledger_path)
    return acceptance_report([verify_run_artifacts(e['run_dir']) for e in ledger['arms']],resamples=10000,bootstrap_seed=2026)


PREPARE_STUDY=False
LAUNCH_TRAINING=False
if PREPARE_STUDY:
    PREPARED_STUDY=prepare_study()
if LAUNCH_TRAINING:
    raise RuntimeError('Call run_study with an explicit frozen ledger and allow_training=True; no automatic launch here')


In [ ]:
def verified_self_checks():
    import ast,io
    from contextlib import redirect_stdout
    from unittest.mock import patch
    from types import SimpleNamespace
    checks=[]
    def check(name,value):
        if not value:raise AssertionError(name)
        checks.append(name)
    def raises(name,fn,kind=ValueError):
        try:fn()
        except kind:checks.append(name)
        else:raise AssertionError(name+' did not raise')
    with redirect_stdout(io.StringIO()):base=adagrpo_self_checks()
    check('retained V2 check count',base['passed']==163)
    check('all frozen experimental functions match',experimental_fingerprint(VERIFIED_NOTEBOOK)==experimental_fingerprint(FROZEN_V2))
    check('formal upper workload',formal_workload_estimate()['total_pipeline_executions_max']==34935)
    raises('training guard before initialization',lambda:run_study('nonexistent'),RuntimeError)
    raises('formal historical runner disabled',lambda:run_formal_study('unused',allow_formal_training=True),RuntimeError)
    tokens={a:(0 if a=='CoT' else 1 if a=='Predict' else 5 if a=='stop' else 2+SIGNATURES.index(a)%3) for a in ACTIONS}
    class TinyPolicy(torch.nn.Module):
        def __init__(self):
            super().__init__();self.logits=torch.nn.Parameter(torch.tensor([.1,-.1,.2,-.2,.05,0.]));self.config=SimpleNamespace(n_positions=1024)
        def forward(self,**kwargs):return SimpleNamespace(logits=self.logits.reshape(1,1,-1).expand(1,2,-1))
    class TinyTokenizer:
        def __call__(self,*args,**kwargs):return {'input_ids':torch.tensor([[0,1]])}
    fake_data={'train':[dict(question=f'fixture q{i}',answer='reason #### 1') for i in range(1000)]}
    def execute(t,q,lm):
        lm.events.append(dict(seconds=.001,prompt_tokens=3,generated_tokens=2,error=None,completed=True))
        return dict(output='[1]' if t['module']=='Predict' else '[0]',success=True,error=None,seconds=.001)
    def fixture(cfg,create_run=False):
        if create_run:raise AssertionError('Runner must reserve paths before initialization')
        seed_everything(cfg['seed']);model=TinyPolicy().eval()
        return dict(config=cfg,data=prepare_selection(cfg,fake_data,persist=False),lm=SimpleNamespace(events=[]),
            llm_metadata=dict(version={'version':'fixture'},model={'digest':'fixture'}),policy=model,tokenizer=TinyTokenizer(),
            optimizer=torch.optim.Adam(model.parameters(),lr=cfg['learning_rate'],foreach=False),first_tokens=tokens,
            initial_policy_digest=parameter_digest(model),run_dir=None,in_episode=False,evaluation_only=False,deadline=None,
            state=dict(episode=0,updates=0,train=[],validation=[],seen=[],best=None,status='initialized',gross_training_attempts=0))
    cfg=build_config('smoke',policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=2)
    first=fixture(cfg);runtime=runtime_identity(first)
    reference=dict(source_run='unused-fixture',config_sha256='fixture',policy_initial_sha256=first['initial_policy_digest'],
        model_digest='fixture',first_tokens=tokens,runtime=runtime)
    # Compare actual copied functions to the frozen original in a distinct namespace.
    frozen_ns={'__name__':'verified_frozen_fixture'}
    with redirect_stdout(io.StringIO()):
        notebook=json.loads(Path(FROZEN_V2).read_text(encoding='utf-8'))
        for i,c in enumerate(notebook['cells']):
            if c['cell_type']=='code':exec(compile(''.join(c['source']),f'frozen-{i}','exec'),frozen_ns)
    check('frozen configuration except artifact identifiers',{k:v for k,v in build_config('smoke').items() if k not in ('protocol','results_root')}=={k:v for k,v in frozen_ns['build_config']('smoke').items() if k not in ('protocol','results_root')})
    left,right=fixture(cfg),fixture(cfg)
    original_cfg=copy.deepcopy(cfg);original_cfg['protocol']='adagrpo-gsm8k-numeric-figure2-v2'
    right['config']=original_cfg
    lt=[];rt=[]
    for index in range(6):
        lg,_=candidate_generator(left,'training',1,index);rg,_=frozen_ns['candidate_generator'](right,'training',1,index)
        lt.append(generate_pipeline(left['policy'],left['tokenizer'],tokens,'same question',cfg,generator=lg))
        rt.append(frozen_ns['generate_pipeline'](right['policy'],right['tokenizer'],tokens,'same question',original_cfg,generator=rg))
    check('exact frozen sampled trajectories',lt==rt)
    lr=[compute_reward('',('[1]' if t['module']=='Predict' else '[0]'),'1')['reward'] for t in lt]
    rr=[frozen_ns['compute_reward']('',('[1]' if t['module']=='Predict' else '[0]'),'1')['reward'] for t in rt]
    check('exact frozen numeric rewards',lr==rr)
    lu=update_policy(left,'same question',lt,[1.,0.,1.,0.,1.,0.]);ru=frozen_ns['update_policy'](right,'same question',rt,[1.,0.,1.,0.,1.,0.])
    check('identical gradients and loss',lu['gradient_norm']==ru['gradient_norm'] and lu['loss']==ru['loss'])
    check('identical learned parameters',torch.equal(left['policy'].logits,right['policy'].logits))
    def optimizer_equal(a,b):
        if isinstance(a,torch.Tensor):return isinstance(b,torch.Tensor) and torch.equal(a,b)
        if isinstance(a,dict):return a.keys()==b.keys() and all(optimizer_equal(v,b[k]) for k,v in a.items())
        if isinstance(a,list):return len(a)==len(b) and all(optimizer_equal(x,y) for x,y in zip(a,b))
        return a==b
    check('identical Adam state',optimizer_equal(left['optimizer'].state_dict(),right['optimizer'].state_dict()))
    lt=[];rt=[]
    for index in range(6):
        lg,_=candidate_generator(left,'training',2,index);rg,_=frozen_ns['candidate_generator'](right,'training',2,index)
        lt.append(generate_pipeline(left['policy'],left['tokenizer'],tokens,'same question',cfg,generator=lg))
        rt.append(frozen_ns['generate_pipeline'](right['policy'],right['tokenizer'],tokens,'same question',original_cfg,generator=rg))
    update_policy(left,'same question',lt,[1.]*6);frozen_ns['update_policy'](right,'same question',rt,[1.]*6)
    check('equal reward Adam behavior unchanged',torch.equal(left['policy'].logits,right['policy'].logits) and optimizer_equal(left['optimizer'].state_dict(),right['optimizer'].state_dict()))
    release_experiment(first);release_experiment(left);release_experiment(right)
    with tempfile.TemporaryDirectory(prefix='adagrpo-verified-') as temp:
        tmp=Path(temp)
        membership_fixture=tmp/'membership_fixture';(membership_fixture/'membership').mkdir(parents=True)
        write_json(membership_fixture/'config.json',dict(selection=dict(ids=dict(train=['gsm8k:train:1'],validation=['gsm8k:train:2']))))
        write_json(membership_fixture/'membership'/'confirmation.json',dict(examples=[dict(id='gsm8k:train:3')],excluded_ids=['gsm8k:train:1']))
        check('evaluation membership ignores exclusion lists',historical_evaluation_ids(membership_fixture)=={'gsm8k:train:2','gsm8k:train:3'})
        patches=dict(expected_reference=lambda:copy.deepcopy(reference),historical_membership=lambda:set(),historical_evaluation_ids=lambda:set(),
            DEVELOPMENT_POOL=list(range(50)),initialize_experiment=fixture,preflight=lambda cfg:dict(fixture=True),
            execute_pipeline=execute,ollama_snapshot=lambda cfg:dict(fixture=True),
            latency_estimate=lambda roots:dict(**formal_workload_estimate(),observations=0,estimated_pipeline_hours_at_mean=None))
        with patch.dict(globals(),patches):
            plan=prepare_study(tmp/'study',wall_limit_seconds=600,_dataset=fake_data,_reference=reference)
            path=Path(plan['ledger_path']);ledger=_ledger(path)
            check('preparation does not start clock',ledger['started_unix'] is None and ledger['deadline_unix'] is None)
            check('fifteen frozen ordered arms',len(ledger['arms'])==15 and ledger['arms'][4]['dependency']=='adaptive:seed42')
            check('formal membership sizes',len(json.loads(Path(ledger['membership_manifest']).read_text())['ids']['train'])==200)
            with patch.dict(globals(),dict(historical_evaluation_ids=lambda:{'gsm8k:train:3'},historical_membership=lambda:{'gsm8k:train:3'})):
                excluded=prepare_study(tmp/'excluded',_dataset=fake_data,_reference=reference)
                membership=json.loads(Path(_ledger(excluded['ledger_path'])['membership_manifest']).read_text())
                check('prior evaluation removed from reused development pool',3 not in membership['indices']['train'] and len(membership['indices']['train'])==200 and membership['excluded_development_indices']==[3])
            check('prepare idempotent',prepare_study(tmp/'study',wall_limit_seconds=600)['ledger_path']==str(path))
            raises('prepared duration cannot change',lambda:prepare_study(tmp/'study',wall_limit_seconds=601))
            with study_owner(path):raises('concurrent runner rejected',lambda:run_study(path,allow_training=True),RuntimeError)
            # Shorten only this explicitly synthetic fixture; reseal its preparation metadata.
            for entry in ledger['arms']:
                if entry['config'] is not None:
                    entry['config'].update(num_episodes=2,training_budget=2*(entry['config']['static_k'] if entry['config']['allocation_mode']=='static' else 4),validation_interval=2,checkpoint_interval=1,policy_device='cpu')
                    entry['config_sha256']=entry['prepared_config_sha256']=stable_hash(entry['config'])
            ledger['plan_sha256']=stable_hash([(a['key'],a['run_dir'],a['prepared_config_sha256'],a['dependency']) for a in ledger['arms']]);_save_ledger(path,ledger)
            real_study_cfg=_study_cfg
            def tiny_study_cfg(method,seed,study_dir,manifest,ledger_path,schedule=None):
                mode='adaptive' if method=='adaptive' else 'scheduled' if method=='shuffled_k' else 'static'
                k=int(method.rsplit('k',1)[1]) if method.startswith('static_k') else 5
                return build_config('formal',allocation_mode=mode,static_k=k,seed=seed,dataset_revision=DEVELOPMENT_REVISION,
                    results_root=str(study_dir),membership_manifest=str(manifest),suite_ledger=str(ledger_path),
                    wall_limit_seconds=None,ollama_num_gpu=0,allocation_schedule=schedule,num_episodes=2,
                    validation_interval=2,checkpoint_interval=1,policy_device='cpu')
            # A failed initialization leaves a reserved arm, no journal work, and a durable deadline.
            def unavailable(*a,**kw):raise RuntimeError('synthetic initialization interruption')
            with patch.dict(globals(),dict(initialize_experiment=unavailable)):
                raises('initialization interruption preserved',lambda:run_study(path,allow_training=True),RuntimeError)
            after=_ledger(path);deadline=after['deadline_unix']
            check('reserved directory persists before initialization',after['arms'][0]['status']=='incomplete' and after['arms'][0]['run_dir']==ledger['arms'][0]['run_dir'])
            raises('explicit resume required',lambda:run_study(path,allow_training=True),RuntimeError)
            original_attempt=recorded_attempt;once=[False]
            def pause_validation(*args,**kwargs):
                item=original_attempt(*args,**kwargs)
                if args[2]=='evaluation' and not once[0]:once[0]=True;raise WallLimitReached('synthetic validation pause')
                return item
            with patch.dict(globals(),dict(recorded_attempt=pause_validation,_study_cfg=tiny_study_cfg)):
                with redirect_stdout(io.StringIO()):paused=run_study(path,resume=True,allow_training=True)
            check('validation interruption returns incomplete',not paused['completed'] and once[0])
            check('deadline unchanged through validation resume',_ledger(path)['deadline_unix']==deadline)
            pending_root=Path(_ledger(path)['arms'][0]['run_dir'])
            ev_before=[e for e in read_jsonl(pending_root/'attempts.jsonl') if e['kind']=='evaluation_attempt_finished']
            check('validation finish durable before progress reuse',len(ev_before)==1)
            # A partial training group is journaled then rolled back, charging one lost attempt.
            once_train=[False]
            def pause_training(*args,**kwargs):
                item=original_attempt(*args,**kwargs)
                if args[2]=='training' and not once_train[0]:once_train[0]=True;raise WallLimitReached('synthetic training pause')
                return item
            with patch.dict(globals(),dict(recorded_attempt=pause_training,_study_cfg=tiny_study_cfg)):
                with redirect_stdout(io.StringIO()):paused=run_study(path,resume=True,allow_training=True)
            check('collection interruption returns incomplete',not paused['completed'] and once_train[0])
            check('interrupted training remains charged',reconcile_run(pending_root)['categories']['training']['gross_attempts']==1)
            # Simulate a process dying after a completed arm, before the next initialization.
            real_preflight=globals()['preflight']
            def pause_between(cfg):
                current=_ledger(path)
                if current['arms'][0]['status']=='completed' and current['arms'][1]['status']!='completed':raise RuntimeError('synthetic between-arm interruption')
                return real_preflight(cfg)
            with patch.dict(globals(),dict(preflight=pause_between,_study_cfg=tiny_study_cfg)):
                with redirect_stdout(io.StringIO()):raises('between-arm interruption preserved',lambda:run_study(path,resume=True,allow_training=True),RuntimeError)
            saved=_ledger(path);check('first arm completed before interruption',saved['arms'][0]['status']=='completed')
            first_bytes=(pending_root/'attempts.jsonl').read_bytes()
            # Validate read-only audit and stale configuration rejection before completing fixtures.
            before={str(p):file_digest(p) for p in (tmp/'study').rglob('*') if p.is_file()}
            audit_study(path)
            check('audit is read only',before=={str(p):file_digest(p) for p in (tmp/'study').rglob('*') if p.is_file()})
            damaged=copy.deepcopy(saved);damaged['arms'][1]['config']['learning_rate']=2e-5;_save_ledger(path,damaged)
            raises('modified configuration rejected',lambda:_ledger(path));_save_ledger(path,saved)
            with patch.dict(globals(),dict(_study_cfg=tiny_study_cfg)):
                with redirect_stdout(io.StringIO()):complete=run_study(path,resume=True,allow_training=True)
            check('all synthetic arms complete and audited',complete['completed'] and len(complete['arms'])==15)
            check('completed arm journal not rerun',(pending_root/'attempts.jsonl').read_bytes()==first_bytes)
            finished=_ledger(path);check('deadline not reset',finished['deadline_unix']==deadline)
            for seed in (42,43,44):
                shuffled=next(a for a in finished['arms'] if a['key']==f'shuffled_k:seed{seed}');src=shuffled['schedule_source']
                check('shuffled source identity '+str(seed),src['source_arm_key']==f'adaptive:seed{seed}')
                check('shuffled counts preserved '+str(seed),sorted(src['source_k'])==sorted(src['shuffled_k']))
                check('shuffled permutation deterministic '+str(seed),np.random.default_rng(seed+90000).permutation(src['source_k']).tolist()==src['shuffled_k'])
            evals=[e for e in read_jsonl(pending_root/'attempts.jsonl') if e['kind']=='evaluation_attempt_started']
            check('partial validation not repeated',len(evals)==100 and len({(e['evaluation_identity'],e['question_id']) for e in evals})==100)
            check('discarded training accounted',reconcile_run(pending_root)['categories']['training']['discarded_or_uncommitted_attempts']==1)
            raises('no acceptance without official tests',lambda:formal_acceptance(path))
            pristine=(pending_root/'latest.json').read_bytes();pointer=json.loads(pristine);pointer['sha256']='0'*64
            write_json(pending_root/'latest.json',pointer,overwrite=True);raises('checkpoint checksum rejected',lambda:_validate_arm_files(finished,finished['arms'][0],True));(pending_root/'latest.json').write_bytes(pristine)
            # Missing checkpoint with recorded work is never reinitialized.
            latest=pending_root/'latest.json';backup=latest.read_bytes();latest.unlink()
            changed=copy.deepcopy(finished);changed['status']='incomplete';changed['arms'][0]['status']='incomplete'
            (pending_root/'summary.json').rename(pending_root/'saved_summary.json');_save_ledger(path,changed)
            raises('missing checkpoint never resets work',lambda:run_study(path,resume=True,allow_training=True));latest.write_bytes(backup)
            (pending_root/'saved_summary.json').rename(pending_root/'summary.json');_save_ledger(path,finished)
            check('missing-checkpoint failure did not delete journal',(pending_root/'attempts.jsonl').read_bytes()==first_bytes)
            # Unmatched started work and absent usage are reported as unknown totals.
            account=tmp/'account';account.mkdir()
            append_journal(dict(run_dir=account),dict(kind='evaluation_attempt_started',attempt_id='pending',question_id='q',split='confirmation',evaluation_identity='i'))
            a=reconcile_run(account)['categories']['confirmation']
            check('pending external calls unknown',a['logical_model_calls']['total'] is None and a['logical_model_calls']['unknown_attempts']==1)
            check('pending tokens unknown',a['prompt_tokens']['total'] is None and a['generated_tokens']['total'] is None)
            check('physical requests unknown',a['physical_http_requests'] is None)
            check('interruption and evaluation category',a['interrupted_attempts']==1 and a['gross_attempts']==1)
            both=unified_accounting(historical_roots=[account,account.resolve()])
            check('archive references deduplicated',both['historical_reused']['unique_runs']==1 and both['historical_reused']['totals']['gross_attempts']==1)
            check('archive not charged as new work',both['new']['totals']['gross_attempts']==0)
            check('aggregate unknown usage retained',both['historical_reused']['totals']['logical_model_calls']['total'] is None and both['historical_reused']['totals']['prompt_tokens']['total'] is None)
            raises('cross-scope duplicate rejected',lambda:unified_accounting([account],[account]))
            legacy=tmp/'legacy';legacy.mkdir();write_json(legacy/'summary.json',dict(episode=1))
            raises('unsupported historical work is not zero',lambda:reconcile_run(legacy))
            append_journal(dict(run_dir=legacy),dict(kind='legacy_format'))
            write_jsonl(legacy/'training.jsonl',[dict(episode=1)])
            raises('unsupported journal schema fails clearly',lambda:reconcile_run(legacy))
            old_protocol=copy.deepcopy(finished);old_protocol['protocol']='adagrpo-gsm8k-numeric-figure2-v2';_save_ledger(path,old_protocol)
            raises('historical study protocol rejected',lambda:_ledger(path));_save_ledger(path,finished)
            altered=copy.deepcopy(finished);altered['arms']=altered['arms'][::-1];_save_ledger(path,altered)
            raises('arm reordering rejected',lambda:_ledger(path));_save_ledger(path,finished)
    real=expected_reference();signature_ids={a:t for a,t in real['first_tokens'].items() if a not in ('CoT','Predict','stop')}
    check('actual signature count is fifteen',len(signature_ids)==15)
    check('actual distinct first tokens is eight',len(set(signature_ids.values()))==8)
    return dict(passed=base['passed']+len(checks),retained_v2_checks=base['passed'],new_checks=len(checks),checks=checks,
        no_external_inference=True,no_real_model_training=True,synthetic_optimizer_steps_only=True)


## Verified workflow
Run the definition cells with the existing **AdaGRPO GPU** kernel. They do not initialize models.
```python
checks = verified_self_checks()  # synthetic policies only
prepared = prepare_study()      # train-split data and ledger only
audit = audit_study(prepared["ledger_path"])
```
Training is a later explicit operation: `run_study(path, allow_training=True)`. After interruption use `resume=True`; it keeps the original clock and attempt costs. A completed study is audited and reused. Final testing remains a separate explicit `load_run(..., for_resume=False)` / `test_model()` operation. `formal_acceptance(path)` requires all fifteen complete arms and their official-test records.

The verified artifact protocol rejects old checkpoints. Historical artifacts are read-only inputs to `unified_accounting`. Runtime/model identity is pinned to the reference recorded at preparation; a changed runtime requires a separately reviewed study rather than silently resuming. No cloud or local model work was launched while implementing this notebook.


## V3 frozen controller and two-hour study
The V2 control dispatches to the original controller. Both arms share policy, reward, loss, optimizer, runtime and candidate streams. All old orchestration entry points are superseded by the explicit study interfaces below.


In [ ]:
# V3 extends the self-contained verified V2 core above. Definition cells are inert.
V3_NOTEBOOK = 'AdaGRPO_GSM8K_v3.ipynb'
V3_ROOT = Path('results/adagrpo_v3')
V3_PROTOCOL = 'adagrpo-gsm8k-numeric-figure2-explore-v3'
V3_CONTROLLER = 'warmup5-credit-refine-v3'
V2_CONTROLLER = 'credit-refine-v2'
V3_MODEL_DIGEST = 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72'
PROTOCOL = V3_PROTOCOL
_v3_base_build = build_config
_v3_base_budget = budget_snapshot
_v3_base_collect = collect_group
_v3_base_decision = controller_decision
_v3_base_reconcile = reconcile_run


def build_config(preset=None, **overrides):
    controller = overrides.pop('controller_version', V2_CONTROLLER)
    warmup = overrides.pop('warmup_episodes', 5 if controller == V3_CONTROLLER else 0)
    if controller not in (V2_CONTROLLER, V3_CONTROLLER): raise ValueError('Unsupported frozen controller')
    if warmup != (5 if controller == V3_CONTROLLER else 0): raise ValueError('Warm-up length is frozen')
    cfg = _v3_base_build(preset, controller_version=V2_CONTROLLER, **overrides)
    cfg.update(controller_version=controller, warmup_episodes=warmup)
    if controller == V3_CONTROLLER:
        if cfg['allocation_mode'] != 'adaptive': raise ValueError('V3 requires adaptive allocation')
        minimum = 5 * min(5, cfg['num_episodes']) + 2 * max(0, cfg['num_episodes']-5)
        if cfg['training_budget'] < minimum: raise ValueError('Budget cannot reserve warm-up and future minimum groups')
    return cfg


def v3_allocation_snapshot(cfg, completed, gross):
    if not 0 <= completed < cfg['num_episodes'] or gross < 0: raise ValueError('Invalid episode or gross cost')
    current=completed+1; future_warmup=max(0,5-current)
    future_adaptive=cfg['num_episodes']-current-future_warmup
    reserve=5*future_warmup+2*future_adaptive; remaining=cfg['training_budget']-gross
    bank=max(0,4*completed-gross); warmup=current<=5
    allowance=5 if warmup else min(6,4+bank,remaining-reserve)
    if remaining-reserve < (5 if warmup else 2) or allowance < (5 if warmup else 2):
        raise BudgetExhausted('Gross budget cannot reserve warm-up/adaptive minimums')
    return dict(allowance=allowance,bank=bank,gross_before=gross,future_reservation=reserve,
        phase='warmup' if warmup else 'adaptive',future_warmup=future_warmup,
        future_adaptive=future_adaptive,warmup_debt=max(0,gross-4*completed))


def budget_snapshot(session):
    if session['config']['controller_version'] != V3_CONTROLLER: return _v3_base_budget(session)
    return v3_allocation_snapshot(session['config'],session['state']['episode'],journal_counts(session))


def controller_decision(cfg,trajectories,rewards,first_tokens,neff,allowance,episode=None):
    if cfg.get('controller_version') != V3_CONTROLLER:
        return _v3_base_decision(cfg,trajectories,rewards,first_tokens,neff,allowance)
    if episode is None or not 1<=episode<=cfg['num_episodes']: raise ValueError('V3 requires the exact episode')
    if any(r not in (0.,1.) for r in rewards): raise ValueError('Controller requires binary rewards')
    if episode<=5:
        if allowance!=5 or len(rewards)>5: raise ValueError('Warm-up requires five retained candidates')
        return dict(stop=len(rewards)==5,reason='warmup_five' if len(rewards)==5 else 'warmup_collect',
            actual_k=len(rewards),neff=neff,allowance=allowance,phase='warmup',
            **contrast_statistics(trajectories,rewards,first_tokens))
    return dict(_v3_base_decision(cfg,trajectories,rewards,first_tokens,neff,allowance),phase='adaptive')


def collect_group(session,example,probe=None):
    if session['config']['controller_version']!=V3_CONTROLLER: return _v3_base_collect(session,example,probe)
    cfg=session['config']; budget=budget_snapshot(session); allowance=budget['allowance']
    before=time.perf_counter();probe=probe or probe_policy(session,example['question'])
    distribution=effective_policy_distribution(probe,session['first_tokens'])
    overhead=time.perf_counter()-before;items=[];trace=[]
    target=5 if budget['phase']=='warmup' else min(2,allowance)
    while True:
        while len(items)<target: items.append(recorded_attempt(session,example,'training',candidate=len(items)))
        before=time.perf_counter()
        decision=controller_decision(cfg,[i['trajectory'] for i in items],[i['reward']['reward'] for i in items],
            session['first_tokens'],distribution['neff'],allowance,episode=session['state']['episode']+1)
        trace.append(decision);overhead+=time.perf_counter()-before
        if decision['stop']: break
        target=min(allowance,len(items)+2)
    return dict(trajectories=[i['trajectory'] for i in items],execution_results=[i['execution'] for i in items],
        reward_details=[i['reward'] for i in items],attempt_ids=[i['attempt_id'] for i in items],actual_k=len(items),
        stop_reason=trace[-1]['reason'],decision_trace=trace,effective_distribution=distribution,
        controller_seconds=overhead,allowance=allowance,bank=budget['bank'],gross_before=budget['gross_before'])


def _v3_atomic_replace(source,target):
    for index in range(8):
        try: os.replace(source,target);return
        except OSError as exc:
            if getattr(exc,'winerror',None) not in (5,32,33) or index==7: raise
            time.sleep(.05*(index+1))


def write_json(path,value,overwrite=False):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and not overwrite: raise FileExistsError(path)
    tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with tmp.open('w',encoding='utf-8') as stream:
            json.dump(value,stream,ensure_ascii=False,indent=2,allow_nan=False);stream.flush();os.fsync(stream.fileno())
        _v3_atomic_replace(tmp,path)
    finally: tmp.unlink(missing_ok=True)


def write_jsonl(path,rows):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        with tmp.open('w',encoding='utf-8') as stream:
            for row in rows: stream.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n')
            stream.flush();os.fsync(stream.fileno())
        _v3_atomic_replace(tmp,path)
    finally: tmp.unlink(missing_ok=True)


def v3_config(controller=V3_CONTROLLER,manifest=None,ledger_path=None):
    return build_config('screen',controller_version=controller,num_episodes=30,train_pool_size=50,
        validation_size=10,validation_interval=30,checkpoint_interval=1,seed=42,data_seed=42,
        dataset_revision=DEVELOPMENT_REVISION,ollama_num_ctx=4096,execution_max_tokens=512,
        ollama_num_gpu=0,training_budget=120,wall_limit_seconds=None,results_root=str(V3_ROOT),
        membership_manifest=str(manifest) if manifest else None,suite_ledger=str(ledger_path) if ledger_path else None,
        test_scope='deferred_no_official_test')


def v3_historical_snapshot():
    paths=set(Path('.').glob('*.ipynb'))|{p for p in Path('results').rglob('*') if p.is_file() and not p.is_relative_to(V3_ROOT)}
    paths.discard(Path(V3_NOTEBOOK))
    return {str(p):file_digest(p) for p in sorted(paths) if not p.name.endswith(('.lock','.tmp'))}


def v3_exclusions():
    used=set();provenance={}
    def visit(value):
        if isinstance(value,str) and re.fullmatch(r'gsm8k:train:\d+',value): used.add(value)
        elif isinstance(value,dict):
            indices=value.get('indices',{})
            if isinstance(indices,dict):
                for key in ('train','validation','confirmation'):
                    if isinstance(indices.get(key),list): used.update(f'gsm8k:train:{i}' for i in indices[key] if type(i) is int)
            for item in value.values(): visit(item)
        elif isinstance(value,list):
            for item in value: visit(item)
    for p in sorted(Path('results').rglob('*')):
        if not p.is_file() or p.is_relative_to(V3_ROOT) or p.suffix not in ('.json','.jsonl'): continue
        try: value=json.loads(p.read_text(encoding='utf-8-sig')) if p.suffix=='.json' else read_jsonl(p)
        except (ValueError,UnicodeError) as exc: raise ValueError(f'Cannot audit historical membership: {p}') from exc
        before=len(used);visit(value)
        if len(used)>before: provenance[str(p)]=file_digest(p)
    used.update(f'gsm8k:train:{i}' for i in np.random.default_rng(42).permutation(7473)[:250])
    return sorted(used),dict(files=provenance,ahnaf_reconstruction='D036: default_rng(42).permutation(7473)[:250]')


def _v3_seal(ledger):
    keys=('protocol','format_version','source_code_sha256','membership_sha256','exclusions_sha256','source_hashes',
        'expected_reference','wall_limit_seconds','clean_workload_upper','arms','smoke_config','smoke_run_dir')
    payload={k:ledger[k] for k in keys}
    payload['arms']=[{k:a[k] for k in ('key','run_dir','config','selection_id')} for a in ledger['arms']]
    return stable_hash(payload)


def _v3_read(ledger_path):
    path=Path(ledger_path);value=json.loads(path.read_text(encoding='utf-8'))
    if value.get('protocol')!=PROTOCOL or value.get('format_version')!=3: raise ValueError('V3 study protocol mismatch')
    if notebook_code_digest(V3_NOTEBOOK)!=value['source_code_sha256']: raise ValueError('Frozen V3 code changed')
    if file_digest(path.parent/'membership.json')!=value['membership_sha256'] or file_digest(path.parent/'exclusions.json')!=value['exclusions_sha256']: raise ValueError('Frozen membership changed')
    if value.get('plan_sha256')!=_v3_seal(value): raise ValueError('Frozen study configuration/order changed')
    clock=path.parent/'study_clock.json'
    if clock.exists():
        timing=json.loads(clock.read_text())
        if timing['plan_sha256']!=value['plan_sha256'] or timing['deadline_unix']-timing['started_unix']!=7200: raise ValueError('Frozen clock identity changed')
        if value.get('started_unix') is not None and (value['started_unix']!=timing['started_unix'] or value['deadline_unix']!=timing['deadline_unix']): raise ValueError('Original deadline changed')
    for a in value['arms']:
        if Path(a['run_dir']).resolve().parent!=(path.parent/'runs').resolve(): raise ValueError('Reserved arm path escaped study')
    return value


def prepare_study(study_dir='results/adagrpo_v3/screen_seed42'):
    root=Path(study_dir);path=root/'study.json'
    if path.exists(): return _v3_read(path)
    if not root.resolve().is_relative_to(V3_ROOT.resolve()): raise ValueError('Study must remain in V3 namespace')
    cfg,datasets=load_gsm8k(v3_config());blocked,provenance=v3_exclusions()
    if len(datasets['train'])!=7473 or 'test' in datasets: raise ValueError('Load only pinned GSM8K train split')
    order=np.random.default_rng(2026102142).permutation(len(datasets['train'])).tolist();blocked_set=set(blocked)
    chosen=[i for i in order if f'gsm8k:train:{i}' not in blocked_set][:80]
    if len(chosen)!=80: raise ValueError('Insufficient unseen questions')
    indices=dict(train=chosen[:50],validation=chosen[50:60],confirmation=chosen[60:80])
    subsets={k:[dataset_example(datasets,'train',i) for i in ids] for k,ids in indices.items()}
    manifest=dict(protocol=PROTOCOL,dataset_revision=cfg['dataset_revision'],
        indices={k:indices[k] for k in ('train','validation')},
        ids={k:[e['id'] for e in subsets[k]] for k in ('train','validation')},
        content_hashes={k:stable_hash(subsets[k]) for k in ('train','validation')},selection_seed=2026102142,
        excluded_ids=blocked,confirmation_examples=subsets['confirmation'])
    root.mkdir(parents=True,exist_ok=True);write_json(root/'membership.json',manifest)
    write_json(root/'exclusions.json',dict(ids=blocked,provenance=provenance))
    ref=expected_reference();reference={k:ref[k] for k in ('policy_initial_sha256','model_digest','first_tokens','runtime')}
    if reference['model_digest']!=V3_MODEL_DIGEST: raise ValueError('Selected execution model differs')
    arms=[]
    for key,controller in [('v2',V2_CONTROLLER),('v3',V3_CONTROLLER)]:
        arm_cfg=v3_config(controller,root/'membership.json',path);selection,_=selection_payload(arm_cfg,datasets)
        arms.append(dict(key=key,run_dir=str(root/'runs'/key),config=arm_cfg,selection_id=stable_hash(selection),status='pending',confirmation_status='pending'))
    if arms[0]['selection_id']!=arms[1]['selection_id']: raise ValueError('Cross-arm selection mismatch')
    smoke_cfg=v3_config(V3_CONTROLLER,None,path);smoke_cfg.update(train_pool_size=10,validation_size=2,membership_manifest=None)
    ledger=dict(protocol=PROTOCOL,format_version=3,status='prepared',created_utc=datetime.now(timezone.utc).isoformat(),
        source_code_sha256=notebook_code_digest(V3_NOTEBOOK),membership_sha256=file_digest(root/'membership.json'),
        exclusions_sha256=file_digest(root/'exclusions.json'),source_hashes=v3_historical_snapshot(),expected_reference=reference,
        wall_limit_seconds=7200,started_unix=None,deadline_unix=None,finished_unix=None,arms=arms,
        smoke_config=smoke_cfg,smoke_run_dir=str(root/'smoke'),smoke_status='pending',
        clean_workload_upper=dict(smoke=14,training=240,validation=40,confirmation=60,total=354),
        official_test_access=False,formal_success_demonstrated=False,continuation_authorized=False)
    ledger['plan_sha256']=_v3_seal(ledger);write_json(path,ledger);return ledger


def _v3_check_sources(ledger):
    changed=[p for p,d in ledger['source_hashes'].items() if not Path(p).is_file() or file_digest(p)!=d]
    if changed: raise ValueError('Historical artifacts changed: '+repr(changed[:10]))
    return dict(preserved_files=len(ledger['source_hashes']),all_hashes_preserved=True)


def _v3_validate_entry(ledger,entry,require_checkpoint=False):
    root=Path(entry['run_dir']);meta_path=root/'config.json'
    if meta_path.exists():
        meta=json.loads(meta_path.read_text(encoding='utf-8'));ref=ledger['expected_reference']
        if meta['config']!=entry['config'] or meta['selection_id']!=entry['selection_id']: raise ValueError('Arm configuration/membership changed')
        if meta['initial_policy_digest']!=ref['policy_initial_sha256'] or meta['first_tokens']!=ref['first_tokens'] or meta['execution_model']['model']['digest']!=ref['model_digest']: raise ValueError('Model/action identity changed')
    if (root/'latest.json').exists():
        p=json.loads((root/'latest.json').read_text());target=(root/p['checkpoint']).resolve()
        if target.parent!=(root/'checkpoints').resolve() or file_digest(target)!=p['sha256']: raise ValueError('Invalid durable checkpoint')
        cp=torch.load(target,map_location='cpu',weights_only=False)
        try:
            if cp.get('protocol')!=PROTOCOL or cp['config']!=entry['config'] or cp['policy_only']: raise ValueError('Incompatible checkpoint')
            if cp['runtime']!=ledger['expected_reference']['runtime']: raise ValueError('Checkpoint runtime changed')
        finally: del cp;gc.collect()
    elif require_checkpoint or read_jsonl(root/'attempts.jsonl') or read_jsonl(root/'training.jsonl'):
        raise RuntimeError('Recorded work lacks a valid resumable checkpoint')


def _v3_initialize(ledger,entry):
    root=Path(entry['run_dir']);cfg=entry['config'];root.mkdir(parents=True,exist_ok=True);(root/'checkpoints').mkdir(exist_ok=True)
    if (root/'latest.json').exists():
        _v3_validate_entry(ledger,entry,True);session=load_run(root,for_resume=True)
    else:
        _v3_validate_entry(ledger,entry);session=initialize_experiment(cfg,create_run=False);session['run_dir']=root
        try:
            ref=ledger['expected_reference']
            if session['initial_policy_digest']!=ref['policy_initial_sha256'] or session['llm_metadata']['model']['digest']!=ref['model_digest'] or session['first_tokens']!=ref['first_tokens'] or runtime_identity(session)!=ref['runtime']: raise ValueError('Frozen runtime/model/initialization mismatch')
            if session['data']['selection_id']!=entry['selection_id']: raise ValueError('Selection mismatch')
            write_json(root/'config.json',dict(config=cfg,selection_id=entry['selection_id'],selection=session['data']['selection'],
                initial_policy_digest=session['initial_policy_digest'],first_tokens=session['first_tokens'],execution_model=session['llm_metadata'],hardware=hardware_metadata()),overwrite=(root/'config.json').exists())
            save_checkpoint(session)
            if entry['key']=='v2': publish_checkpoint(session,'initial',policy_only=True)
        except BaseException: release_experiment(session);raise
    session['progress']=True;return session


def _v3_examples(ledger_path):
    return json.loads((Path(ledger_path).parent/'membership.json').read_text(encoding='utf-8'))['confirmation_examples']


def _v3_eval_audit(result,session_root,examples,cfg):
    if result['question_ids']!=[e['id'] for e in examples] or result['questions']!=len(examples): raise ValueError('Confirmation membership mismatch')
    if result['identity_id']!=stable_hash(result['identity']) or result['identity']['split']!='confirmation': raise ValueError('Evaluation identity mismatch')
    events={e['attempt_id']:e for e in read_jsonl(Path(session_root)/'attempts.jsonl') if e['kind']=='evaluation_attempt_finished'}
    for row in result['records']:
        event=events.get(row['attempt_id'],{})
        if event.get('evaluation_identity')!=result['identity_id'] or event.get('trajectory')!=row['trajectory'] or event.get('reward')!=row['reward']: raise ValueError('Evaluation journal mismatch')
        if compute_reward('',row['execution']['output'],row['target'])!=row['reward']: raise ValueError('Numeric reward mismatch')
        if row['trajectory']['sampling_seed']!=candidate_seed(cfg,'evaluation',question_id=row['question_id'],split='confirmation'): raise ValueError('Evaluation stream mismatch')
    if result['numeric_exact_match_accuracy']!=sum(r['reward']['numeric_exact_match'] for r in result['records'])/len(examples): raise ValueError('Evaluation accuracy mismatch')


def v3_pair(a,b):
    if a['question_ids']!=b['question_ids']: raise ValueError('Paired membership mismatch')
    x=np.array([int(r['reward']['numeric_exact_match']) for r in a['records']]);y=np.array([int(r['reward']['numeric_exact_match']) for r in b['records']]);d=x-y
    samples=d[np.random.default_rng(20261021).integers(0,len(d),(10000,len(d)))].mean(axis=1)
    return dict(accuracy_difference=float(d.mean()),a_only=int(((x==1)&(y==0)).sum()),b_only=int(((x==0)&(y==1)).sum()),
        ties=int((x==y).sum()),descriptive_paired_95_interval=np.quantile(samples,[.025,.975]).tolist(),resamples=10000)


def reconcile_run(root):
    result=_v3_base_reconcile(root);events=read_jsonl(Path(root)/'attempts.jsonl')
    started={e['attempt_id']:e for e in events if e['kind']=='evaluation_attempt_started'};committed=set()
    for path in Path(root).glob('*confirmation*.json'):
        value=json.loads(path.read_text(encoding='utf-8'))
        if isinstance(value,dict): committed.update(r['attempt_id'] for r in value.get('records',[]) if 'attempt_id' in r)
    cat=result['categories']['confirmation'];valid=committed&set(started)
    cat['committed_attempts']=len(valid);cat['discarded_or_uncommitted_attempts']=cat['gross_attempts']-len(valid)
    for key in ('gross_attempts','committed_attempts','discarded_or_uncommitted_attempts','interrupted_attempts','repeated_attempts'):
        result['totals'][key]=sum(c[key] for c in result['categories'].values())
    return result


def v3_screen_gate(summaries,evaluations):
    correct={k:sum(r['reward']['numeric_exact_match'] for r in v['records']) for k,v in evaluations.items()}
    criteria=dict(both_completed=all(s['completed'] for s in summaries.values()),
        beats_v2=correct['v3']>=correct['v2']+1,beats_initial=correct['v3']>=correct['initial']+1,
        budget=summaries['v3']['gross_training_attempts']<=120,
        finite_updates=all(math.isfinite(r['gradient_norm']) and math.isfinite(r['loss']) for s in summaries.values() for r in s['_training']),
        post_warmup_k_varies=len(set(summaries['v3']['actual_k'][5:]))>1)
    return dict(promising=all(criteria.values()),criteria=criteria,correct=correct,
        scope='One-seed exploratory screen; no matched static or formal success claim')


def v3_report(ledger_path):
    ledger=_v3_read(ledger_path);root=Path(ledger_path).parent;summaries={};evaluations={}
    for a in ledger['arms']:
        verified=verify_run_artifacts(a['run_dir']);summaries[a['key']]=dict(verified['summary'],_training=verified['train'])
        evaluations[a['key']]=json.loads((root/(a['key']+'_confirmation.json')).read_text())
    evaluations['initial']=json.loads((root/'initial_confirmation.json').read_text())
    costs={a['key']:reconcile_run(a['run_dir']) for a in ledger['arms']};costs['smoke']=reconcile_run(ledger['smoke_run_dir'])
    return dict(protocol=PROTOCOL,status='completed',screening=v3_screen_gate(summaries,evaluations),
        accuracies={k:v['numeric_exact_match_accuracy'] for k,v in evaluations.items()},
        pairs=dict(v3_minus_v2=v3_pair(evaluations['v3'],evaluations['v2']),v3_minus_initial=v3_pair(evaluations['v3'],evaluations['initial'])),
        training_attempts={k:s['gross_training_attempts'] for k,s in summaries.items()},actual_k={k:s['actual_k'] for k,s in summaries.items()},
        selected_checkpoints={k:dict(episode=s['best']['episode'],score=s['best']['score']) for k,s in summaries.items()},
        accounting=costs,total_new_attempts=sum(c['totals']['gross_attempts'] for c in costs.values()),
        model_work_wall_seconds=datetime.now(timezone.utc).timestamp()-ledger['started_unix'],
        matched_static_comparison_run=False,formal_success_demonstrated=False,official_test_access=False)


def audit_study(ledger_path):
    ledger=_v3_read(ledger_path);root=Path(ledger_path).parent;out=[];examples=_v3_examples(ledger_path)
    manifest=json.loads((root/'membership.json').read_text());excluded=set(json.loads((root/'exclusions.json').read_text())['ids'])
    ids=manifest['ids']['train']+manifest['ids']['validation']+[e['id'] for e in examples]
    if len(set(ids))!=80 or set(ids)&excluded: raise ValueError('Fresh membership exclusion/leakage failed')
    for a in ledger['arms']:
        run=Path(a['run_dir']);_v3_validate_entry(ledger,a,require_checkpoint=a['status'] in ('running','incomplete','completed'))
        summary=json.loads((run/'summary.json').read_text()) if (run/'summary.json').exists() else None
        if a['status']=='completed': verify_run_artifacts(run)
        for name in (a['key']+'_confirmation.json','initial_confirmation.json' if a['key']=='v2' else '__absent__'):
            if (root/name).exists(): _v3_eval_audit(json.loads((root/name).read_text()),run,examples,a['config'])
        out.append(dict(key=a['key'],status=a['status'],episode=summary.get('episode') if summary else None,accounting=reconcile_run(run)))
    if ledger['status']=='completed' and not all(a['status']=='completed' and a['confirmation_status']=='completed' for a in ledger['arms']): raise ValueError('False study completion')
    return dict(protocol=PROTOCOL,status=ledger['status'],complete=ledger['status']=='completed',arms=out,
        smoke=reconcile_run(ledger['smoke_run_dir']),history=_v3_check_sources(ledger),
        original_deadline_unix=ledger['deadline_unix'],official_test_access=False)


def v3_nested_equal(a,b):
    if isinstance(a,torch.Tensor): return isinstance(b,torch.Tensor) and torch.equal(a,b)
    if isinstance(a,dict): return isinstance(b,dict) and a.keys()==b.keys() and all(v3_nested_equal(v,b[k]) for k,v in a.items())
    if isinstance(a,(list,tuple)): return type(a)==type(b) and len(a)==len(b) and all(v3_nested_equal(x,y) for x,y in zip(a,b))
    return a==b


def _v3_smoke(ledger,path):
    root=Path(ledger['smoke_run_dir']);cfg,datasets=load_gsm8k(ledger['smoke_config']);selection,_=selection_payload(cfg,datasets)
    entry=dict(key='smoke',run_dir=str(root),config=cfg,selection_id=stable_hash(selection));session=None
    try:
        session=_v3_initialize(ledger,entry)
        if not session['state']['validation']:
            v=evaluate_module(session,episode=0);session['state']['validation'].append(v);save_checkpoint(session,v)
        while session['state']['episode']<2:
            target=session['state']['episode']+1;original=synchronize_logs
            def pause_after_commit(current):
                original(current)
                if current is session and current['state']['episode']>=target:
                    globals()['synchronize_logs']=original
                    save_checkpoint(current);raise WallLimitReached('Intentional bounded smoke pause after committed warm-up episode')
            globals()['synchronize_logs']=pause_after_commit
            try: train_grpo(session)
            finally: globals()['synchronize_logs']=original
            if session['state']['episode']<target: raise RuntimeError('Smoke stopped before planned episode')
            digest=parameter_digest(session['policy']);adam=copy.deepcopy(session['optimizer'].state_dict())
            release_experiment(session);session=None
            session=load_run(root,for_resume=True);session['progress']=True
            if parameter_digest(session['policy'])!=digest or not v3_nested_equal(adam,session['optimizer'].state_dict()): raise ValueError('Live policy/Adam recovery mismatch')
            append_journal(session,dict(kind='smoke_recovery_verified',episode=target,policy_sha256=digest,adam_exact=True))
        final=evaluate_module(session,episode=2);write_json(root/'smoke_validation.json',final,overwrite=True)
        session['state']['status']='bounded_smoke_paused';save_checkpoint(session);write_summary(session)
        if journal_counts(session)!=10 or any(r['actual_k']!=5 for r in session['state']['train']): raise ValueError('Smoke five-attempt contract failed')
        ledger['smoke_status']='completed_intentional_partial';write_json(path,ledger,overwrite=True)
    finally: release_experiment(session)


def run_study(ledger_path,resume=False,allow_training=False):
    if not allow_training: raise RuntimeError('Explicit allow_training=True required before model work')
    path=Path(ledger_path)
    with study_owner(path):
        ledger=_v3_read(path)
        if ledger['status']=='completed': return audit_study(path)
        if ledger['started_unix'] is not None and not resume: raise RuntimeError('Existing study requires resume=True')
        if ledger['started_unix'] is None:
            clock=path.parent/'study_clock.json'
            if clock.exists():timing=json.loads(clock.read_text())
            else:
                now=datetime.now(timezone.utc).timestamp()
                timing=dict(started_unix=now,deadline_unix=now+7200,plan_sha256=ledger['plan_sha256'])
                write_json(clock,timing)
            ledger['started_unix']=timing['started_unix'];ledger['deadline_unix']=timing['deadline_unix']
        if datetime.now(timezone.utc).timestamp()>=ledger['deadline_unix']: raise WallLimitReached('Original deadline expired')
        ledger['status']='running';write_json(path,ledger,overwrite=True);session=None
        try:
            _v3_check_sources(ledger);info=preflight(ledger['arms'][0]['config'])
            if info['execution_model']['digest']!=V3_MODEL_DIGEST: raise ValueError('Pinned model missing or changed')
            write_json(path.parent/'preflight.json',info,overwrite=True)
            if ledger['smoke_status']!='completed_intentional_partial': _v3_smoke(ledger,path)
            for a in ledger['arms']:
                run=Path(a['run_dir']);print('ARM',a['key'],'state',a['status'],flush=True)
                summary=json.loads((run/'summary.json').read_text()) if (run/'summary.json').exists() else {}
                if not summary.get('completed'):
                    a['status']='initializing';write_json(path,ledger,overwrite=True);session=_v3_initialize(ledger,a)
                    a['status']='running';write_json(path,ledger,overwrite=True)
                    train_grpo(session)
                    if session['state']['status']!='completed':
                        a['status']='incomplete';ledger['status']='incomplete';ledger['stop_reason']=session['state']['status'];write_json(path,ledger,overwrite=True)
                        return audit_study(path)
                    release_experiment(session);session=None
                verify_run_artifacts(run);a['status']='completed';write_json(path,ledger,overwrite=True)
            # Confirmation outcomes are unavailable until both checkpoint selections finish.
            if not (path.parent/'initial_confirmation.json').exists():
                initial_arm=next(a for a in ledger['arms'] if a['key']=='v2')
                session=load_run(initial_arm['run_dir'],for_resume=False,role='initial');session['progress']=True
                if session['state']['episode']!=0 or parameter_digest(session['policy'])!=ledger['expected_reference']['policy_initial_sha256']: raise ValueError('Initial checkpoint identity mismatch')
                initial=evaluate_confirmation_subset(session,examples=_v3_examples(path),episode=0)
                write_json(Path(initial_arm['run_dir'])/'initial_confirmation.json',initial)
                write_json(path.parent/'initial_confirmation.json',initial);release_experiment(session);session=None
            for a in ledger['arms']:
                run=Path(a['run_dir'])
                result_path=path.parent/(a['key']+'_confirmation.json')
                if not result_path.exists():
                    session=load_run(run,for_resume=False);session['progress']=True
                    result=evaluate_confirmation_subset(session,examples=_v3_examples(path))
                    write_json(run/'selected_confirmation.json',result);write_json(result_path,result)
                    release_experiment(session);session=None
                a['confirmation_status']='completed';write_json(path,ledger,overwrite=True)
            report=v3_report(path);write_json(path.parent/'report.json',report,overwrite=True)
            ledger.update(status='completed',finished_unix=datetime.now(timezone.utc).timestamp());write_json(path,ledger,overwrite=True)
            audit=audit_study(path);write_json(path.parent/'final_audit.json',audit,overwrite=True)
            if report['screening']['promising']: prepare_continuation(path)
            print('V3 SCREEN COMPLETE',json.dumps({k:report[k] for k in ('accuracies','training_attempts','screening','total_new_attempts')},indent=2),flush=True)
            return audit
        except BaseException as exc:
            ledger['status']='incomplete';ledger['last_error']=f'{type(exc).__name__}: {exc}'
            for a in ledger['arms']:
                if a['status'] in ('initializing','running'): a['status']='incomplete'
            write_json(path,ledger,overwrite=True);raise
        finally: release_experiment(session)


def prepare_continuation(ledger_path):
    ledger=_v3_read(ledger_path);report=json.loads((Path(ledger_path).parent/'report.json').read_text())
    if ledger['status']!='completed' or not report['screening']['promising']: raise RuntimeError('Screen failed; further model work blocked')
    result=dict(protocol=PROTOCOL,status='prepared_requires_separate_approval',allow_training=False,
        source_ledger=str(ledger_path),source_ledger_sha256=file_digest(ledger_path),methods=['v2','v3','static_k4','static_k5','initial'],
        seed=42,episodes=30,static_training_attempts_max=270,validation_attempts_max=40,
        additional_fresh_confirmation_questions=100,confirmation_attempts_max=500,total_additional_executions_max=810,
        identity_requirements='Same membership/schedule/runtime/validation; audit source checkpoint hashes',
        gate=dict(training_savings_min=.20,accuracy_gap_max=.02,beats_static4=True,beats_initial=True),
        formal_upper=dict(methods=6,seeds=3,training=13800,validation=4500,testing=23742,total=42042),
        formal_shuffled='Preserve first five K=5 allocations; permute remaining V3 K sequence',
        official_test_access=False,final_test_explicit_operation=True)
    write_json(Path(ledger_path).parent/'continuation_plan.json',result,overwrite=True);return result


def run_formal_study(*args,**kwargs): raise RuntimeError('V3 formal work requires successful screens and separate approval')


def run_improvement_suite(*args,**kwargs): raise RuntimeError('Use V3 study interfaces explicitly')


def run_operation(*args,**kwargs): raise RuntimeError('Use V3 study interfaces explicitly')


CONFIG=v3_config()


def verify_run_artifacts(root,summary=None):
    root=Path(root); summary=summary or json.loads((root/'summary.json').read_text(encoding='utf-8'))
    meta=json.loads((root/'config.json').read_text(encoding='utf-8')); cfg=summary['config']
    def require(ok,message):
        if not ok: raise ValueError(f'Invalid artifacts in {root}: {message}')
    require(cfg==meta['config'] and cfg['protocol']==PROTOCOL,'protocol/config snapshot')
    require(summary['completed'] and summary['episode']==summary['durable_episode']==cfg['num_episodes'],'incomplete run')
    require(stable_hash(meta['selection'])==summary['selection_id']==meta['selection_id'],'selection digest')
    require(summary['initial_policy_digest']==meta['initial_policy_digest'],'initial policy digest')
    require(summary['runtime_identity']['packages']==meta['hardware']['packages'] and summary['runtime_identity']['execution_model_digest']==meta['execution_model']['model']['digest'],'runtime snapshot identity')
    rows=read_jsonl(root/'training.jsonl'); vals=read_jsonl(root/'validation.jsonl'); journal=read_jsonl(root/'attempts.jsonl')
    if cfg['allocation_mode']=='scheduled' and cfg['preset']=='formal':
        source=json.loads((root/'schedule_source.json').read_text())
        source_summary=json.loads((Path(source['adaptive_run'])/'summary.json').read_text())
        require(source_summary['completed'] and source_summary['method']=='adaptive' and source_summary['config']['seed']==cfg['seed'],'shuffled source completed/seed')
        require(source_summary['actual_k']==source['source_k'] and sorted(source['source_k'])==sorted(cfg['allocation_schedule']),'shuffled K multiset')
        require(np.random.default_rng(source['permutation_seed']).permutation(source['source_k']).tolist()==cfg['allocation_schedule']==source['shuffled_k'],'shuffled permutation')
    starts={e['attempt_id']:e for e in journal if e['kind']=='training_attempt_started'}
    finishes={e['attempt_id']:e for e in journal if e['kind']=='training_attempt_finished'}
    require(len(starts)==sum(e['kind']=='training_attempt_started' for e in journal),'duplicate attempt IDs')
    require([r['episode'] for r in rows]==list(range(1,cfg['num_episodes']+1)),'episode sequence')
    committed=set(); seen=set(); cumulative=0
    for r in rows:
        k=r['actual_k']; require(cfg['k_min']<=k<=cfg['k_max'] if cfg['allocation_mode']=='adaptive' else k==cfg['allocation_schedule'][r['episode']-1] if cfg['allocation_mode']=='scheduled' else k==cfg['static_k'],'group bound')
        require(k==r['attempted_pipeline_evaluations']==r['generated_candidates']==r['replayed_candidates'],'candidate totals')
        require(len(r['rewards'])==len(r['advantages'])==len(r['candidate_details'])==len(r['attempt_ids'])==k,'candidate arrays')
        require(r['successful_pipeline_evaluations']+r['failed_pipeline_evaluations']==k,'candidate outcome totals')
        mean,adv=group_advantages(r['rewards']); require(math.isclose(mean,r['group_mean']) and np.allclose(adv,r['advantages']),'centered advantages')
        require(all(x in (0.,1.) for x in r['rewards']),'binary reward')
        require(r['zero_advantage_group']==bool(np.all(adv==0)),'zero-advantage flag')
        require(math.isfinite(r['loss']) and math.isfinite(r['gradient_norm']),'finite update')
        for aid,detail,reward in zip(r['attempt_ids'],r['candidate_details'],r['rewards']):
            require(aid in starts and aid in finishes and finishes[aid].get('error') is None and aid not in committed,'journal provenance')
            require(starts[aid]['episode']==r['episode'] and starts[aid]['question_id']==r['question_id'],'journal question')
            require(detail['reward']==finishes[aid]['reward'] and detail['trajectory']==finishes[aid]['trajectory'],'journal candidate')
            require(detail['trajectory']['sampling_seed']==candidate_seed(cfg,'training',r['episode'],r['attempt_ids'].index(aid)),'training sampling seed')
            require(compute_reward('',detail['execution']['output'],r['target'])['reward']==reward,'numeric reward recomputation')
            committed.add(aid)
        offset=meta['selection']['episode_pool_offsets'][r['episode']-1]
        require(offset==r['question_pool_offset'] and meta['selection']['ids']['train'][offset]==r['question_id'],'episode schedule')
        seen.add(r['question_id']); require(r['unique_training_questions']==len(seen),'unique questions')
        require(r['optimizer_update_count']==r['episode'] and r['optimizer_step'],'one optimizer step per episode')
        require(r['controller_decisions'][-1]['stop'] and r['controller_decisions'][-1]['reason']==r['stop_reason'],'stop trace')
        if 'first_tokens' in meta:
            distribution=effective_policy_distribution(r['policy_diagnostics']['before_update'],meta['first_tokens'])
            require(math.isclose(distribution['neff'],r['effective_policy_classes']['neff']),'effective policy diversity')
            before_count=list(starts).index(r['attempt_ids'][0])
            reserve=(cfg['num_episodes']-r['episode'])*(cfg['k_min'] if cfg['allocation_mode']=='adaptive' else cfg['static_k'])
            bank=max(0,cfg['adaptive_budget_per_episode']*(r['episode']-1)-before_count) if cfg['allocation_mode']=='adaptive' else 0
            expected_allowance=min(cfg['k_max'],cfg['training_budget']-before_count-reserve,4+bank) if cfg['allocation_mode']=='adaptive' else cfg['allocation_schedule'][r['episode']-1] if cfg['allocation_mode']=='scheduled' else cfg['static_k']
            if cfg.get('controller_version')==V3_CONTROLLER:
                snapshot=v3_allocation_snapshot(cfg,r['episode']-1,before_count)
                bank=snapshot['bank'];expected_allowance=snapshot['allowance']
                require(r['actual_k']==5 if r['episode']<=5 else r['actual_k']<=expected_allowance,'warm-up/refinement bound')
            require(r['bank']==bank and r['gross_before']==before_count,'bank/gross provenance')
            require(r['allowance']==expected_allowance,'reserved episode allowance')
            for index,decision in enumerate(r['controller_decisions']):
                prefix=decision['actual_k']
                expected_decision=controller_decision(cfg,[d['trajectory'] for d in r['candidate_details'][:prefix]],r['rewards'][:prefix],meta['first_tokens'],distribution['neff'],r['allowance'],episode=r['episode'])
                require(decision==expected_decision,'controller decision recomputation')
                require(decision['stop']==(index==len(r['controller_decisions'])-1),'stopping at first decision')
        elif cfg['preset']=='formal': require(False,'formal controller provenance missing')
        cumulative+=k
    require(cumulative==summary['committed_training_attempts'] and len(starts)==summary['gross_training_attempts'],'gross/committed totals')
    require(len(starts)<=cfg['training_budget'] and summary['discarded_training_attempts']==len(starts)-cumulative,'gross budget')
    expected=[0]+[e for e in range(1,cfg['num_episodes']+1) if e%cfg['validation_interval']==0 or e==cfg['num_episodes']]
    require([v['episode'] for v in vals]==expected,'validation schedule')
    for v in vals:
        require(v['question_ids']==meta['selection']['ids']['validation'],'validation identity')
        require(not set(v['question_ids']) & set(meta['selection']['ids']['train']),'validation leakage')
        require(v['questions']==v['attempted_pipeline_evaluations']==cfg['validation_size'],'validation size')
        require(v['cumulative_training_pipeline_evaluations']==sum(r['actual_k'] for r in rows if r['episode']<=v['episode']),'variable-K validation budget')
        require(all(compare_answers(r['execution']['output'],r['target'])==r['reward']['numeric_exact_match'] for r in v['records']),'validation reward recomputation')
        require(v['numeric_exact_match_accuracy']==sum(r['reward']['numeric_exact_match'] for r in v['records'])/v['questions'],'validation score')
        all_finished={e['attempt_id']:e for e in journal if e['kind']=='evaluation_attempt_finished'}
        require(v['identity_id']==stable_hash(v['identity']),'evaluation identity hash')
        require(v['identity']['selection_id']==summary['selection_id'] and v['identity']['sampling_version']==cfg['sampling_version'],'evaluation identity configuration')
        for record in v['records']:
            event=all_finished.get(record['attempt_id'],{})
            require(event.get('evaluation_identity')==v['identity_id'] and event.get('trajectory')==record['trajectory'] and event.get('reward')==record['reward'],'evaluation journal provenance')
            require(record['trajectory']['sampling_seed']==candidate_seed(cfg,'evaluation',question_id=record['question_id'],split='validation'),'evaluation sampling seed')
    best=max(vals,key=lambda v:(v['numeric_exact_match_accuracy'],-v['episode']))
    require(summary['best']['episode']==best['episode'] and summary['best']['score']==best['numeric_exact_match_accuracy'],'best checkpoint selection')
    for role in ('best','latest'):
        pointer=json.loads((root/(role+'.json')).read_text(encoding='utf-8')); path=(root/pointer['checkpoint']).resolve()
        require(path.parent==(root/'checkpoints').resolve() and file_digest(path)==pointer['sha256'],role+' checkpoint checksum')
        require(pointer['episode']==(best['episode'] if role=='best' else cfg['num_episodes']),role+' checkpoint episode')
    tests=[json.loads(p.read_text(encoding='utf-8')) for p in root.glob('full_test_*.json')]
    require(len(tests)<=1,'repeated final tests')
    for t in tests:
        bp=json.loads((root/'best.json').read_text(encoding='utf-8'))
        require(t['selected_checkpoint']==dict(role='best',sha256=bp['sha256'],episode=bp['episode']),'test checkpoint')
        require(t['questions']==len(set(t['question_ids']))==1319 and t['split']=='full_test','full official test identity')
        require(t['question_ids']==[f'gsm8k:test:{i}' for i in range(1319)],'official test question IDs')
        require(all(compare_answers(r['execution']['output'],r['target'])==r['reward']['numeric_exact_match'] for r in t['records']),'test reward recomputation')
        require(t['numeric_exact_match_accuracy']==sum(r['reward']['numeric_exact_match'] for r in t['records'])/1319,'test accuracy')
        require(summary['final_held_out_accuracy']==t['numeric_exact_match_accuracy'],'test summary')
    return dict(root=str(root),summary=summary,train=rows,validation=vals,tests=tests,metadata=meta)

def evaluate_confirmation_subset(session,episode=None,examples=None):
    split='confirmation'
    if examples is None: raise ValueError('Explicit confirmation membership required')
    if split=='full_test' and examples is None: raise ValueError('Explicit final test examples required')
    examples=examples if examples is not None else session['data']['subsets']['validation']
    train_ids=set(session['data']['selection']['ids']['train'])
    if not examples or len({e['id'] for e in examples})!=len(examples) or any(e['id'] in train_ids for e in examples): raise ValueError('Evaluation leakage/duplicates/empty')
    cfg=session['config']; ep=session['state']['episode'] if episode is None else episode
    identity=dict(protocol=PROTOCOL,policy_sha256=parameter_digest(session['policy']),episode=ep,
        selection_id=session['data']['selection_id'],sampling_version=cfg['sampling_version'],seed=cfg['seed'],
        execution_digest=session['llm_metadata']['model']['digest'],runtime=runtime_identity(session),
        execution_configuration={k:cfg[k] for k in ('reward_mode','ollama_model','ollama_num_gpu','ollama_num_ctx','execution_max_tokens','execution_temperature')},
        split=split,questions=stable_hash(examples))
    identity_id=stable_hash(identity); root=Path(session['run_dir']); pending=root/f'{split}_progress_ep{ep}.json'
    if pending.exists():
        old=json.loads(pending.read_text(encoding='utf-8'))
        if old['identity']!=identity:
            archive=root/(pending.stem+'_discarded_'+uuid.uuid4().hex[:8]+'.json')
            os.replace(pending,archive)
            append_journal(session,dict(kind='evaluation_reuse_rejected',episode=ep,old_identity=old['identity_id'],new_identity=identity_id,archive=archive.name))
    # The finished journal is authoritative even if a crash occurred just before progress publication.
    done={}; ids={e['id'] for e in examples}
    for e in read_jsonl(root/'attempts.jsonl'):
        if e['kind']=='evaluation_attempt_finished' and e.get('evaluation_identity')==identity_id and e.get('error') is None and e['question_id'] in ids:
            done.setdefault(e['question_id'],dict(question_id=e['question_id'],attempt_id=e['attempt_id'],trajectory=e['trajectory'],execution=e['execution'],reward=e['reward'],_resources=e['resources'],_seconds=e['seconds']))
    mode=session['policy'].training; session['policy'].eval(); session['evaluation_identity']=identity_id
    def publish():
        write_json(pending,dict(identity=identity,identity_id=identity_id,records=list(done.values()),
            evaluated_checkpoint=json.loads((root/'latest.json').read_text()) if (root/'latest.json').exists() else None),overwrite=True)
    publish()
    try:
        with torch.no_grad():
            for ex in examples:
                if ex['id'] in done: continue
                before=len(session['lm'].events); begin=time.perf_counter()
                item=recorded_attempt(session,ex,'evaluation',split=split)
                done[ex['id']]=dict(question_id=ex['id'],**item,_resources=call_statistics(session['lm'].events[before:]),_seconds=time.perf_counter()-begin)
                publish()
    finally:
        session.pop('evaluation_identity',None); session['policy'].train(mode)
    records=[dict(done[e['id']],target=e['answer']) for e in examples]
    resources=[r['_resources'] for r in records]
    def total(key):
        values=[r.get(key) for r in resources]; return sum(values) if all(v is not None for v in values) else None
    return dict(split=split,episode=ep,identity=identity,identity_id=identity_id,
        question_ids=[r['question_id'] for r in records],questions=len(records),records=records,
        numeric_exact_match_accuracy=sum(r['reward']['numeric_exact_match'] for r in records)/len(records),
        attempted_pipeline_evaluations=len(records),seconds=sum(r['_seconds'] for r in records),
        time_scope='committed candidate execution incl sampling; discarded work reported in gross ledger',
        cumulative_training_pipeline_evaluations=sum(r['actual_k'] for r in session['state']['train']),
        gross_training_attempts=journal_counts(session),selected_checkpoint=session.get('loaded_checkpoint'),
        execution_llm_calls=total('execution_llm_calls'),physical_http_requests=None,prompt_tokens=total('prompt_tokens'),
        generated_tokens=total('generated_tokens'),llm_seconds=total('llm_seconds'),llm_failed_calls=total('llm_failed_calls'),
        call_count_definition='logical LM.forward attempts for committed evaluation records')

def save_checkpoint(session,validation=None):
    if validation is not None:
        old=session['state']['best']
        if old is None or validation['numeric_exact_match_accuracy']>old['score']:
            best=dict(episode=validation['episode'],score=validation['numeric_exact_match_accuracy'])
            session['state']['best']=best
            try: best['pointer']=publish_checkpoint(session,'best',policy_only=True)
            except BaseException:
                session['state']['best']=old; raise
    publish_checkpoint(session,'latest')
    # Both publications are durable. Preserve the references in latest state.
    keep={json.loads((session['run_dir']/'latest.json').read_text())['checkpoint']}
    if session['state']['best']: keep.add(session['state']['best']['pointer']['checkpoint'])
    if (session['run_dir']/'initial.json').exists(): keep.add(json.loads((session['run_dir']/'initial.json').read_text())['checkpoint'])
    for old in (session['run_dir']/'checkpoints').glob('*.pt'):
        if str(Path('checkpoints')/old.name) not in keep: old.unlink()
    synchronize_logs(session); write_summary(session)

In [ ]:
def v3_self_checks():
    import io,ast
    from contextlib import redirect_stdout
    from types import SimpleNamespace
    from unittest.mock import patch
    checks=[]
    def check(name,condition):
        if not condition: raise AssertionError(name)
        checks.append(name)
    def raises(name,fn,kind=ValueError):
        try: fn()
        except kind: checks.append(name)
        else: raise AssertionError(name+' did not raise')
    with redirect_stdout(io.StringIO()): retained=adagrpo_self_checks()
    check('retained 163 V2 engineering checks',retained['passed']==163)
    checks=list(retained['checks'])+checks
    raises('model work requires explicit opt-in',lambda:run_study('does-not-exist'),RuntimeError)
    raises('formal work remains blocked',lambda:run_formal_study(allow_formal_training=True),RuntimeError)
    raises('warm-up cannot fit two-episode budget',lambda:build_config('smoke',controller_version=V3_CONTROLLER))
    raises('warm-up count cannot be retuned',lambda:build_config(controller_version=V3_CONTROLLER,warmup_episodes=4))
    raises('warm-up cannot be combined with static mode',lambda:build_config('screen',controller_version=V3_CONTROLLER,allocation_mode='static'))
    cfg=v3_config()
    for c in range(30):
        for gross in range(121):
            future_w=max(0,4-c);reserve=5*future_w+2*(29-c-future_w)
            minimum=5 if c<5 else 2
            feasible=120-gross-reserve>=minimum
            try: value=v3_allocation_snapshot(cfg,c,gross)
            except BudgetExhausted:
                if feasible: raise AssertionError('Rejected feasible reservation')
                continue
            if not feasible or gross+value['allowance']+reserve>120: raise AssertionError('Invalid reservation')
            if c<5 and value['allowance']!=5: raise AssertionError('Warm-up bound')
            if c>=5 and not 2<=value['allowance']<=6: raise AssertionError('Adaptive bound')
    check('exhaustive 3630 budget/reservation cases',True)
    first=v3_allocation_snapshot(cfg,0,0)
    check('five future warm-up groups fully reserved',first['allowance']==5 and first['future_reservation']==70)
    check('warm-up debt is explicit',v3_allocation_snapshot(cfg,5,25)['warmup_debt']==5)
    check('debt prevents immediate six-candidate refinement',v3_allocation_snapshot(cfg,5,25)['allowance']==4)
    check('earned savings permit six candidates',v3_allocation_snapshot(cfg,8,29)['allowance']==6)
    check('interrupted work reduces final allowance',v3_allocation_snapshot(cfg,29,117)['allowance']==3)
    raises('insufficient final group stops incomplete',lambda:v3_allocation_snapshot(cfg,29,119),BudgetExhausted)
    class TinyPolicy(torch.nn.Module):
        def __init__(self):
            super().__init__();self.logits=torch.nn.Parameter(torch.tensor([.1,-.1,.2,-.2,.05,0.]));self.config=SimpleNamespace(n_positions=1024)
        def forward(self,**kw):return SimpleNamespace(logits=self.logits.reshape(1,1,-1).expand(1,2,-1))
    class TinyTokenizer:
        def __call__(self,*a,**kw):return {'input_ids':torch.tensor([[0,1]])}
    tokens={a:(0 if a=='CoT' else 1 if a=='Predict' else 5 if a=='stop' else 2+SIGNATURES.index(a)%3) for a in ACTIONS}
    fake={'train':[dict(question=f'fixture q{i}',answer='reason #### 1') for i in range(100)]}
    def trajectory(module='CoT',signature=None):
        signature=signature or SIGNATURES[0]
        return dict(module=module,signature=signature,actions=[module,signature,'stop'])
    a,b=trajectory(),trajectory('Predict');collision=trajectory(signature=SIGNATURES[3])
    for ep in range(1,6):
        check('warm-up never stops after two successes '+str(ep),not controller_decision(cfg,[a,b],[1.,1.],tokens,2.,5,episode=ep)['stop'])
        check('warm-up stops after five '+str(ep),controller_decision(cfg,[a,b,a,b,a],[1.]*5,tokens,2.,5,episode=ep)['reason']=='warmup_five')
    for rewards,ts,limit,reason in [([1.,1.],[a,b],6,'easy_probe'),([0.]*4,[a,b,a,b],6,'bounded_failure'),
        ([1.,0.,1.,0.],[a,a,b,b],6,'credit_cancelled'),([1.,0.]*3,[a,b]*3,6,'informative_refinement'),
        ([0.]*3,[a,b,a],3,'allowance_reached'),([1.,0.,1.,0.,1.],[a,b,a,b,a],5,'allowance_reached')]:
        check('post-warm-up '+reason+' K'+str(len(rewards)),controller_decision(cfg,ts,rewards,tokens,3.,limit,episode=6)['reason']==reason)
    check('collision cancellation retained',controller_decision(cfg,[a,collision,a,collision],[1.,0.,1.,0.],tokens,3.,6,episode=6)['reason']=='credit_cancelled')
    raises('missing phase cannot silently pick controller',lambda:controller_decision(cfg,[a,b],[1.,0.],tokens,2.,5))
    def execute(t,q,lm):
        lm.events.append(dict(seconds=.001,prompt_tokens=3,generated_tokens=2,error=None,completed=True))
        return dict(output='[1]' if t['module']=='Predict' else '[0]',success=True,error=None,seconds=.001)
    with tempfile.TemporaryDirectory(prefix='adagrpo-v3-engineering-') as tmp:
        def fixture(config=None,create_run=True):
            config=config or build_config('screen',controller_version=V3_CONTROLLER,num_episodes=30,training_budget=120,
                policy_device='cpu',train_pool_size=10,validation_size=2,validation_interval=30,checkpoint_interval=1,
                dataset_revision='fixture',results_root=tmp,wall_limit_seconds=None)
            model=TinyPolicy().eval();root=Path(tmp)/uuid.uuid4().hex;root.mkdir();(root/'checkpoints').mkdir()
            ss=dict(config=config,policy=model,optimizer=torch.optim.Adam(model.parameters(),lr=1e-5,foreach=False),
                tokenizer=TinyTokenizer(),first_tokens=tokens,lm=SimpleNamespace(events=[]),
                llm_metadata=dict(model=dict(digest='fixture'),version=dict(version='fixture')),
                run_dir=root,data=prepare_selection(config,fake,persist=False),initial_policy_digest=parameter_digest(model),
                deadline=None,in_episode=False,evaluation_only=False,state=dict(episode=0,updates=0,train=[],validation=[],seen=[],best=None,status='initialized',gross_training_attempts=0))
            if create_run:
                write_json(root/'config.json',dict(config=config,selection_id=ss['data']['selection_id'],selection=ss['data']['selection'],
                    initial_policy_digest=ss['initial_policy_digest'],first_tokens=tokens,execution_model=ss['llm_metadata'],hardware=hardware_metadata()))
            return ss
        with patch.dict(globals(),execute_pipeline=execute,initialize_experiment=fixture,ollama_snapshot=lambda cfg:{}):
            frozen={'__name__':'frozen_v2_equivalence'}
            with redirect_stdout(io.StringIO()):
                old=json.loads(Path(FROZEN_V2).read_text(encoding='utf-8'))
                for i,cell in enumerate(old['cells']):
                    if cell['cell_type']=='code':exec(compile(''.join(cell['source']),f'frozen-v2-{i}','exec'),frozen)
            v2_cfg=build_config('screen',num_episodes=30,training_budget=120,policy_device='cpu',dataset_revision='fixture',train_pool_size=10,validation_size=2,results_root=tmp,wall_limit_seconds=None)
            left=fixture(v2_cfg);right=fixture(dict(v2_cfg,protocol='adagrpo-gsm8k-numeric-figure2-v2'))
            ex=left['data']['subsets']['train'][0]
            frozen['execute_pipeline']=execute
            lg=collect_group(left,ex);rg=frozen['collect_group'](right,ex)
            check('V2 control exact original sampled trajectories',lg['trajectories']==rg['trajectories'])
            check('V2 control exact original reward and stopping decisions',lg['reward_details']==rg['reward_details'] and lg['decision_trace']==rg['decision_trace'])
            lu=update_policy(left,ex['question'],lg['trajectories'],[r['reward'] for r in lg['reward_details']])
            ru=frozen['update_policy'](right,ex['question'],rg['trajectories'],[r['reward'] for r in rg['reward_details']])
            check('V2 control exact original loss and gradients',lu['loss']==ru['loss'] and lu['gradient_norm']==ru['gradient_norm'])
            check('V2 control exact original Adam state',v3_nested_equal(left['optimizer'].state_dict(),right['optimizer'].state_dict()))
            check('V2 control exact original updated parameters',parameter_digest(left['policy'])==parameter_digest(right['policy']))
            s=fixture();ex=s['data']['subsets']['train'][0]
            correct=lambda t,q,lm:dict(execute(t,q,lm),output='[1]')
            with patch.dict(globals(),execute_pipeline=correct):g=collect_group(s,ex)
            check('collector retains five successes in warm-up',g['actual_k']==5 and len(g['trajectories'])==5 and journal_counts(s)==5)
            update=update_policy(s,ex['question'],g['trajectories'],[1.]*5)
            check('equal-reward warm-up still performs Adam step',update['optimizer_step'] and update['zero_advantage_group'])
            # Same five samples and reward feedback must give the exact same original update.
            left,right=fixture(),fixture();trajs=[]
            for k in range(5):
                gen,_=candidate_generator(left,'training',1,k)
                trajs.append(generate_pipeline(left['policy'],left['tokenizer'],tokens,ex['question'],left['config'],generator=gen))
            lu=update_policy(left,ex['question'],trajs,[1.,0.,1.,0.,1.]);ru=update_policy(right,ex['question'],trajs,[1.,0.,1.,0.,1.])
            check('warm-up loss and gradients unchanged',lu['loss']==ru['loss'] and lu['gradient_norm']==ru['gradient_norm'])
            check('warm-up Adam state exact',v3_nested_equal(left['optimizer'].state_dict(),right['optimizer'].state_dict()))
            full=fixture()
            initial=publish_checkpoint(full,'initial',policy_only=True);initial_digest=parameter_digest(full['policy'])
            with redirect_stdout(io.StringIO()):train_grpo(full)
            audit=verify_run_artifacts(full['run_dir'])
            check('complete synthetic study arm audited',audit['summary']['completed'] and full['state']['updates']==30)
            check('synthetic warm-up exact five groups',all(r['actual_k']==5 for r in full['state']['train'][:5]))
            check('synthetic gross budget ceiling',journal_counts(full)<=120)
            check('finite gradients throughout synthetic arm',all(math.isfinite(r['gradient_norm']) for r in full['state']['train']))
            baseline=load_run(full['run_dir'],for_resume=False,role='initial')
            check('initial policy remains recoverable after checkpoint rotation',baseline['state']['episode']==0 and parameter_digest(baseline['policy'])==initial_digest)
            # Interrupt after the fifth durable update, restore optimizer, and compare final parameters.
            interrupted=fixture();original=synchronize_logs
            def pause(current):
                original(current)
                if current is interrupted and current['state']['episode']==5:
                    globals()['synchronize_logs']=original;save_checkpoint(current);raise WallLimitReached('fixture boundary pause')
            with redirect_stdout(io.StringIO()):
                with patch.dict(globals(),synchronize_logs=pause):train_grpo(interrupted)
                resumed=load_run(interrupted['run_dir'],for_resume=True);train_grpo(resumed)
            check('warm-up boundary recovery parameter equality',parameter_digest(resumed['policy'])==parameter_digest(full['policy']))
            check('warm-up boundary recovery Adam equality',v3_nested_equal(resumed['optimizer'].state_dict(),full['optimizer'].state_dict()))
            check('warm-up recovery candidate counts identical',[r['actual_k'] for r in resumed['state']['train']]==[r['actual_k'] for r in full['state']['train']])
            # Partial external work is charged, without any policy update from the incomplete group.
            lost=fixture();save_checkpoint(lost);before=parameter_digest(lost['policy']);count=[0]
            training_questions={e['question'] for e in lost['data']['subsets']['train']}
            def failing(t,q,lm):
                if q in training_questions:
                    count[0]+=1
                    if count[0]==3:raise InfrastructureFailure('synthetic collection interruption')
                return execute(t,q,lm)
            with redirect_stdout(io.StringIO()):
                with patch.dict(globals(),execute_pipeline=failing):train_grpo(lost)
            check('incomplete warm-up performs no optimizer step',lost['state']['updates']==0 and parameter_digest(lost['policy'])==before)
            with redirect_stdout(io.StringIO()):recovered=load_run(lost['run_dir'],for_resume=True)
            check('lost warm-up attempts remain charged',journal_counts(recovered)==3 and budget_snapshot(recovered)['gross_before']==3)
            ev=fixture();save_checkpoint(ev);eval_count=[0]
            def fail_eval(t,q,lm):
                eval_count[0]+=1
                if eval_count[0]==2:raise InfrastructureFailure('synthetic validation interruption')
                return execute(t,q,lm)
            with patch.dict(globals(),execute_pipeline=fail_eval):
                raises('validation interruption recorded',lambda:evaluate_module(ev),InfrastructureFailure)
            old_events=len([e for e in read_jsonl(ev['run_dir']/'attempts.jsonl') if e['kind']=='evaluation_attempt_started'])
            result=evaluate_module(ev)
            new_events=len([e for e in read_jsonl(ev['run_dir']/'attempts.jsonl') if e['kind']=='evaluation_attempt_started'])
            check('partial validation reuses its finished question',old_events==2 and new_events==3 and result['questions']==2)
            confirmation=[dataset_example(fake,'train',i) for i in (90,91)]
            r=evaluate_confirmation_subset(ev,examples=confirmation)
            write_json(ev['run_dir']/'selected_confirmation.json',r)
            _v3_eval_audit(r,ev['run_dir'],confirmation,ev['config'])
            check('explicit confirmation journal audited',True)
            check('confirmation accounting counts completed work',reconcile_run(ev['run_dir'])['categories']['confirmation']['committed_attempts']==2)
            # Full fixture exercise of the actual smoke save/load orchestration.
            smoke_cfg=fixture()['config'];smoke_root=Path(tmp)/'smoke';ldg=Path(tmp)/'smoke-study.json'
            prototype=fixture(smoke_cfg)
            ledger=dict(smoke_run_dir=str(smoke_root),smoke_config=smoke_cfg,expected_reference=dict(
                policy_initial_sha256=prototype['initial_policy_digest'],model_digest='fixture',first_tokens=tokens,runtime=runtime_identity(prototype)))
            with patch.dict(globals(),load_gsm8k=lambda cfg:(cfg,fake)):
                with redirect_stdout(io.StringIO()):_v3_smoke(ledger,ldg)
            sr=read_jsonl(smoke_root/'training.jsonl');events=read_jsonl(smoke_root/'attempts.jsonl')
            check('actual smoke orchestration pauses at two groups',len(sr)==2 and [r['actual_k'] for r in sr]==[5,5])
            check('actual smoke proves both policy and Adam recovery',sum(e['kind']=='smoke_recovery_verified' for e in events)==2)
            check('actual smoke remains explicitly partial',json.loads((smoke_root/'summary.json').read_text())['completed'] is False)
            # Exercise the actual multi-arm runner, including a crash between arms.
            study=Path(tmp)/'study';study.mkdir();mp=study/'membership.json';lp=study/'study.json'
            subsets={k:[dataset_example(fake,'train',i) for i in ids] for k,ids in dict(train=list(range(50)),validation=list(range(50,60)),confirmation=list(range(60,80))).items()}
            manifest=dict(dataset_revision='fixture',indices=dict(train=list(range(50)),validation=list(range(50,60))),
                ids={k:[e['id'] for e in subsets[k]] for k in ('train','validation')},
                content_hashes={k:stable_hash(subsets[k]) for k in ('train','validation')},confirmation_examples=subsets['confirmation'])
            write_json(mp,manifest);write_json(study/'exclusions.json',dict(ids=[]))
            arms=[]
            for key,control in [('v2',V2_CONTROLLER),('v3',V3_CONTROLLER)]:
                config=build_config('screen',controller_version=control,num_episodes=30,training_budget=120,
                    train_pool_size=50,validation_size=10,validation_interval=30,checkpoint_interval=1,policy_device='cpu',
                    dataset_revision='fixture',results_root=str(study),membership_manifest=str(mp),suite_ledger=str(lp),wall_limit_seconds=None)
                selection,_=selection_payload(config,fake)
                arms.append(dict(key=key,run_dir=str(study/'runs'/key),config=config,selection_id=stable_hash(selection),status='pending',confirmation_status='pending'))
            smoke_config=build_config('screen',controller_version=V3_CONTROLLER,num_episodes=30,training_budget=120,
                train_pool_size=10,validation_size=2,validation_interval=30,checkpoint_interval=1,policy_device='cpu',
                dataset_revision='fixture',results_root=str(study),suite_ledger=str(lp),wall_limit_seconds=None)
            ref=fixture(smoke_config)
            plan=dict(protocol=PROTOCOL,format_version=3,status='prepared',source_code_sha256=notebook_code_digest(V3_NOTEBOOK),
                membership_sha256=file_digest(mp),exclusions_sha256=file_digest(study/'exclusions.json'),source_hashes={},
                expected_reference=dict(policy_initial_sha256=ref['initial_policy_digest'],model_digest='fixture',first_tokens=tokens,runtime=runtime_identity(ref)),
                wall_limit_seconds=7200,clean_workload_upper=dict(total=354),arms=arms,smoke_config=smoke_config,
                smoke_run_dir=str(study/'smoke'),smoke_status='pending',started_unix=None,deadline_unix=None,finished_unix=None)
            plan['plan_sha256']=_v3_seal(plan);write_json(lp,plan)
            original_audit=verify_run_artifacts;crashed=[False]
            def boundary_crash(root,*args,**kwargs):
                value=original_audit(root,*args,**kwargs)
                if Path(root).name=='v2' and not crashed[0]:
                    crashed[0]=True;raise RuntimeError('synthetic study interruption between arms')
                return value
            with patch.dict(globals(),load_gsm8k=lambda cfg:(cfg,fake),preflight=lambda cfg:dict(execution_model=dict(digest=V3_MODEL_DIGEST)),verify_run_artifacts=boundary_crash):
                with redirect_stdout(io.StringIO()):raises('study interruption between arms recorded',lambda:run_study(lp,allow_training=True),RuntimeError)
            deadline=json.loads(lp.read_text())['deadline_unix'];old_count=reconcile_run(study/'runs/v2')['categories']['training']['gross_attempts']
            with patch.dict(globals(),load_gsm8k=lambda cfg:(cfg,fake),preflight=lambda cfg:dict(execution_model=dict(digest=V3_MODEL_DIGEST))):
                with redirect_stdout(io.StringIO()):done=run_study(lp,resume=True,allow_training=True)
            check('multi-arm study recovery completes audited run',done['complete'])
            check('completed first arm is reused without retraining',reconcile_run(study/'runs/v2')['categories']['training']['gross_attempts']==old_count)
            check('study recovery retains original deadline',json.loads(lp.read_text())['deadline_unix']==deadline)
            before=[read_jsonl(Path(a['run_dir'])/'attempts.jsonl') for a in arms]
            with redirect_stdout(io.StringIO()):run_study(lp,resume=True,allow_training=True)
            check('completed study reuse launches no work',before==[read_jsonl(Path(a['run_dir'])/'attempts.jsonl') for a in arms])
            honest=json.loads(lp.read_text());changed=copy.deepcopy(honest);changed['deadline_unix']+=1;write_json(lp,changed,overwrite=True)
            raises('persisted original deadline cannot change',lambda:_v3_read(lp))
            write_json(lp,honest,overwrite=True)
            bad=json.loads(lp.read_text());bad['arms'][1]['config']['warmup_episodes']=4;write_json(lp,bad,overwrite=True)
            raises('changed frozen controller rejected',lambda:_v3_read(lp))
            # A checkpoint with recorded work cannot be treated as a fresh initialization.
            malformed=Path(tmp)/'orphan';malformed.mkdir();write_jsonl(malformed/'attempts.jsonl',[dict(kind='training_attempt_started',attempt_id='orphan')])
            raises('orphan model work blocks reinitialization',lambda:_v3_validate_entry(ledger,dict(run_dir=str(malformed),config=smoke_cfg,selection_id='x')),RuntimeError)
            pointer=json.loads((ev['run_dir']/'latest.json').read_text());cp=ev['run_dir']/pointer['checkpoint']
            payload=torch.load(cp,weights_only=False);payload['protocol']='v2-historical';torch.save(payload,cp)
            pointer['sha256']=file_digest(cp);write_json(ev['run_dir']/'latest.json',pointer,overwrite=True)
            raises('historical checkpoint cannot resume as V3',lambda:load_run(ev['run_dir'],for_resume=True))
        # OS lock excludes a second owner on the same ledger.
        lockpath=Path(tmp)/'lock-study.json'
        with study_owner(lockpath):raises('concurrent study runner rejected',lambda:_v3_lock_probe(lockpath),RuntimeError)
    summary=lambda ks:dict(completed=True,gross_training_attempts=sum(ks),actual_k=ks,_training=[dict(gradient_norm=1.,loss=0.)])
    ev=lambda n:dict(records=[dict(reward=dict(numeric_exact_match=i<n)) for i in range(20)])
    sums=dict(v2=summary([4]*30),v3=summary([5]*5+[2,4]*12+[2]))
    check('screen gate accepts predefined improvement',v3_screen_gate(sums,dict(v2=ev(12),v3=ev(14),initial=ev(13)))['promising'])
    check('screen gate rejects tie with initialization',not v3_screen_gate(sums,dict(v2=ev(12),v3=ev(14),initial=ev(14)))['promising'])
    check('formal six-method workload arithmetic',13800+4500+23742==42042)
    return dict(passed=len(checks),retained_checks=163,new_checks=len(checks)-163,checks=checks,
        scope='Synthetic correctness only; no external inference, model downloads, or experimental accuracy claims')


def _v3_lock_probe(path):
    with study_owner(path):pass


def test_model(*args,**kwargs):
    raise RuntimeError('Official testing is deferred; this notebook exposes only the authorized local screen')


## Explicit operation

1. `checks = v3_self_checks()`
2. `plan = prepare_study()`
3. `audit_study("results/adagrpo_v3/screen_seed42/study.json")`
4. Authorized local execution only: `run_study("results/adagrpo_v3/screen_seed42/study.json", allow_training=True)`
5. Recovery: the same call with `resume=True`; the persisted deadline is never reset.

Static controls and cloud work require separate approval after the screen passes. Frozen controller settings must not be tuned against confirmation outcomes.
